# FSO-SC — 동결된 최종 200장 평가

**재학습·모델 선택·채널 조정은 하지 않습니다.** 수정 완료한 실험 설정을 그대로 사용하고, 이전 개발 평가200장을 제외한 나머지800장 중 미사용200장을 평가합니다. 멤버십은 실행 전에 고정한 **파일SHA-256 manifest**로 확인하며 결과에 따라 다시 뽑지 않습니다. 기존 평가 점수는 재사용하지 않습니다.

## 입력 / 실행

GPU 런타임에서 위부터 모두 실행하세요. 기존 Drive의 `GEE_test.zip`과 Stage2 100epoch 최종 가중치만 필요합니다. 기준 실행에서는 ZIP·가중치·전체1000파일SHA와 고정200장 멤버십·실험 설정이 맞지 않으면 시작하지 않습니다. 학습·resume·best와 기존 결과는 변경하지 않습니다.

두 잡음 프로필 × 기존14조건 × FSO-SC/BPG/BPG+LDPC × 200장 = **16800회**, 무채널 두 방식400회를 포함해 **17200회**입니다. 완료 조건은 같은 protocol로 재실행할 때만 재사용합니다. 새 파일 **`final_holdout_summary.json`**과 그래프·표·이미지별 기록이 `final_holdout_v1/<run_id>`에 저장됩니다.

## 고정한 설정

- 기존 모델/손실/최종 checkpoint, 256²→32²×14bit source0.21875bpp, Alex LPIPS0.1.
- native BPG0.9.8/x265/8bit/YCbCr420/speed4, 파일 크기만으로 정하는 평균 source-rate matching. 개별 rate는 다를 수 있고, QP 선택에 LPIPS/SNR은 사용하지 않습니다. padding으로 rate를 꾸미지 않습니다.
- LDPC3/4,n2048,k1536, 같은 seeded RA 대체 행렬·soft normalized-min-sum0.8·50회·LLRclip30.
- 수정한8QAM CFO/phase 추정, 기존 QPSK/16QAM, 기존 두 채널 프로필. 실제 위상이나 송신 bit는 DSP·LDPC에 주지 않습니다. 진폭·잡음PSD CSI와 프레임/길이 정보는 기존 가정입니다.
- 같은 source 평균과 동일 channel use는 다릅니다. LDPC/QAM padding을 포함한 실제 bpp·심볼 수를 따로 기록합니다.
- BPG header검증/native디코드 실패의 논문식 marker1과, 회색 fallback의 실제LPIPS·실패율을 구분합니다. 오류 프레임을 버리지 않습니다.

이200장은 **현재 모델·평가 코드 조정에 사용하지 않은 image-disjoint 확인 세트**입니다. 같은1000장 pool이므로 독립 데이터셋이나 지리적으로 독립된 표본이라고 주장하지 않습니다. 로컬 GEE train/test ZIP의 동일파일·동일전처리RGB 중복은 없었지만, ImageNet 전체·인접지역 유사성과 checkpoint의 실제 학습 이력을 이 검사만으로 증명하지는 않습니다.

기준 dataset manifest는 평가값을 보지 않고 SHA-256 순위(seed20261029)로 동결했습니다. 논문의 미공개 원본 대신NAIP를 사용하며, 채널·LDPC행렬·DSP 등 미공개 구현 가정은 여전히 존재합니다. **이 최종 평가도 저자의 VPI/FPGA 전체 복제나 논문3dB이득을 자동으로 입증하지 않습니다.** 두 프로필 모두 보고하며 유리한 결과만 고르지 마세요.

공개 저장소에서 자기 데이터·자기 학습 가중치를 평가할 때는 설정 셀의 `DATA_MODE="own_data"`, `ENFORCE_REFERENCE_MODEL=False`를 명시적으로 사용합니다. 이 경우에도 실험 설정·seed는 고정되고 결과에는 reference 재실행이 아님을 표시합니다. 새 corpus의 개발200/확인200 분리를 평가 전 파일 해시로 고정합니다.


In [ ]:
# Colab/Linux only. Installs tools; no training or Google Drive API.
!pip -q install lpips==0.1.4 cmake==3.28.3
!apt-get -qq update
!apt-get -qq install build-essential yasm libpng-dev libjpeg-dev zlib1g-dev


In [ ]:
import base64
import gc
import hashlib
import json
import os
from pathlib import Path
import platform
import shutil
import time
import numpy as np
import torch
from PIL import Image
from google.colab import drive

drive.mount("/content/drive")
DRIVE_DATA_DIR = Path("/content/drive/MyDrive/FSO_data")
RUN_NAME = 'paper_faithful_public_run1'
MODEL_OUTPUT_DIR = DRIVE_DATA_DIR/"checkpoints"/RUN_NAME
DATA_MODE = 'own_data'
ENFORCE_REFERENCE_MODEL = False
LOCAL_CACHE = Path("/content/fso_sc_final_holdout")
LOCAL_CACHE.mkdir(parents=True,exist_ok=True)
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime before starting this comparison")
device = torch.device("cuda")
print("GPU:",torch.cuda.get_device_name())
for required in (DRIVE_DATA_DIR/"GEE_test.zip",MODEL_OUTPUT_DIR/"stage2_noise_training_final.pth"):
    if not required.is_file():
        raise FileNotFoundError(f"Required existing file missing: {required}")


In [ ]:
"""ZIP-to-local-cache adapter. No training, authentication, or downloads at import."""
import hashlib
import json
import shutil
import stat
import zipfile
from pathlib import Path, PurePosixPath

from PIL import Image


def prepare_uploaded_zip(source_zip, cache_root, dataset_name, expected_count):
    source_zip, cache_root = Path(source_zip), Path(cache_root)
    if dataset_name not in {"imagenet", "gee_train", "gee_test"}:
        raise ValueError("Unknown dataset key")
    if not source_zip.is_file():
        raise FileNotFoundError(f"Drive ZIP not found: {source_zip}")
    archive_dir = cache_root / "archives"
    archive_dir.mkdir(parents=True, exist_ok=True)
    source_stat = source_zip.stat()
    signature = hashlib.sha256(
        f"{source_zip.resolve()}:{source_stat.st_size}:{source_stat.st_mtime_ns}".encode()
    ).hexdigest()[:20]
    local_zip = archive_dir / f"{dataset_name}-{signature}.zip"
    if not local_zip.exists():
        print(f"Copying ZIP to local runtime: {source_zip.name}", flush=True)
        partial = local_zip.with_suffix(".zip.partial")
        shutil.copyfile(source_zip, partial)
        if partial.stat().st_size != source_stat.st_size:
            raise RuntimeError("ZIP copy size differs from source")
        partial.replace(local_zip)
    digest = hashlib.sha256()
    with local_zip.open("rb") as handle:
        for block in iter(lambda: handle.read(4 * 1024**2), b""):
            digest.update(block)
    zip_sha256 = digest.hexdigest()
    destination = cache_root / "images" / dataset_name / zip_sha256[:20]
    marker = destination / ".prepared.json"
    if marker.exists():
        status = json.loads(marker.read_text(encoding="utf-8"))
        if status != {"zip_sha256": zip_sha256, "image_count": expected_count}:
            raise RuntimeError(f"Unexpected cache marker: {marker}")
        images = [p for p in destination.rglob("*")
                  if p.is_file() and p.suffix.lower() in {".png", ".jpg", ".jpeg"}]
        if len(images) != expected_count:
            raise RuntimeError(f"Local cache changed: {destination}")
        print(f"Reuse {dataset_name}: {len(images):,} images", flush=True)
        return destination
    staging = destination.with_name(destination.name + ".incomplete")
    staging.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(local_zip) as archive:
        members = archive.infolist()
        files = []
        for item in members:
            path = PurePosixPath(item.filename)
            if (not path.parts or path.is_absolute() or ".." in path.parts
                    or "\\" in item.filename or ":" in item.filename
                    or stat.S_ISLNK(item.external_attr >> 16)):
                raise ValueError(f"Unsafe ZIP member: {item.filename}")
            if not item.is_dir():
                files.append(item.filename)
        if len(files) != len(set(files)):
            raise ValueError("Duplicate ZIP member names")
        image_members = [name for name in files
                         if PurePosixPath(name).suffix.lower() in {".png", ".jpg", ".jpeg"}]
        if len(image_members) != expected_count:
            raise RuntimeError(
                f"{source_zip.name}: expected {expected_count:,} images, "
                f"found {len(image_members):,}"
            )
        remaining = sum(item.file_size for item in members)
        if shutil.disk_usage(staging).free < remaining + 512 * 1024**2:
            raise RuntimeError("Insufficient local runtime disk space for extraction")
        # ZipFile checks CRC while extracting. Only isolated cache contents are written.
        print(f"Extracting {source_zip.name}: {expected_count:,} images", flush=True)
        archive.extractall(staging)
        for name in image_members:
            with Image.open(staging / name) as image:
                image.verify()
    status = {"zip_sha256": zip_sha256, "image_count": expected_count}
    (staging / ".prepared.json").write_text(json.dumps(status), encoding="utf-8")
    if destination.exists():
        raise FileExistsError(f"Refusing to overwrite existing folder: {destination}")
    staging.rename(destination)
    print(f"Ready {dataset_name}: {expected_count:,} images", flush=True)
    return destination

import os
import numpy as np
from pathlib import Path
import torch
from PIL import Image
from torch.utils.data import ConcatDataset, Dataset

PAPER_TRAIN_COUNTS = {"imagenet": 10_000, "google_earth": 1_000}
PAPER_DATA_PROTOCOL = "paper_all_11000_training_no_disclosed_split_v1"
PAPER_INPUT_SIZE = 256


class MoVQCenterCropBicubic:
    def __call__(self, image):
        width, height = image.size
        side = min(width, height)
        image = image.crop(((width-side)/2, (height-side)/2,
                            (width+side)/2, (height+side)/2))
        image = image.resize((256, 256), resample=Image.Resampling.BICUBIC, reducing_gap=1)
        array = np.asarray(image.convert("RGB"), dtype=np.float32) / 127.5 - 1
        return torch.from_numpy(np.ascontiguousarray(array.transpose(2, 0, 1)))


class VisionDataset(Dataset):
    def __init__(self, img_dir, transform=None, expected_count=None):
        self.img_dir = Path(img_dir)
        self.transform = transform
        self.img_names = sorted(path.relative_to(self.img_dir).as_posix()
                                for path in self.img_dir.rglob("*")
                                if path.is_file() and path.suffix.lower() in {".png", ".jpg", ".jpeg"})
        if expected_count is not None and len(self.img_names) != expected_count:
            raise RuntimeError(f"{self.img_dir}: expected {expected_count} images, found {len(self.img_names)}")

    def __len__(self):
        return len(self.img_names)

    def __getitem__(self, index):
        with Image.open(self.img_dir / self.img_names[index]) as image:
            image = image.convert("RGB")
            return self.transform(image) if self.transform else image.copy()




In [ ]:
"""Compact, paired FSO-SC evaluation helpers; no training or work at import.

The reduced grid, 200-image subset, seeds and bootstrap are reproduction choices,
not settings disclosed by the paper. All test results, including bad DSP frames,
are retained. The caller owns checkpoint loading, output files and resume logic.
"""
import hashlib
import json
import math

import numpy as np
import torch


COMPACT_EVAL_VERSION = "fso_sc_compact_eval_v1"
COMPACT_BASE_SEED = 20261028


def condition_key(condition):
    """Identity depends on physical parameters, never model/stage/group labels."""
    return json.dumps(condition["channel_kwargs"], sort_keys=True, separators=(",", ":"))


def compact_conditions():
    catalog = {}
    def add(group, power=36, modulation="QPSK", zenith=0, pointing=20,
            radiance=1e-4, weather="clear", rho=.95):
        kwargs = {"launch_power_dbm": float(power), "modulation": modulation,
                  "zenith_angle_deg": float(zenith), "pointing_error_urad": float(pointing),
                  "background_light_radiance_w_m2_sr_um": float(radiance),
                  "weather": weather, "scattered_power_ratio": float(rho)}
        key = json.dumps(kwargs, sort_keys=True, separators=(",", ":"))
        if key not in catalog:
            catalog[key] = {"id": hashlib.sha256(key.encode()).hexdigest()[:16],
                            "group": group, "groups": [group], "channel_kwargs": kwargs,
                            "label": (f"{modulation}; Pt={power:g}dBm; zenith={zenith:g}; "
                                      f"pointing={pointing:g}urad; BGL={radiance:g}; "
                                      f"weather={weather}; rho={rho:g}")}
        elif group not in catalog[key]["groups"]:
            catalog[key]["groups"].append(group)
        return catalog[key]
    main = [add("main", power=p, modulation=m, zenith=z)
            for p in (30, 33, 36) for m in ("QPSK", "8QAM", "16QAM") for z in (0, 30, 60, 80)]
    pointing = [add("pointing", modulation="16QAM", zenith=60, pointing=p) for p in (18, 20, 22)]
    background = [add("background", power=33, modulation="16QAM", pointing=19, radiance=b)
                  for b in (1e-4, 1e-2, 1)]
    weather = [add("weather", pointing=15, weather=w) for w in ("clear", "dust", "rain", "snow", "fog")]
    turbulence = [add("turbulence", power=33, modulation="16QAM", pointing=19, rho=r)
                  for r in (.95, .75, .596)]
    main_ids = {c["id"] for c in main}
    robustness = [c for c in catalog.values() if c["id"] not in main_ids]
    ablation = [c for c in main if c["channel_kwargs"]["launch_power_dbm"] == 33
                and c["channel_kwargs"]["zenith_angle_deg"] in (30, 60)]
    anchor_values = {(36, "QPSK", 0), (33, "8QAM", 60), (30, "16QAM", 80)}
    anchors = [c for c in main if (c["channel_kwargs"]["launch_power_dbm"],
                                  c["channel_kwargs"]["modulation"],
                                  c["channel_kwargs"]["zenith_angle_deg"]) in anchor_values]
    return {"main": main, "robustness": robustness, "all": list(catalog.values()),
            "pointing": pointing, "background": background, "weather": weather,
            "turbulence": turbulence, "ablation": ablation, "anchors": anchors}


def evaluation_plan(image_count=200):
    conditions = compact_conditions()
    anchor_ids = {c["id"] for c in conditions["anchors"]}
    stage2 = [{"condition": c, "repeats": 3 if c["id"] in anchor_ids else 1}
              for c in conditions["all"]]
    stage1 = [{"condition": c, "repeats": 1} for c in conditions["ablation"]]
    n = int(image_count)
    if n < 1:
        raise ValueError("image_count must be positive")
    counts = {"stage2_physical_decodes": sum(j["repeats"] for j in stage2) * n,
              "stage1_physical_decodes": len(stage1) * n, "clean_decodes": 2 * n}
    counts["total_decodes"] = sum(counts.values())
    return {"version": COMPACT_EVAL_VERSION, "image_count": n, "stage2": stage2,
            "stage1_ablation": stage1, "counts": counts, "conditions": conditions}


def select_test_names(names, count=200, seed=COMPACT_BASE_SEED, expected_total=1000):
    names = sorted(str(name) for name in names)
    if len(names) != len(set(names)):
        raise ValueError("Test image identifiers must be unique, including subdirectories")
    if expected_total is not None and len(names) != expected_total:
        raise ValueError(f"Expected {expected_total} test images, found {len(names)}")
    if not 1 <= count <= len(names):
        raise ValueError("Invalid test subset count")
    indices = np.random.default_rng(seed).choice(len(names), size=count, replace=False)
    return sorted(names[int(i)] for i in indices)


def channel_seed(condition, image_name, repeat, base_seed=COMPACT_BASE_SEED):
    material = json.dumps([int(base_seed), condition_key(condition), str(image_name), int(repeat)],
                          separators=(",", ":"), ensure_ascii=False)
    return int.from_bytes(hashlib.sha256(material.encode("utf-8")).digest()[:8], "big") % (2**63 - 1)


def payload_bpp(indices, image_hw, bits=14):
    if indices.ndim < 2 or indices.shape[0] < 1 or len(image_hw) != 2:
        raise ValueError("Indices must retain a batch dimension and image_hw must be (H,W)")
    h, w = map(int, image_hw)
    if min(h, w, bits) < 1:
        raise ValueError("Image dimensions and bit width must be positive")
    return indices.numel() * int(bits) / (indices.shape[0] * h * w)


def image_means(records, metric="lpips"):
    grouped = {}
    seen = set()
    for record in records:
        identifier = (record["image_name"], int(record["repeat"]))
        if identifier in seen:
            raise ValueError("Duplicate image/repeat record")
        seen.add(identifier)
        score = float(record[metric])
        if not math.isfinite(score):
            raise FloatingPointError("Nonfinite evaluation score")
        grouped.setdefault(identifier[0], []).append(score)
    if not grouped:
        raise ValueError("Cannot summarize zero images")
    return {name: float(np.mean(values)) for name, values in sorted(grouped.items())}


def bootstrap_mean_ci(values, seed=COMPACT_BASE_SEED, bootstrap_samples=2000):
    values = np.asarray(list(values), dtype=np.float64)
    if values.ndim != 1 or not len(values) or not np.isfinite(values).all() or bootstrap_samples < 1:
        raise ValueError("Finite nonempty image-level values and positive bootstrap count required")
    draw = np.random.default_rng(seed).integers(0, len(values), size=(bootstrap_samples, len(values)))
    lower, upper = np.quantile(values[draw].mean(axis=1), (.025, .975))
    return {"mean": float(values.mean()), "ci95": [float(lower), float(upper)],
            "independent_images": len(values), "bootstrap_samples": int(bootstrap_samples),
            "ci_unit": "image; channel repeats averaged within each image"}


def summarize_records(records, seed=COMPACT_BASE_SEED, bootstrap_samples=2000):
    averaged = image_means(records)
    result = bootstrap_mean_ci(averaged.values(), seed, bootstrap_samples)
    result.update({"lpips_mean": result["mean"], "unique_images": len(averaged),
                   "image_channel_realizations": len(records), "per_image_lpips": averaged})
    count_keys = ("bit_errors", "valid_bits", "index_errors", "indices", "symbol_errors", "valid_symbols")
    totals = {key: sum(int(r.get(key, 0)) for r in records) for key in count_keys}
    result.update({"counts": totals,
                   "payload_ber": totals["bit_errors"] / max(totals["valid_bits"], 1),
                   "index_error_rate": totals["index_errors"] / max(totals["indices"], 1),
                   "symbol_error_rate": totals["symbol_errors"] / max(totals["valid_symbols"], 1),
                   "mse_mean": float(np.mean(list(image_means(records, "mse").values()))),
                   "failure_policy": "all finite frames retained, including bad DSP recovery; nonfinite raises"})
    return result


def paired_model_difference(stage2_records, stage1_records, seed=COMPACT_BASE_SEED, bootstrap_samples=2000):
    # Restrict repeated anchors to repeat0 when comparing the one-repeat ablation.
    right = {(r["image_name"], r["repeat"]): r for r in stage1_records}
    left = {(r["image_name"], r["repeat"]): r for r in stage2_records
            if (r["image_name"], r["repeat"]) in right}
    if len(left) != len(right) or not left:
        raise ValueError("Paired comparison requires identical image/repeat coverage")
    differences = []
    for key in sorted(right):
        a, b = left[key], right[key]
        if a.get("seed") != b.get("seed"):
            raise ValueError("Paired comparison requires the same channel seed")
        differences.append({"image_name": key[0], "repeat": key[1], "lpips": float(a["lpips"])-float(b["lpips"])})
    result = bootstrap_mean_ci(image_means(differences).values(), seed, bootstrap_samples)
    result.update({"difference": "Stage2 minus Stage1 LPIPS; negative favors Stage2",
                   "paired_realizations": len(differences)})
    return result


@torch.inference_mode()
def evaluate_condition(model, images_cpu, cached_indices_cpu, metric, channel, condition, names,
                       repeats=1, device="cuda", base_seed=COMPACT_BASE_SEED,
                       progress=None, sample_callback=None, sample_names=()):
    """Decode cached indices only; no encoder calls, optimizer steps or file writes.

    Set channel=None for a clean, error-free semantic reconstruction. Physical
    channel receives a fresh local generator per image/repeat, shared by stages.
    """
    if repeats < 1 or len(images_cpu) != len(names) or len(cached_indices_cpu) != len(names):
        raise ValueError("Invalid repetitions or cache/image/name lengths")
    if not len(names) or len(set(names)) != len(names):
        raise ValueError("Require nonempty unique image identifiers")
    model.eval()
    if hasattr(metric, "eval"):
        metric.eval()
    device = torch.device(device)
    samples = set(sample_names)
    records = []
    for i, name in enumerate(names):
        images = images_cpu[i:i+1].to(device)
        indices = cached_indices_cpu[i:i+1].to(device=device, dtype=torch.long)
        for repeat in range(repeats):
            seed = channel_seed(condition, name, repeat, base_seed)
            if channel is None:
                received, statistics = indices, {"valid_bits": indices.numel()*14, "indices": indices.numel()}
            else:
                received, statistics = channel(indices, return_statistics=True,
                                               generator=torch.Generator(device=device).manual_seed(seed),
                                               **condition["channel_kwargs"])
            prediction = model.decode_indices(received, received.shape)
            if prediction.shape != images.shape or not bool(torch.isfinite(prediction).all()):
                raise FloatingPointError(f"Invalid reconstruction for {name}, repeat {repeat}; frame not discarded")
            clipped = prediction.clamp(-1, 1)
            value = metric(clipped, images).reshape(-1)
            if value.numel() != 1 or not bool(torch.isfinite(value).all()):
                raise FloatingPointError(f"Invalid LPIPS for {name}, repeat {repeat}")
            record = {"image_name": str(name), "repeat": repeat, "seed": seed,
                      "lpips": float(value[0]), "mse": float((clipped-images).square().mean()),
                      "mse_raw": float((prediction-images).square().mean()),
                      "clipped_fraction": float((prediction.abs()>1).float().mean())}
            for key in ("bit_errors", "valid_bits", "index_errors", "indices", "symbol_errors", "valid_symbols"):
                record[key] = int(statistics.get(key, 0))
            diagnostics = statistics.get("channel", {}).get("signal_dsp", {})
            record["weak_CFR_peak_frames_diagnostic_only"] = sum(
                float(strength) < 10 for strength in diagnostics.get("cfr_peak_to_mean_power", []))
            records.append(record)
            if sample_callback is not None and name in samples and repeat == 0:
                sample_callback(name, repeat, clipped.detach().cpu())
            if progress is not None:
                progress(len(records), len(names)*repeats)
    return {"version": COMPACT_EVAL_VERSION, "condition": condition, "repeats": int(repeats),
            "records": records, "summary": summarize_records(records, seed=base_seed)}

EVAL_CODE_SHA256 = "aed6522206b907df78ddfbf2121910bb8d9e32cb4a316148074308f7206cd501"


In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

MODEL_VERSION = "fso_sc_paper_faithful_v6"
SEMANTIC_MODEL_CONFIG = {
    "group_norm_groups": 32,
    "activation": "SiLU",
    "downsample": "asymmetric_right_bottom_pad_then_stride2_conv",
    "upsample": "nearest_neighbor_then_conv",
    "loss_reduction": "mean",
    "input_preprocessing": "largest-center-square+bicubic256+RGB[-1,1]",
    "conv_initialization": "PyTorch_default",
    "codebook_initialization": "uniform(-1/K,1/K)",
    "decoder_attention_order": "VQGAN_inferred_interleaved_Res_Attn",
    "decoder_condition": "raw_Uq; post_quant_conv_only_for_decoder_feature",
}


def group_norm(channels, affine=True):
    return nn.GroupNorm(32, channels, eps=1e-6, affine=affine)


class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.norm1 = group_norm(in_channels)
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.norm2 = group_norm(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.shortcut = (
            nn.Conv2d(in_channels, out_channels, 1)
            if in_channels != out_channels
            else nn.Identity()
        )

    def forward(self, x):
        h = self.conv1(F.silu(self.norm1(x)))
        h = self.conv2(F.silu(self.norm2(h)))
        return self.shortcut(x) + h


class AttnBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.norm = group_norm(channels)
        self.q = nn.Conv2d(channels, channels, 1)
        self.k = nn.Conv2d(channels, channels, 1)
        self.v = nn.Conv2d(channels, channels, 1)
        self.proj_out = nn.Conv2d(channels, channels, 1)

    def forward(self, x):
        h = self.norm(x)
        q, k, v = self.q(h), self.k(h), self.v(h)
        batch, channels, height, width = q.shape
        q = q.reshape(batch, channels, height * width).permute(0, 2, 1)
        k = k.reshape(batch, channels, height * width)
        attention = torch.bmm(q, k) * (channels ** -0.5)
        attention = F.softmax(attention, dim=2)
        v = v.reshape(batch, channels, height * width)
        h = torch.bmm(v, attention.permute(0, 2, 1)).reshape(
            batch, channels, height, width
        )
        return x + self.proj_out(h)


class SpatialNorm(nn.Module):
    """Paper Eq. (25): gamma(U_q) * GN(F) + beta(U_q)."""

    def __init__(self, channels, latent_dim):
        super().__init__()
        # MoVQ replaces VQGAN's normalization with spatial conditioning while
        # retaining GroupNorm's learned affine parameters.
        self.norm = group_norm(channels, affine=True)
        self.conv_gamma = nn.Conv2d(latent_dim, channels, 1)
        self.conv_beta = nn.Conv2d(latent_dim, channels, 1)

    def forward(self, x, quantized_condition):
        if x.shape[2:] != quantized_condition.shape[2:]:
            quantized_condition = F.interpolate(
                quantized_condition, size=x.shape[2:], mode="nearest"
            )
        gamma = self.conv_gamma(quantized_condition)
        beta = self.conv_beta(quantized_condition)
        return gamma * self.norm(x) + beta


class SpatialResBlock(nn.Module):
    def __init__(self, in_channels, out_channels, latent_dim=4):
        super().__init__()
        self.norm1 = SpatialNorm(in_channels, latent_dim)
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.norm2 = SpatialNorm(out_channels, latent_dim)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.shortcut = (
            nn.Conv2d(in_channels, out_channels, 1)
            if in_channels != out_channels
            else nn.Identity()
        )

    def forward(self, x, quantized_condition):
        h = self.conv1(F.silu(self.norm1(x, quantized_condition)))
        h = self.conv2(F.silu(self.norm2(h, quantized_condition)))
        return self.shortcut(x) + h


class SpatialAttnBlock(nn.Module):
    def __init__(self, channels, latent_dim=4):
        super().__init__()
        self.norm = SpatialNorm(channels, latent_dim)
        self.q = nn.Conv2d(channels, channels, 1)
        self.k = nn.Conv2d(channels, channels, 1)
        self.v = nn.Conv2d(channels, channels, 1)
        self.proj_out = nn.Conv2d(channels, channels, 1)

    def forward(self, x, quantized_condition):
        h = self.norm(x, quantized_condition)
        q, k, v = self.q(h), self.k(h), self.v(h)
        batch, channels, height, width = q.shape
        q = q.reshape(batch, channels, height * width).permute(0, 2, 1)
        k = k.reshape(batch, channels, height * width)
        attention = torch.bmm(q, k) * (channels ** -0.5)
        attention = F.softmax(attention, dim=2)
        v = v.reshape(batch, channels, height * width)
        h = torch.bmm(v, attention.permute(0, 2, 1)).reshape(
            batch, channels, height, width
        )
        return x + self.proj_out(h)


class Upsample(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, 3, padding=1)

    def forward(self, x):
        return self.conv(F.interpolate(x, scale_factor=2.0, mode="nearest"))


class Downsample(nn.Module):
    """VQGAN downsample: asymmetric right/bottom pad then padding-free Conv."""

    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, 3, stride=2, padding=0)

    def forward(self, x):
        return self.conv(F.pad(x, (0, 1, 0, 1)))


class Encoder(nn.Module):
    def __init__(self, in_channels=3, latent_dim=4):
        super().__init__()
        self.in_conv = nn.Conv2d(in_channels, 256, 3, padding=1)
        self.res1 = nn.Sequential(ResidualBlock(256, 256), ResidualBlock(256, 256))
        self.down1 = Downsample(256)
        self.res2 = nn.Sequential(ResidualBlock(256, 512), ResidualBlock(512, 512))
        self.down2 = Downsample(512)
        self.res3 = nn.Sequential(ResidualBlock(512, 512), ResidualBlock(512, 512))
        self.down3 = Downsample(512)
        self.res4 = nn.Sequential(ResidualBlock(512, 1024), ResidualBlock(1024, 1024))
        self.attn1 = nn.Sequential(AttnBlock(1024), AttnBlock(1024))
        self.res5 = ResidualBlock(1024, 1024)
        self.attn2 = AttnBlock(1024)
        self.res6 = ResidualBlock(1024, 1024)
        self.out_norm = group_norm(1024)
        self.out_conv = nn.Conv2d(1024, latent_dim, 3, padding=1)

    def forward(self, x):
        x = self.res1(self.in_conv(x))
        x = self.res2(self.down1(x))
        x = self.res3(self.down2(x))
        x = self.res4(self.down3(x))
        x = self.attn1(x)
        x = self.attn2(self.res5(x))
        return self.out_conv(F.silu(self.out_norm(self.res6(x))))


class Decoder(nn.Module):
    """Fig. 3 block set arranged in the dimensionally valid MoVQ/VQGAN order."""

    def __init__(self, latent_dim=4):
        super().__init__()
        self.in_conv = nn.Conv2d(latent_dim, 1024, 3, padding=1)

        # VQGAN mid block: Res -> Attn -> Res.
        self.mid_res1 = SpatialResBlock(1024, 1024, latent_dim)
        self.mid_attn = SpatialAttnBlock(1024, latent_dim)
        self.mid_res2 = SpatialResBlock(1024, 1024, latent_dim)

        # Deepest level: three Res+Attn pairs, then 32 -> 64.
        self.deep_res = nn.ModuleList([
            SpatialResBlock(1024, 1024, latent_dim) for _ in range(3)
        ])
        self.deep_attn = nn.ModuleList([
            SpatialAttnBlock(1024, latent_dim) for _ in range(3)
        ])
        self.up_1024 = Upsample(1024)

        # 64x64 level: 1024 -> 512 followed by two 512 blocks, then 64 -> 128.
        self.level_512a = nn.ModuleList([
            SpatialResBlock(1024, 512, latent_dim),
            SpatialResBlock(512, 512, latent_dim),
            SpatialResBlock(512, 512, latent_dim),
        ])
        self.up_512a = Upsample(512)

        # 128x128 level: three 512 blocks, then 128 -> 256.
        self.level_512b = nn.ModuleList([
            SpatialResBlock(512, 512, latent_dim) for _ in range(3)
        ])
        self.up_512b = Upsample(512)

        # 256x256 level: 512 -> 256 followed by two 256 blocks.
        self.level_256 = nn.ModuleList([
            SpatialResBlock(512, 256, latent_dim),
            SpatialResBlock(256, 256, latent_dim),
            SpatialResBlock(256, 256, latent_dim),
        ])
        self.out_norm = SpatialNorm(256, latent_dim)
        self.out_conv = nn.Conv2d(256, 3, 3, padding=1)

    def forward(self, decoder_input, quantized_condition):
        h = self.mid_res1(self.in_conv(decoder_input), quantized_condition)
        h = self.mid_attn(h, quantized_condition)
        h = self.mid_res2(h, quantized_condition)

        for res_block, attn_block in zip(self.deep_res, self.deep_attn):
            h = attn_block(
                res_block(h, quantized_condition), quantized_condition
            )
        h = self.up_1024(h)

        for block in self.level_512a:
            h = block(h, quantized_condition)
        h = self.up_512a(h)

        for block in self.level_512b:
            h = block(h, quantized_condition)
        h = self.up_512b(h)

        for block in self.level_256:
            h = block(h, quantized_condition)
        return self.out_conv(F.silu(self.out_norm(h, quantized_condition)))


class VectorQuantizer(nn.Module):
    def __init__(
        self,
        num_embeddings=16384,
        embedding_dim=4,
        commitment_cost=1.0,
        distance_chunk_size=2048,
    ):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.num_embeddings = num_embeddings
        self.commitment_cost = commitment_cost
        self.distance_chunk_size = distance_chunk_size
        self.embedding = nn.Embedding(num_embeddings, embedding_dim)
        self.embedding.weight.data.uniform_(
            -1.0 / num_embeddings, 1.0 / num_embeddings
        )

    @torch.no_grad()
    def nearest_indices(self, flat_input):
        # VQ assignment remains FP32 even when the surrounding model uses AMP.
        vectors = flat_input.detach().float()
        codebook = self.embedding.weight.detach().float()
        codebook_norm = codebook.square().sum(dim=1).unsqueeze(0)
        nearest = []
        for chunk in vectors.split(self.distance_chunk_size, dim=0):
            distances = (
                chunk.square().sum(dim=1, keepdim=True)
                + codebook_norm
                - 2.0 * chunk @ codebook.t()
            )
            nearest.append(distances.argmin(dim=1))
        return torch.cat(nearest, dim=0)

    def lookup(self, encoding_indices, spatial_shape=None):
        if spatial_shape is None:
            if encoding_indices.ndim != 3:
                raise ValueError("indices must have shape [batch, height, width]")
            batch, height, width = encoding_indices.shape
        else:
            batch, height, width = spatial_shape
        flat_indices = encoding_indices.reshape(-1).long()
        quantized = F.embedding(flat_indices, self.embedding.weight)
        return quantized.view(batch, height, width, self.embedding_dim).permute(
            0, 3, 1, 2
        ).contiguous()

    def forward(self, inputs):
        channels_last = inputs.permute(0, 2, 3, 1).contiguous()
        batch, height, width, channels = channels_last.shape
        if channels != self.embedding_dim:
            raise ValueError(f"Expected embedding_dim={self.embedding_dim}, got {channels}")

        flat_input = channels_last.view(-1, self.embedding_dim)
        flat_indices = self.nearest_indices(flat_input)
        encoding_indices = flat_indices.view(batch, height, width)
        quantized = self.lookup(encoding_indices, (batch, height, width))

        encoder_loss = F.mse_loss(quantized.detach().float(), inputs.float())
        codebook_loss = F.mse_loss(quantized.float(), inputs.detach().float())
        vq_loss = codebook_loss + self.commitment_cost * encoder_loss

        quantized_st = inputs + (quantized.to(inputs.dtype) - inputs).detach()
        return quantized_st, vq_loss, encoding_indices


class FSO_VQVAE(nn.Module):
    def __init__(self, num_embeddings=16384, embedding_dim=4, commitment_cost=1.0):
        super().__init__()
        self.encoder = Encoder(in_channels=3, latent_dim=embedding_dim)
        self.pre_quant_conv = nn.Conv2d(embedding_dim, embedding_dim, 1)
        self.vq = VectorQuantizer(num_embeddings, embedding_dim, commitment_cost)
        self.post_quant_conv = nn.Conv2d(embedding_dim, embedding_dim, 1)
        self.decoder = Decoder(latent_dim=embedding_dim)

    def encode(self, x):
        if x.ndim != 4 or x.shape[1] != 3:
            raise ValueError("Expected RGB input with shape [batch, 3, height, width]")
        if x.shape[-2] % 8 or x.shape[-1] % 8:
            raise ValueError("Input height and width must be divisible by 8")
        continuous = self.pre_quant_conv(self.encoder(x))
        quantized, vq_loss, indices = self.vq(continuous)
        return quantized, vq_loss, indices

    def decode_quantized(self, quantized):
        decoder_input = self.post_quant_conv(quantized)
        return self.decoder(decoder_input, quantized_condition=quantized)

    def decode_indices(self, indices, spatial_shape):
        return self.decode_quantized(self.vq.lookup(indices, spatial_shape))

    def forward(self, x):
        quantized, vq_loss, indices = self.encode(x)
        return self.decode_quantized(quantized), vq_loss, indices


# The architecture is intentionally not instantiated here; it is large (~hundreds of M parameters).

In [ ]:
from dataclasses import dataclass
import math

import torch
import torch.nn.functional as F


BITS_PER_INDEX = 14
QAM_MAPPING_VERSION = "gray_qpsk_rect8qam_16qam_unit_energy_v1"
INTEGER_DTYPES = {torch.uint8, torch.int8, torch.int16, torch.int32, torch.int64}


def integers_to_bits(values, width):
    values = values.long()
    shifts = torch.arange(width - 1, -1, -1, device=values.device)
    return ((values.unsqueeze(-1) >> shifts) & 1).to(torch.uint8)


def bits_to_integers(bits):
    bits = bits.long()
    width = bits.shape[-1]
    weights = (2 ** torch.arange(width - 1, -1, -1, device=bits.device)).long()
    return (bits * weights).sum(dim=-1)


def indices_to_bits(indices, bits_per_index=BITS_PER_INDEX):
    if indices.dtype not in INTEGER_DTYPES:
        raise TypeError("indices must use an integer dtype")
    if indices.numel() and (
        (indices < 0).any().item() or (indices >= 2**bits_per_index).any().item()
    ):
        raise ValueError("Index is outside the fixed-width bit representation")
    return integers_to_bits(indices, bits_per_index)


def bits_to_indices(bits, bits_per_index=BITS_PER_INDEX):
    if bits.shape[-1] != bits_per_index:
        raise ValueError(f"Last dimension must be {bits_per_index} bits")
    if bits.dtype not in INTEGER_DTYPES and bits.dtype != torch.bool:
        raise TypeError("bits must use an integer or bool dtype")
    if bits.numel() and ((bits != 0) & (bits != 1)).any().item():
        raise ValueError("bits must contain only 0 and 1")
    return bits_to_integers(bits)


def qam_constellation(modulation, device):
    modulation = modulation.upper()
    supported = {"QPSK": 2, "8QAM": 3, "16QAM": 4}
    if modulation not in supported:
        raise ValueError(f"Unsupported modulation: {modulation}")
    bits_per_symbol = supported[modulation]
    labels = integers_to_bits(
        torch.arange(2**bits_per_symbol, device=device), bits_per_symbol
    ).float()

    if modulation == "QPSK":
        i = 1.0 - 2.0 * labels[:, 0]
        q = 1.0 - 2.0 * labels[:, 1]
        normalization = math.sqrt(2.0)
    elif modulation == "8QAM":
        # Rectangular Gray-labelled 8-QAM. The paper does not specify its 8-QAM geometry.
        gray_levels = torch.tensor([-3.0, -1.0, 3.0, 1.0], device=device)
        i_code = (2 * labels[:, 0] + labels[:, 1]).long()
        i = gray_levels[i_code]
        q = 2.0 * labels[:, 2] - 1.0
        normalization = math.sqrt(6.0)
    else:
        gray_levels = torch.tensor([-3.0, -1.0, 3.0, 1.0], device=device)
        i_code = (2 * labels[:, 0] + labels[:, 1]).long()
        q_code = (2 * labels[:, 2] + labels[:, 3]).long()
        i = gray_levels[i_code]
        q = gray_levels[q_code]
        normalization = math.sqrt(10.0)

    points = torch.complex(i, q) / normalization
    return points.to(torch.complex64), labels.to(torch.uint8), bits_per_symbol


def qam_modulate(bits, modulation):
    points, _, bits_per_symbol = qam_constellation(modulation, bits.device)
    if bits.shape[-1] != bits_per_symbol:
        raise ValueError(f"{modulation} expects {bits_per_symbol} bits per symbol")
    symbol_ids = bits_to_integers(bits).long()
    return points[symbol_ids]


def qam_demodulate(symbols, modulation):
    points, labels, _ = qam_constellation(modulation, symbols.device)
    # Cartesian-product QAM: limiting observations outside the outer levels
    # preserves exact nearest-point decisions and avoids large-float cancellation.
    i_limit = 2 * points.real.abs().max()
    q_limit = 2 * points.imag.abs().max()
    stable = torch.complex(symbols.real.clamp(-i_limit, i_limit),
                           symbols.imag.clamp(-q_limit, q_limit))
    distances = (stable.reshape(-1, 1) - points.reshape(1, -1)).abs().square()
    nearest = distances.argmin(dim=1)
    return labels[nearest].view(*symbols.shape, labels.shape[-1])


@dataclass(frozen=True)
class FSOChannelOutput:
    """Receiver observation before hard QAM decision.

    Return `is_equalized=True` after coherent receiver DSP, or provide the
    complex channel gain so this link can perform one-tap equalization.
    """

    symbols: torch.Tensor
    channel_gain: object = None
    is_equalized: bool = False
    diagnostics: object = None


class IdentityFSOChannel:
    """No-impairment receiver hook used to test the digital path."""

    def __call__(self, symbols, **channel_kwargs):
        return FSOChannelOutput(symbols=symbols, is_equalized=True)


def equalize_fso_output(channel_output, transmitted):
    if isinstance(channel_output, FSOChannelOutput):
        received = channel_output.symbols
        channel_gain = channel_output.channel_gain
        is_equalized = channel_output.is_equalized
    elif isinstance(channel_output, tuple) and len(channel_output) == 2:
        received, channel_gain = channel_output
        is_equalized = False
    else:
        raise TypeError(
            "FSO receiver must return FSOChannelOutput or (symbols, channel_gain)"
        )

    if not torch.is_tensor(received):
        raise TypeError("FSO receiver must return a torch.Tensor observation")
    if received.device != transmitted.device:
        raise ValueError(
            f"FSO receiver changed device: {transmitted.device} -> {received.device}"
        )
    if not torch.is_complex(received):
        raise TypeError("FSO receiver output must be complex baseband symbols")
    if received.shape != transmitted.shape:
        raise ValueError(
            f"FSO receiver changed symbol shape: {transmitted.shape} -> {received.shape}"
        )

    if not is_equalized:
        if not torch.is_tensor(channel_gain):
            raise TypeError("Unequalized FSO output must include a tensor channel_gain")
        if channel_gain.device != received.device:
            raise ValueError("channel_gain and received symbols must share a device")
        try:
            channel_gain = torch.broadcast_to(channel_gain, received.shape)
        except RuntimeError as error:
            raise ValueError("channel_gain is not broadcastable to the symbol shape") from error
        if (channel_gain.abs() == 0).any().item():
            raise ValueError("channel_gain contains an exact zero")
        if not torch.isfinite(channel_gain).all().item():
            raise ValueError("channel_gain contains NaN or Inf")
        denominator = channel_gain.abs().square().clamp_min(
            torch.finfo(received.real.dtype).tiny
        )
        received = received * channel_gain.conj() / denominator

    if not torch.isfinite(received).all().item():
        raise ValueError("FSO receiver output contains NaN or Inf")
    return received


@dataclass
class QAMIndexLink:
    """14-bit digital link ending in equalized complex [B, S] symbols."""

    fso_channel: object
    modulation: str = "QPSK"
    bits_per_index: int = BITS_PER_INDEX
    receiver_config: object = None

    def reproducibility_config(self):
        if self.receiver_config is not None and not isinstance(
            self.receiver_config, dict
        ):
            raise TypeError("receiver_config must be a serializable dict or None")
        return {
            "qam_mapping_version": QAM_MAPPING_VERSION,
            "modulation": self.modulation.upper(),
            "bits_per_index": self.bits_per_index,
            "receiver_config": self.receiver_config,
        }

    @torch.no_grad()
    def __call__(self, indices, return_statistics=False, **channel_kwargs):
        if indices.ndim < 2:
            raise ValueError("indices must retain a batch dimension")
        original_shape = indices.shape
        batch = original_shape[0]
        valid_bits_per_sample = indices[0].numel() * self.bits_per_index
        bitstream = indices_to_bits(indices, self.bits_per_index).reshape(batch, -1)
        _, _, bits_per_symbol = qam_constellation(self.modulation, indices.device)
        padding = (-valid_bits_per_sample) % bits_per_symbol
        if padding:
            bitstream = F.pad(bitstream, (0, padding))

        grouped_bits = bitstream.view(batch, -1, bits_per_symbol)
        transmitted = qam_modulate(grouped_bits, self.modulation)
        channel_output = self.fso_channel(transmitted, **channel_kwargs)
        received = equalize_fso_output(channel_output, transmitted)
        recovered_grouped = qam_demodulate(received, self.modulation)
        recovered = recovered_grouped.reshape(batch, -1)
        valid_mask = torch.arange(
            recovered.shape[1], device=indices.device
        ).view(1, -1) < valid_bits_per_sample
        transmitted_bit_error_mask = recovered != bitstream
        data_bit_error_mask = transmitted_bit_error_mask & valid_mask
        # SER is defined over every transmitted QAM symbol, including the final
        # padded symbol. Payload BER/TER continue to exclude padding bits.
        symbol_error_mask = transmitted_bit_error_mask.view(
            batch, -1, bits_per_symbol
        ).any(dim=-1)
        if padding:
            recovered = recovered[:, :-padding]
        recovered = recovered.view(*original_shape, self.bits_per_index)
        recovered_indices = bits_to_indices(
            recovered, self.bits_per_index
        ).view(original_shape)
        if not return_statistics:
            return recovered_indices

        bit_errors = int(data_bit_error_mask.sum().item())
        valid_bits = batch * valid_bits_per_sample
        transmitted_bit_errors = int(transmitted_bit_error_mask.sum().item())
        transmitted_bits = bitstream.numel()
        symbol_errors = int(symbol_error_mask.sum().item())
        valid_symbols = symbol_error_mask.numel()
        index_errors = int((recovered_indices != indices).sum().item())
        statistics = {
            "modulation": self.modulation.upper(),
            "padding_bits_per_sample": padding,
            "bit_errors": bit_errors,
            "valid_bits": valid_bits,
            "ber": bit_errors / max(valid_bits, 1),
            "transmitted_bit_errors": transmitted_bit_errors,
            "transmitted_bits": transmitted_bits,
            "transmitted_ber": transmitted_bit_errors / max(transmitted_bits, 1),
            "symbol_errors": symbol_errors,
            "valid_symbols": valid_symbols,
            "ser": symbol_errors / max(valid_symbols, 1),
            "index_errors": index_errors,
            "indices": indices.numel(),
            "ter": index_errors / max(indices.numel(), 1),
        }
        if isinstance(channel_output, FSOChannelOutput) and channel_output.diagnostics:
            statistics["channel"] = channel_output.diagnostics
        return recovered_indices, statistics


def qam_identity_roundtrip_test():
    test_indices = torch.tensor([
        [0, 1, 2, 8191, 8192, 16383, 77],
        [77, 16383, 8192, 8191, 2, 1, 0],
    ]).view(2, 1, 7)
    for modulation in ("QPSK", "8QAM", "16QAM"):
        link = QAMIndexLink(IdentityFSOChannel(), modulation=modulation)
        recovered = link(test_indices)
        assert torch.equal(recovered, test_indices), f"{modulation} round trip failed"
    print("Identity-channel index/QAM round-trip passed: QPSK, 8QAM, 16QAM")


qam_identity_roundtrip_test()

In [ ]:
"""Paper-parameter symbol-rate FSO channel with explicit heterodyne shot floor.

The vacuum/shot term is a physically motivated missing receiver detail, not an
explicit Chen et al. parameter or an LPIPS-curve calibration. One complex
heterodyne mode/symbol has E|n|^2=h*nu/eta, so the optical-field (sqrt W)
convention gives N_complex=(q_e/R)*R_s W; I and Q each have half that variance.
Reference: A. Martinez, Physical Review A 77,032116 (2008),
doi:10.1103/PhysRevA.77.032116, https://ir.cwi.nl/pub/13524/13524D.pdf .
"""

from dataclasses import asdict, dataclass, replace
import math
import warnings

import numpy as np
import torch


PAPER_FSO_CHANNEL_VERSION = "chen2025_eq3_18_heterodyne_shot_v6"
WEATHER_ATTENUATION_DB_PER_KM = {
    "clear": 0.147,
    "dust": 1.043,
    "fog": 33.979,
    "rain": 1.403,
    "snow": 2.145,
}

# The paper specifies FOV=0.1 but does not print a unit, and it does not give
# the optical filter width or the proprietary VPI receiver/DSP implementation.
# These assumptions are explicit and stored with the final model metadata.
PAPER_UNDISCLOSED_RECEIVER_ASSUMPTIONS = {
    "fov_interpretation": "0.1 rad",
    "optical_filter_bandwidth": "1 nm",
    "background_noise_conversion": (
        "Eq.-(4) optical background power is used as unit-energy complex "
        "matched-filter noise variance; proprietary coherent receiver is undisclosed"
    ),
    "ase_model": (
        "Table-I launch power is post-transmitter-EDFA; standard two-polarization "
        "EDFA ASE is propagated through the optical path"
    ),
    "ase_noise_bandwidth": "28 GHz symbol-rate equivalent",
    "diversity_ase_correlation": "one common transmitter-ASE waveform across Rx branches",
    "photodiode_responsivity": (
        "Table-I R=0.85 A/W determines eta=R*h*nu/q_e; "
        "one-complex-mode/symbol heterodyne shot floor q_e*Rs/R is input-referred; "
        "strong LO and negligible receiver thermal/RIN/dark current are assumptions"
    ),
    "receiver_shot_noise": (
        "complex heterodyne mode variance h*nu/eta per symbol; "
        "N_complex=q_e*Rs/R W, with half per real quadrature; "
        "physically motivated undisclosed implementation detail, not curve-fitted"
    ),
    "receiver_shot_noise_reference": "https://doi.org/10.1103/PhysRevA.77.032116",
    "pulse_shaping": "symbol-rate matched-filter equivalent; RRC rolloff undisclosed",
    "carrier_recovery": (
        "raw channel returns true complex gain for an explicitly ideal/oracle baseline; "
        "the separate SignalDSP wrapper uses only block magnitude and estimates phase/frequency"
    ),
    "rytov_quadrature": "xi=u^2 ground-clustered trapezoid with convergence self-test",
    "malaga_phase_difference": "pi/2 chosen phase assumption; paper undisclosed",
    "malaga_beta_fit": (
        "paper omits beta and phase; nearest Eq.-(15) natural beta is used with "
        "an explicit feasibility/residual diagnostic (strict policy is available)"
    ),
    "fading_coherence": "one independent block fade per image and receiver",
    "8qam_geometry": "rectangular Gray 8-QAM; paper only states 8-QAM",
}


def paper_generator_device_matches(generator, tensor_device):
    generator_device = generator.device
    if generator_device.type != tensor_device.type:
        return False
    if tensor_device.type != "cuda":
        return generator_device == tensor_device
    generator_index = (
        torch.cuda.current_device()
        if generator_device.index is None
        else generator_device.index
    )
    tensor_index = (
        torch.cuda.current_device() if tensor_device.index is None else tensor_device.index
    )
    return generator_index == tensor_index


@dataclass(frozen=True)
class PaperFSOSystemConfig:
    # Paper Table I / Eqs. (4)-(18).
    wavelength_m: float = 1550e-9
    satellite_altitude_m: float = 620e3
    receiver_altitude_m: float = 0.0
    earth_radius_m: float = 6371e3
    earth_gravitational_parameter: float = 3.986004418e14
    light_speed_m_s: float = 299792458.0
    planck_constant_j_s: float = 6.62607015e-34
    electron_charge_c: float = 1.602176634e-19
    beam_radius_m: float = 0.05
    receiver_aperture_m: float = 0.318
    rms_wind_speed_m_s: float = 21.0
    ground_cn2_m_neg_two_thirds: float = 1.7e-14
    atmospheric_range_m: float = 20e3
    fog_range_m: float = 8e3
    field_of_view_rad: float = 0.1
    optical_filter_bandwidth_um: float = 1e-3  # 1 nm; not disclosed.
    symbol_rate_baud: float = 28e9
    amplifier_gain_db: float = 30.0
    amplifier_noise_figure_db: float = 4.0
    lo_linewidth_hz: float = 1000.0
    photodiode_responsivity_a_w: float = 0.85
    doppler_residual_ratio: float = 0.1
    malaga_omega: float = 0.5
    malaga_b0: float = 0.25
    malaga_phase_difference_rad: float = math.pi / 2
    malaga_beta: object = None
    max_derived_beta: int = 1000
    beta_fit_policy: str = "nearest_warn"
    beta_fit_relative_tolerance: float = 1e-3
    integration_points: int = 4097
    integration_grid_power: float = 2.0
    num_receivers: int = 1
    combining: str = "MRC"
    include_turbulence: bool = True
    include_background_noise: bool = True
    include_ase_noise: bool = True
    include_shot_noise: bool = True
    heterodyne_shot_noise_convention: float = 1.0
    include_doppler: bool = True
    include_phase_noise: bool = True

    # Convenience/reference condition; this combination is not a paper-wide default.
    launch_power_dbm: float = 33.0
    zenith_angle_deg: float = 30.0
    pointing_error_urad: float = 20.0
    scattered_power_ratio: float = 0.95
    background_light_radiance_w_m2_sr_um: float = 1e-4
    weather: str = "clear"

    def __post_init__(self):
        if self.integration_points < 257 or self.integration_points % 2 == 0:
            raise ValueError("integration_points must be an odd integer >= 257")
        if self.integration_grid_power < 1.0:
            raise ValueError("integration_grid_power must be at least 1")
        if self.photodiode_responsivity_a_w <= 0:
            raise ValueError("Photodiode responsivity must be positive")
        quantum_efficiency = (
            self.photodiode_responsivity_a_w * self.planck_constant_j_s
            * self.light_speed_m_s / self.wavelength_m / self.electron_charge_c
        )
        if not 0 < quantum_efficiency <= 1:
            raise ValueError("Responsivity implies unphysical photodiode quantum efficiency")
        if self.heterodyne_shot_noise_convention <= 0:
            raise ValueError("Heterodyne complex-noise convention factor must be positive")
        if self.num_receivers < 1:
            raise ValueError("num_receivers must be positive")
        if self.combining.upper() not in {"MRC", "EGC"}:
            raise ValueError("combining must be MRC or EGC")
        if not math.isclose(self.malaga_omega + 2 * self.malaga_b0, 1.0):
            raise ValueError("The paper normalizes Malaga power as Omega + 2*b0 = 1")
        if self.beta_fit_policy not in {"nearest_warn", "strict"}:
            raise ValueError("beta_fit_policy must be nearest_warn or strict")
        if self.beta_fit_relative_tolerance < 0:
            raise ValueError("beta_fit_relative_tolerance must be nonnegative")


def _trapezoid(values, grid):
    if hasattr(np, "trapezoid"):
        return np.trapezoid(values, grid)
    return np.trapz(values, grid)


def hufnagel_valley_cn2(altitude_m, config):
    h = np.asarray(altitude_m, dtype=np.float64)
    v = config.rms_wind_speed_m_s
    a = config.ground_cn2_m_neg_two_thirds
    return (
        0.00594 * (v / 27.0) ** 2 * (1e-5 * h) ** 10 * np.exp(-h / 1000.0)
        + 2.7e-16 * np.exp(-h / 1500.0)
        + a * np.exp(-h / 100.0)
    )


def satellite_slant_range_m(zenith_angle_deg, config):
    phi = math.radians(float(zenith_angle_deg))
    if not 0.0 <= phi < math.pi / 2:
        raise ValueError("zenith_angle_deg must be in [0, 90)")
    ground_radius = config.earth_radius_m + config.receiver_altitude_m
    satellite_radius = config.earth_radius_m + config.satellite_altitude_m
    return (
        math.sqrt(
            satellite_radius**2
            - ground_radius**2 * math.sin(phi) ** 2
        )
        - ground_radius * math.cos(phi)
    )


def path_altitude_m(path_fraction, slant_range_m, zenith_angle_deg, config):
    xi = np.asarray(path_fraction, dtype=np.float64)
    phi = math.radians(float(zenith_angle_deg))
    ground_radius = config.earth_radius_m + config.receiver_altitude_m
    distance = xi * slant_range_m
    radius = np.sqrt(
        ground_radius**2
        + distance**2
        + 2.0 * ground_radius * distance * math.cos(phi)
    )
    return radius - config.earth_radius_m


def shadowed_rician_second_moment(beta, omega_prime, gamma):
    mean_y = omega_prime + gamma
    beta_independent = omega_prime**2 + 4 * omega_prime * gamma + 2 * gamma**2
    return (
        beta_independent + omega_prime**2 / float(beta)
    ) / mean_y**2


def fit_malaga_beta_eq15(target_second_moment, omega_prime, gamma, max_beta):
    """Nearest natural beta plus an honest Eq.-(15) feasibility diagnostic."""
    mean_y = omega_prime + gamma
    beta_independent = omega_prime**2 + 4 * omega_prime * gamma + 2 * gamma**2
    denominator = target_second_moment * mean_y**2 - beta_independent
    continuous_beta = (
        omega_prime**2 / denominator if denominator > 0 else float("inf")
    )
    continuum_feasible = 1.0 <= continuous_beta <= float(max_beta)
    if math.isfinite(continuous_beta):
        candidates = {
            1,
            int(max_beta),
            min(max(int(math.floor(continuous_beta)), 1), int(max_beta)),
            min(max(int(math.ceil(continuous_beta)), 1), int(max_beta)),
        }
    else:
        candidates = {int(max_beta)}
    beta = min(
        candidates,
        key=lambda value: abs(
            shadowed_rician_second_moment(value, omega_prime, gamma)
            - target_second_moment
        ),
    )
    fitted = shadowed_rician_second_moment(beta, omega_prime, gamma)
    relative_residual = abs(fitted - target_second_moment) / max(
        abs(target_second_moment), 1e-30
    )
    return {
        "beta": int(beta),
        "continuous_beta": continuous_beta,
        "continuum_feasible_within_search": continuum_feasible,
        "fitted_second_moment": fitted,
        "relative_residual": relative_residual,
        "minimum_second_moment": beta_independent / mean_y**2,
        "maximum_natural_second_moment": shadowed_rician_second_moment(
            1, omega_prime, gamma
        ),
    }


def compute_paper_link_budget(
    config,
    launch_power_dbm=None,
    zenith_angle_deg=None,
    pointing_error_urad=None,
    scattered_power_ratio=None,
    background_light_radiance_w_m2_sr_um=None,
    weather=None,
):
    """Implement paper Eqs. (4)-(18) and Table I as scalar diagnostics."""
    launch_power_dbm = (
        config.launch_power_dbm if launch_power_dbm is None else float(launch_power_dbm)
    )
    zenith_angle_deg = (
        config.zenith_angle_deg if zenith_angle_deg is None else float(zenith_angle_deg)
    )
    pointing_error_urad = (
        config.pointing_error_urad
        if pointing_error_urad is None
        else float(pointing_error_urad)
    )
    rho = (
        config.scattered_power_ratio
        if scattered_power_ratio is None
        else float(scattered_power_ratio)
    )
    radiance = (
        config.background_light_radiance_w_m2_sr_um
        if background_light_radiance_w_m2_sr_um is None
        else float(background_light_radiance_w_m2_sr_um)
    )
    weather = (config.weather if weather is None else weather).lower()
    if weather not in WEATHER_ATTENUATION_DB_PER_KM:
        raise ValueError(f"Unsupported weather: {weather}")
    if not 0.0 <= rho <= 1.0:
        raise ValueError("scattered_power_ratio must be in [0, 1]")
    if radiance < 0.0:
        raise ValueError("background light radiance cannot be negative")

    wavelength = config.wavelength_m
    wavenumber = 2.0 * math.pi / wavelength
    slant_range = satellite_slant_range_m(zenith_angle_deg, config)

    theta0 = 1.0  # Collimated input beam: F0 -> infinity.
    lambda0 = 2.0 * slant_range / (wavenumber * config.beam_radius_m**2)
    denominator = theta0**2 + lambda0**2
    theta1 = theta0 / denominator
    lambda1 = lambda0 / denominator
    diffraction_radius = config.beam_radius_m / math.sqrt(theta1**2 + lambda1**2)

    # The Hufnagel-Valley ground term decays on a ~100-m altitude scale. A
    # uniform 4097-point slant grid undersamples it, so cluster samples near xi=0.
    xi = np.linspace(
        0.0, 1.0, config.integration_points, dtype=np.float64
    ) ** config.integration_grid_power
    altitude = path_altitude_m(xi, slant_range, zenith_angle_deg, config)
    cn2 = hufnagel_valley_cn2(altitude, config)
    mu_2d = float(_trapezoid(cn2 * xi ** (5.0 / 3.0), xi))
    spreading_factor = (
        4.35
        * mu_2d
        * lambda1 ** (5.0 / 6.0)
        * wavenumber ** (7.0 / 6.0)
        * slant_range ** (11.0 / 6.0)
    )
    long_term_radius = diffraction_radius * math.sqrt(1.0 + spreading_factor)

    geometric_scintillation_db = -10.0 * math.log10(
        config.receiver_aperture_m**2 / (2.0 * long_term_radius**2)
    )
    pointing_offset_m = slant_range * pointing_error_urad * 1e-6
    pointing_db = (10.0 / math.log(10.0)) * (
        2.0 * pointing_offset_m**2 / long_term_radius**2
    )
    weather_range_m = config.fog_range_m if weather == "fog" else config.atmospheric_range_m
    atmospheric_db = WEATHER_ATTENUATION_DB_PER_KM[weather] * weather_range_m / 1000.0
    total_attenuation_db = geometric_scintillation_db + pointing_db + atmospheric_db

    complex_kernel = (
        xi ** (5.0 / 6.0)
        * (
            lambda1 * xi
            + 1j * (1.0 - (1.0 - theta1) * xi)
        ) ** (5.0 / 6.0)
        - lambda1 ** (5.0 / 6.0) * xi ** (5.0 / 3.0)
    )
    rytov_variance = float(
        8.7
        * wavenumber ** (7.0 / 6.0)
        * slant_range ** (11.0 / 6.0)
        * np.real(_trapezoid(cn2 * complex_kernel, xi))
    )
    rytov_variance = max(rytov_variance, 0.0)
    sigma_lnx2 = (
        0.49
        * rytov_variance
        / (
            1.0
            + (1.0 + theta1) * 0.56 * rytov_variance ** (6.0 / 5.0)
        ) ** (7.0 / 6.0)
    )
    sigma_lny2 = (
        0.51
        * rytov_variance
        / (1.0 + 0.69 * rytov_variance ** (6.0 / 5.0)) ** (5.0 / 6.0)
    )
    alpha = 1.0 / math.expm1(max(sigma_lnx2, 1e-12))

    omega = config.malaga_omega
    b0 = config.malaga_b0
    phase_difference = config.malaga_phase_difference_rad
    omega_prime = (
        omega
        + 2.0 * b0 * rho
        + 2.0 * math.sqrt(2.0 * b0 * omega * rho) * math.cos(phase_difference)
    )
    gamma = 2.0 * b0 * (1.0 - rho)
    target_y_second_moment = math.exp(sigma_lny2)
    if config.malaga_beta is None:
        beta_fit = fit_malaga_beta_eq15(
            target_y_second_moment,
            omega_prime,
            gamma,
            config.max_derived_beta,
        )
        beta = beta_fit["beta"]
        beta_fit_feasible = (
            beta_fit["continuum_feasible_within_search"]
            and beta_fit["relative_residual"]
            <= config.beta_fit_relative_tolerance
        )
        beta_source = "eq15_nearest_natural"
        if config.beta_fit_policy == "strict" and not beta_fit_feasible:
            raise ValueError(
                "Paper-undisclosed Malaga beta/phase cannot satisfy Eq. (15) "
                f"within tolerance: target={target_y_second_moment:.8g}, "
                f"nearest={beta_fit['fitted_second_moment']:.8g}, "
                f"relative residual={beta_fit['relative_residual']:.3%}"
            )
    else:
        beta = int(config.malaga_beta)
        if beta < 1:
            raise ValueError("malaga_beta must be a natural number")
        fitted = shadowed_rician_second_moment(beta, omega_prime, gamma)
        beta_fit = {
            "beta": beta,
            "continuous_beta": None,
            "continuum_feasible_within_search": None,
            "fitted_second_moment": fitted,
            "relative_residual": abs(fitted - target_y_second_moment)
            / max(abs(target_y_second_moment), 1e-30),
            "minimum_second_moment": None,
            "maximum_natural_second_moment": None,
        }
        beta_fit_feasible = None
        beta_source = "explicit_user_assumption"
    fitted_y_second_moment = beta_fit["fitted_second_moment"]

    earth_radius = config.earth_radius_m
    orbit_radius = earth_radius + config.satellite_altitude_m
    orbital_angular_velocity = math.sqrt(
        config.earth_gravitational_parameter / orbit_radius**3
    )
    phi = math.radians(zenith_angle_deg)
    phi_max = math.pi / 2.0
    eta = math.cos(
        math.acos(earth_radius / orbit_radius * math.cos(phi_max)) - phi_max
    )
    optical_frequency_hz = config.light_speed_m_s / wavelength
    doppler_denominator = math.sqrt(
        earth_radius**2
        + orbit_radius**2
        - 2.0 * earth_radius * orbit_radius * math.cos(phi) * eta
    )
    doppler_hz = -(
        optical_frequency_hz
        / config.light_speed_m_s
        * orbital_angular_velocity
        * earth_radius
        * orbit_radius
        * math.sin(phi)
        * eta
        / doppler_denominator
    )
    residual_doppler_hz = config.doppler_residual_ratio * doppler_hz

    background_noise_w = (
        radiance
        * math.pi**2
        * config.field_of_view_rad**2
        * config.receiver_aperture_m**2
        / 16.0
        * config.optical_filter_bandwidth_um
    )
    amplifier_gain_linear = 10.0 ** (config.amplifier_gain_db / 10.0)
    noise_factor = 10.0 ** (config.amplifier_noise_figure_db / 10.0)
    spontaneous_emission_factor = (
        noise_factor - 1.0 / amplifier_gain_linear
    ) / (2.0 * (1.0 - 1.0 / amplifier_gain_linear))
    ase_noise_w = (
        2.0
        * spontaneous_emission_factor
        * config.planck_constant_j_s
        * optical_frequency_hz
        * (amplifier_gain_linear - 1.0)
        * config.symbol_rate_baud
    )

    # One detected complex temporal mode per Nyquist symbol. Equivalent
    # input field noise is independent between receiver branches, not common
    # transmitter ASE. No unknown LO power or arbitrary thermal floor is fitted.
    quantum_efficiency = (
        config.photodiode_responsivity_a_w * config.planck_constant_j_s
        * optical_frequency_hz / config.electron_charge_c
    )
    receiver_shot_noise_w = (
        config.heterodyne_shot_noise_convention * config.electron_charge_c
        / config.photodiode_responsivity_a_w * config.symbol_rate_baud
    )

    return {
        "channel_version": PAPER_FSO_CHANNEL_VERSION,
        "launch_power_dbm": launch_power_dbm,
        "launch_power_w": 10.0 ** ((launch_power_dbm - 30.0) / 10.0),
        "zenith_angle_deg": zenith_angle_deg,
        "pointing_error_urad": pointing_error_urad,
        "scattered_power_ratio": rho,
        "background_light_radiance_w_m2_sr_um": radiance,
        "weather": weather,
        "slant_range_m": slant_range,
        "theta1": theta1,
        "lambda1": lambda1,
        "diffraction_radius_m": diffraction_radius,
        "long_term_radius_m": long_term_radius,
        "geometric_scintillation_db": geometric_scintillation_db,
        "pointing_db": pointing_db,
        "atmospheric_db": atmospheric_db,
        "total_attenuation_db": total_attenuation_db,
        "rytov_variance": rytov_variance,
        "sigma_lnx2": sigma_lnx2,
        "sigma_lny2": sigma_lny2,
        "malaga_alpha": alpha,
        "malaga_beta": beta,
        "malaga_beta_source": beta_source,
        "malaga_beta_fit_feasible": beta_fit_feasible,
        "malaga_beta_continuous": beta_fit["continuous_beta"],
        "malaga_beta_fit_relative_residual": beta_fit["relative_residual"],
        "malaga_beta_minimum_second_moment": beta_fit["minimum_second_moment"],
        "malaga_beta_maximum_natural_second_moment": beta_fit[
            "maximum_natural_second_moment"
        ],
        "malaga_omega_prime": omega_prime,
        "malaga_gamma": gamma,
        "malaga_mean_y": omega_prime + gamma,
        "target_y_second_moment": target_y_second_moment,
        "fitted_y_second_moment": fitted_y_second_moment,
        "doppler_hz": doppler_hz,
        "residual_doppler_hz": residual_doppler_hz,
        "background_noise_w": background_noise_w,
        "amplifier_gain_linear": amplifier_gain_linear,
        "ase_noise_w": ase_noise_w,
        "photodiode_quantum_efficiency": quantum_efficiency,
        "receiver_shot_noise_w": receiver_shot_noise_w,
        "complex_shot_noise_convention": config.heterodyne_shot_noise_convention,
    }


def _real_dtype(complex_dtype):
    if complex_dtype == torch.complex128:
        return torch.float64
    return torch.float32


def _complex_normal(shape, device, complex_dtype, generator=None):
    real_dtype = _real_dtype(complex_dtype)
    real = torch.randn(shape, device=device, dtype=real_dtype, generator=generator)
    imag = torch.randn(shape, device=device, dtype=real_dtype, generator=generator)
    return torch.complex(real, imag) / math.sqrt(2.0)


def _gamma_unit_mean(shape_parameter, shape, device, dtype, generator=None):
    concentration = torch.full(
        shape,
        float(shape_parameter),
        device=device,
        dtype=dtype,
    )
    # PyTorch distributions do not expose a generator argument. This native
    # primitive does and is available in the paper's PyTorch 2.0 runtime.
    return torch._standard_gamma(concentration, generator=generator) / float(
        shape_parameter
    )


class PaperMalagaFSOChannel:
    """Paper Eq. (3) channel with Eqs. (4)-(18) and ideal coherent DSP.

    This is a symbol-rate matched-filter equivalent. Block Malaga fading is
    sampled independently per image/receiver. Doppler and LO phase are applied
    before ideal CFR/CPR/equalization through the returned complex channel gain.
    """

    def __init__(self, config=None):
        self.config = config or PaperFSOSystemConfig()
        self._budget_cache = {}
        self._reproducibility_config_cache = None
        self._warned_infeasible_beta_fit = False

    def reproducibility_config(self):
        if self._reproducibility_config_cache is None:
            reference = compute_paper_link_budget(self.config)
            self._reproducibility_config_cache = {
                "channel_version": PAPER_FSO_CHANNEL_VERSION,
                "system": asdict(self.config),
                "undisclosed_assumptions": PAPER_UNDISCLOSED_RECEIVER_ASSUMPTIONS,
                "reference_beta_diagnostic": {
                    "beta": reference["malaga_beta"],
                    "source": reference["malaga_beta_source"],
                    "fit_feasible": reference["malaga_beta_fit_feasible"],
                    "target_second_moment": round(
                        reference["target_y_second_moment"], 12
                    ),
                    "fitted_second_moment": round(
                        reference["fitted_y_second_moment"], 12
                    ),
                    "relative_residual": round(
                        reference["malaga_beta_fit_relative_residual"], 12
                    ),
                },
            }
        return self._reproducibility_config_cache

    def _budget(self, **condition):
        key = tuple(sorted(condition.items()))
        if key not in self._budget_cache:
            self._budget_cache[key] = compute_paper_link_budget(
                self.config, **condition
            )
        budget = self._budget_cache[key]
        if (
            self.config.malaga_beta is None
            and self.config.beta_fit_policy == "nearest_warn"
            and not budget["malaga_beta_fit_feasible"]
            and not self._warned_infeasible_beta_fit
        ):
            warnings.warn(
                "The paper omits Malaga beta and phase, and Eq. (15) has no "
                "fit within the configured range/tolerance for this condition; "
                f"using beta={budget['malaga_beta']} with relative moment "
                f"residual={budget['malaga_beta_fit_relative_residual']:.3%}. "
                "Set an explicit malaga_beta or beta_fit_policy='strict'.",
                RuntimeWarning,
            )
            self._warned_infeasible_beta_fit = True
        return budget

    def sample_malaga_power(self, shape, budget, device, dtype, generator=None):
        if not self.config.include_turbulence:
            return torch.ones(shape, device=device, dtype=dtype)
        large_scale = _gamma_unit_mean(
            budget["malaga_alpha"], shape, device, dtype, generator
        )
        shadowing = _gamma_unit_mean(
            budget["malaga_beta"], shape, device, dtype, generator
        )
        rho = budget["scattered_power_ratio"]
        coherent_phasor = (
            math.sqrt(self.config.malaga_omega)
            + math.sqrt(2.0 * self.config.malaga_b0 * rho)
            * complex(
                math.cos(-self.config.malaga_phase_difference_rad),
                math.sin(-self.config.malaga_phase_difference_rad),
            )
        )
        diffuse = _complex_normal(
            shape,
            device,
            torch.complex128 if dtype == torch.float64 else torch.complex64,
            generator,
        ) * math.sqrt(budget["malaga_gamma"])
        small_field = shadowing.sqrt().to(diffuse.dtype) * coherent_phasor + diffuse
        small_power = small_field.abs().square().to(dtype)
        return large_scale * small_power / budget["malaga_mean_y"]

    @torch.no_grad()
    def __call__(
        self,
        symbols,
        launch_power_dbm=None,
        zenith_angle_deg=None,
        pointing_error_urad=None,
        scattered_power_ratio=None,
        background_light_radiance_w_m2_sr_um=None,
        weather=None,
        generator=None,
    ):
        if symbols.ndim != 2 or not torch.is_complex(symbols):
            raise ValueError("symbols must be complex baseband with shape [batch, symbols]")
        if generator is not None and not paper_generator_device_matches(
            generator, symbols.device
        ):
            raise ValueError("generator and symbols must use the same device")

        condition = {
            "launch_power_dbm": (
                self.config.launch_power_dbm
                if launch_power_dbm is None
                else float(launch_power_dbm)
            ),
            "zenith_angle_deg": (
                self.config.zenith_angle_deg
                if zenith_angle_deg is None
                else float(zenith_angle_deg)
            ),
            "pointing_error_urad": (
                self.config.pointing_error_urad
                if pointing_error_urad is None
                else float(pointing_error_urad)
            ),
            "scattered_power_ratio": (
                self.config.scattered_power_ratio
                if scattered_power_ratio is None
                else float(scattered_power_ratio)
            ),
            "background_light_radiance_w_m2_sr_um": (
                self.config.background_light_radiance_w_m2_sr_um
                if background_light_radiance_w_m2_sr_um is None
                else float(background_light_radiance_w_m2_sr_um)
            ),
            "weather": self.config.weather if weather is None else str(weather),
        }
        budget = self._budget(**condition)
        batch, symbol_count = symbols.shape
        receivers = self.config.num_receivers
        real_dtype = _real_dtype(symbols.dtype)
        fading_shape = (batch, receivers, 1)
        scintillation_power = self.sample_malaga_power(
            fading_shape,
            budget,
            symbols.device,
            real_dtype,
            generator,
        )

        attenuation_amplitude = 10.0 ** (-budget["total_attenuation_db"] / 20.0)
        # Table I reports transmitted optical power after the transmitter AMP.
        # Therefore its 30-dB gain must not be applied to the signal a second time.
        propagation_amplitude = attenuation_amplitude * scintillation_power.sqrt()

        time_index = torch.arange(
            symbol_count, device=symbols.device, dtype=real_dtype
        )
        phase = torch.zeros(
            (batch, receivers, symbol_count),
            device=symbols.device,
            dtype=real_dtype,
        )
        if self.config.include_doppler:
            phase = phase + (
                2.0
                * math.pi
                * budget["residual_doppler_hz"]
                / self.config.symbol_rate_baud
                * time_index
            ).view(1, 1, -1)
        if self.config.include_phase_noise and self.config.lo_linewidth_hz > 0:
            increment_std = math.sqrt(
                2.0
                * math.pi
                * self.config.lo_linewidth_hz
                / self.config.symbol_rate_baud
            )
            increments = torch.randn(
                phase.shape,
                device=symbols.device,
                dtype=real_dtype,
                generator=generator,
            ) * increment_std
            phase = phase + increments.cumsum(dim=-1)

        propagation_gain = propagation_amplitude.to(symbols.dtype) * torch.exp(
            1j * phase
        ).to(symbols.dtype)
        channel_gain = math.sqrt(budget["launch_power_w"]) * propagation_gain
        received = channel_gain * symbols[:, None, :]

        if self.config.include_background_noise and budget["background_noise_w"] > 0:
            received = received + _complex_normal(
                received.shape, symbols.device, symbols.dtype, generator
            ) * math.sqrt(budget["background_noise_w"])
        if self.config.include_shot_noise and budget["receiver_shot_noise_w"] > 0:
            received = received + _complex_normal(
                received.shape, symbols.device, symbols.dtype, generator
            ) * math.sqrt(budget["receiver_shot_noise_w"])
        if self.config.include_ase_noise and budget["ase_noise_w"] > 0:
            transmitter_ase = _complex_normal(
                (batch, 1, symbol_count),
                symbols.device,
                symbols.dtype,
                generator,
            ) * math.sqrt(budget["ase_noise_w"])
            received = received + propagation_gain * transmitter_ase

        # The transmitter-EDFA ASE is one common source waveform, so diversity
        # cannot average it away. Receiver background noise is branch-independent.
        equivalent_noise_variance = torch.zeros(
            (batch, 1), device=symbols.device, dtype=real_dtype
        )
        if self.config.include_ase_noise:
            equivalent_noise_variance = equivalent_noise_variance + (
                budget["ase_noise_w"] / budget["launch_power_w"]
            )
        receiver_independent_noise_w = (
            budget["background_noise_w"] if self.config.include_background_noise else 0.0
        ) + (
            budget["receiver_shot_noise_w"] if self.config.include_shot_noise else 0.0
        )
        if receiver_independent_noise_w > 0:
            if receivers == 1 or self.config.combining.upper() == "MRC":
                combined_signal_power = channel_gain[..., :1].abs().square().sum(dim=1)
                equivalent_noise_variance = equivalent_noise_variance + (
                    receiver_independent_noise_w
                    / combined_signal_power.clamp_min(torch.finfo(real_dtype).tiny)
                )
            else:
                combined_signal_amplitude = channel_gain[..., :1].abs().sum(dim=1)
                equivalent_noise_variance = equivalent_noise_variance + (
                    receivers
                    * receiver_independent_noise_w
                    / combined_signal_amplitude.square().clamp_min(
                        torch.finfo(real_dtype).tiny
                    )
                )
        if bool((equivalent_noise_variance > 0).any()):
            instantaneous_snr = equivalent_noise_variance.clamp_min(
                torch.finfo(real_dtype).tiny
            ).reciprocal()
            snr_linear_sum = float(instantaneous_snr.sum().item())
            snr_sample_count = int(instantaneous_snr.numel())
            mean_snr_linear = snr_linear_sum / max(snr_sample_count, 1)
            mean_snr_db = 10.0 * math.log10(max(mean_snr_linear, 1e-30))
        else:
            snr_linear_sum = float("inf")
            snr_sample_count = batch
            mean_snr_db = float("inf")
        diagnostics = {
            "channel_version": PAPER_FSO_CHANNEL_VERSION,
            "mean_snr_db": mean_snr_db,
            "snr_linear_sum": snr_linear_sum,
            "snr_sample_count": snr_sample_count,
            "num_receivers": receivers,
            "combining": self.config.combining.upper(),
            "budget": budget,
        }

        if receivers == 1:
            return FSOChannelOutput(
                symbols=received[:, 0, :],
                channel_gain=channel_gain[:, 0, :],
                is_equalized=False,
                diagnostics=diagnostics,
            )

        if self.config.combining.upper() == "MRC":
            denominator = (
                channel_gain.abs().square().sum(dim=1)
            ).clamp_min(torch.finfo(real_dtype).tiny)
            combined = (channel_gain.conj() * received).sum(dim=1) / denominator
        else:
            magnitudes = channel_gain.abs().clamp_min(1e-30)
            combined = (
                channel_gain.conj() / magnitudes * received
            ).sum(dim=1) / magnitudes.sum(dim=1).clamp_min(1e-30)
        return FSOChannelOutput(
            symbols=combined,
            is_equalized=True,
            diagnostics=diagnostics,
        )


@dataclass
class PaperFSOIndexChannel:
    receiver: object
    modulations: tuple = ("QPSK", "8QAM", "16QAM")
    bits_per_index: int = BITS_PER_INDEX

    @property
    def receiver_config(self):
        return self.receiver.reproducibility_config()

    def reproducibility_config(self):
        return {
            "qam_mapping_version": QAM_MAPPING_VERSION,
            "modulations": tuple(name.upper() for name in self.modulations),
            "bits_per_index": self.bits_per_index,
            "receiver": self.receiver_config,
        }

    def __call__(
        self,
        indices,
        modulation="QPSK",
        return_statistics=False,
        **channel_kwargs,
    ):
        modulation = modulation.upper()
        if modulation not in tuple(name.upper() for name in self.modulations):
            raise ValueError(f"Unsupported paper modulation: {modulation}")
        link = QAMIndexLink(
            self.receiver,
            modulation=modulation,
            bits_per_index=self.bits_per_index,
            receiver_config=self.receiver_config,
        )
        return link(
            indices,
            return_statistics=return_statistics,
            **channel_kwargs,
        )


def paper_fig4_conditions():
    return [
        {
            "label": f"{modulation}, Pt={power} dBm, zenith={zenith} deg",
            "channel_kwargs": {
                "modulation": modulation,
                "launch_power_dbm": float(power),
                "zenith_angle_deg": float(zenith),
                "pointing_error_urad": 20.0,
                "scattered_power_ratio": 0.95,
                "background_light_radiance_w_m2_sr_um": 1e-4,
                "weather": "clear",
            },
        }
        for power in (30, 33, 36)
        for modulation in ("QPSK", "8QAM", "16QAM")
        for zenith in range(0, 81, 10)
    ]


def paper_fig6_pointing_conditions():
    return [
        {
            "label": (
                f"{modulation}, Pt=36 dBm, zenith={zenith} deg, "
                f"pointing={pointing} urad"
            ),
            "channel_kwargs": {
                "modulation": modulation,
                "launch_power_dbm": 36.0,
                "zenith_angle_deg": float(zenith),
                "pointing_error_urad": float(pointing),
                "scattered_power_ratio": 0.95,
                "background_light_radiance_w_m2_sr_um": 1e-4,
                "weather": "clear",
            },
        }
        for modulation in ("QPSK", "8QAM", "16QAM")
        for zenith in (0, 30, 60)
        for pointing in range(18, 23)
    ]


def paper_fig6_background_conditions():
    return [
        {
            "label": (
                f"{modulation}, Pt={power} dBm, "
                f"BGL={radiance:g} W/m2/sr/um"
            ),
            "channel_kwargs": {
                "modulation": modulation,
                "launch_power_dbm": float(power),
                "zenith_angle_deg": 0.0,
                "pointing_error_urad": 19.0,
                "scattered_power_ratio": 0.95,
                "background_light_radiance_w_m2_sr_um": float(radiance),
                "weather": "clear",
            },
        }
        for modulation in ("QPSK", "8QAM", "16QAM")
        for radiance in (1e-4, 1e-2, 1.0)
        for power in range(28, 37)
    ]


def paper_fig7_weather_conditions():
    return [
        {
            "label": f"QPSK, Pt={power} dBm, weather={weather}",
            "channel_kwargs": {
                "modulation": "QPSK",
                "launch_power_dbm": float(power),
                "zenith_angle_deg": 0.0,
                "pointing_error_urad": 15.0,
                "scattered_power_ratio": 0.95,
                "background_light_radiance_w_m2_sr_um": 1e-4,
                "weather": weather,
            },
        }
        for weather in ("clear", "dust", "rain", "snow", "fog")
        for power in range(28, 37)
    ]


def paper_fig8_turbulence_conditions():
    """Return the common FSO-SC rho/Pt sweep, not the BPG codec pipelines."""
    return [
        {
            "label": f"16QAM, Pt={power} dBm, rho={rho}",
            "channel_kwargs": {
                "modulation": "16QAM",
                "launch_power_dbm": float(power),
                "zenith_angle_deg": 0.0,
                "pointing_error_urad": 19.0,
                "scattered_power_ratio": float(rho),
                "background_light_radiance_w_m2_sr_um": 1e-4,
                "weather": "clear",
            },
        }
        for rho in (0.95, 0.75, 0.596)
        for power in range(28, 37)
    ]


def paper_fig8_diversity_index_channels(base_config=None):
    """Expose the paper's 2/4-Rx EGC/MRC comparisons.

    The paper does not disclose inter-branch turbulence correlation, so the
    receiver implementation records its independent-block-fade assumption.
    """
    base_config = PaperFSOSystemConfig() if base_config is None else base_config
    channels = {
        "1Rx": PaperFSOIndexChannel(
            PaperMalagaFSOChannel(replace(base_config, num_receivers=1))
        )
    }
    for receivers in (2, 4):
        for combining in ("EGC", "MRC"):
            label = f"{receivers}Rx-{combining}"
            receiver_config = replace(
                base_config,
                num_receivers=receivers,
                combining=combining,
            )
            channels[label] = PaperFSOIndexChannel(
                PaperMalagaFSOChannel(receiver_config)
            )
    return channels


def paper_fso_equation_self_test():
    config = PaperFSOSystemConfig()
    reference = compute_paper_link_budget(
        config,
        launch_power_dbm=33.0,
        zenith_angle_deg=0.0,
        pointing_error_urad=20.0,
        weather="clear",
    )
    assert abs(reference["slant_range_m"] - 620e3) < 1e-6
    assert abs(reference["geometric_scintillation_db"] - 28.69) < 0.05
    assert abs(reference["pointing_db"] - 35.68) < 0.05
    assert abs(reference["atmospheric_db"] - 2.94) < 1e-9
    # The paper omits beta and the coherent phase. With the documented pi/2
    # phase, its Eq. (15) has no feasible beta here; this must stay explicit.
    assert reference["malaga_beta_fit_feasible"] is False
    assert reference["malaga_beta_fit_relative_residual"] > 0.0
    try:
        compute_paper_link_budget(
            PaperFSOSystemConfig(beta_fit_policy="strict"),
            zenith_angle_deg=0.0,
            scattered_power_ratio=0.95,
        )
    except ValueError:
        pass
    else:
        raise AssertionError("Strict beta fitting must reject the underdetermined fit")
    expected_ranges_km = {
        0: 620.000,
        20: 655.958,
        30: 705.628,
        60: 1107.733,
        80: 1977.252,
    }
    for zenith, expected_km in expected_ranges_km.items():
        actual_km = satellite_slant_range_m(zenith, config) / 1000.0
        assert abs(actual_km - expected_km) < 0.001
    expected_weather_db = {
        "clear": 2.94,
        "dust": 20.86,
        "rain": 28.06,
        "snow": 42.90,
        "fog": 271.832,
    }
    for weather, expected_db in expected_weather_db.items():
        weather_budget = compute_paper_link_budget(config, weather=weather)
        assert abs(weather_budget["atmospheric_db"] - expected_db) < 0.001
    peak = compute_paper_link_budget(config, zenith_angle_deg=20.0)
    high = compute_paper_link_budget(config, zenith_angle_deg=80.0)
    assert abs(peak["doppler_hz"]) > abs(high["doppler_hz"])

    convergence_reference = compute_paper_link_budget(
        PaperFSOSystemConfig(integration_points=32769),
        zenith_angle_deg=0.0,
        pointing_error_urad=20.0,
    )
    assert abs(
        reference["rytov_variance"] / convergence_reference["rytov_variance"] - 1.0
    ) < 5e-3
    assert abs(
        reference["malaga_alpha"] / convergence_reference["malaga_alpha"] - 1.0
    ) < 5e-3

    receiver = PaperMalagaFSOChannel(config)
    generator_a = torch.Generator().manual_seed(123)
    generator_b = torch.Generator().manual_seed(123)
    sample_a = receiver.sample_malaga_power(
        (4096, 1, 1), reference, torch.device("cpu"), torch.float32, generator_a
    )
    sample_b = receiver.sample_malaga_power(
        (4096, 1, 1), reference, torch.device("cpu"), torch.float32, generator_b
    )
    assert torch.equal(sample_a, sample_b)
    assert bool((sample_a >= 0).all()) and bool(torch.isfinite(sample_a).all())
    assert 0.9 < float(sample_a.mean()) < 1.1
    print(
        "Paper FSO Eq. (3)-(18) self-test passed: "
        f"asg={reference['geometric_scintillation_db']:.2f} dB, "
        f"ape={reference['pointing_db']:.2f} dB, "
        f"alpha={reference['malaga_alpha']:.2f}, beta={reference['malaga_beta']} "
        f"(Eq15 residual={reference['malaga_beta_fit_relative_residual']:.2%})"
    )


paper_fso_equation_self_test()


In [ ]:
"""Signal-estimated symbol-rate coherent carrier recovery for the FSO notebook.

Embed after ``qam_index_link`` and ``paper_fso_channel_v1``. This is a
plausible receiver implementation, NOT the undisclosed VPI DSP used by Chen
et al. It introduces no pilot symbols and never uses transmitted symbol values
or the simulated channel phase to recover the carrier. The default magnitude
mode retains explicitly assumed block-amplitude CSI; ``blind_rms`` is also
available, but its uniform-symbol assumption is unsuitable for strongly biased
semantic bitstreams.

Reference for feed-forward blind phase search: Pfau, Hoffmann, and Noe,
JLT 27(8), 989-999 (2009), doi:10.1109/JLT.2008.2010511. Fourth-power spectral
CFR is a conventional modulation-removal approximation, with its finite-frame
failure modes retained. Exact frame/timing/IQ calibration and an initial phase
branch near zero remain assumptions. No RRC/ADC waveform is synthesized here.
"""

from dataclasses import asdict, dataclass
import math

import torch
import torch.nn.functional as F


SIGNAL_DSP_VERSION = "fourth_power_fft_newton_bps_v1"
SIGNAL_DSP_REFERENCE = {
    "blind_phase_search": "https://doi.org/10.1109/JLT.2008.2010511",
    "scope": "symbol-rate, single receiver; no waveform/timing/IQ calibration model",
    "cfr": "fourth-power FFT peak plus continuous periodogram Newton refinement",
    "cpr": "windowed minimum-distance blind phase search and symmetry unwrapping",
    "phase_ambiguity": (
        "initial carrier phase lies in the branch nearest zero, consistent with "
        "the simulator phase origin; no transmitted data is used to select a branch"
    ),
    "no_pilots": True,
    "amplitude_default": "known block magnitude only, not the complex channel gain",
    "failure_policy": "keep estimates/errors; do not drop low-SNR frames",
}


@dataclass(frozen=True)
class SignalDSPConfig:
    amplitude_mode: str = "known_block_magnitude"
    cfo_fft_oversampling: int = 4
    cfo_newton_steps: int = 4
    bps_test_phases: int = 64
    bps_window_symbols: int = 65
    cfo_enabled: bool = True
    cpr_enabled: bool = True

    def __post_init__(self):
        if self.amplitude_mode not in {"known_block_magnitude", "blind_rms"}:
            raise ValueError("Unsupported amplitude_mode")
        if self.cfo_fft_oversampling < 1 or self.cfo_newton_steps < 0:
            raise ValueError("Invalid CFR FFT/refinement configuration")
        if self.bps_test_phases < 8:
            raise ValueError("Use at least eight BPS trial phases")
        if self.bps_window_symbols < 1 or self.bps_window_symbols % 2 != 1:
            raise ValueError("BPS window must be a positive odd integer")


def _qam_axis_levels(modulation, device, dtype):
    modulation = modulation.upper()
    if modulation == "QPSK":
        i = q = torch.tensor([-1.0, 1.0], device=device, dtype=dtype) / math.sqrt(2)
        period = math.pi / 2
    elif modulation == "16QAM":
        i = q = torch.tensor([-3.0, -1.0, 1.0, 3.0], device=device, dtype=dtype) / math.sqrt(10)
        period = math.pi / 2
    elif modulation == "8QAM":
        # Rectangular geometry in qam_constellation, NOT square-QAM symmetry.
        i = torch.tensor([-3.0, -1.0, 1.0, 3.0], device=device, dtype=dtype) / math.sqrt(6)
        q = torch.tensor([-1.0, 1.0], device=device, dtype=dtype) / math.sqrt(6)
        period = math.pi
    else:
        raise ValueError(f"Unsupported modulation: {modulation}")
    return i, q, period


def _estimate_cfo_fourth_power(observed, symbol_rate_baud, config):
    batch, count = observed.shape
    # The fourth harmonic repeats at Fs/4, so the unambiguous CFO interval is
    # [-Fs/8, Fs/8). Paper residual offsets are within that acquisition range.
    nfft = 1 << max(1, (count * config.cfo_fft_oversampling - 1).bit_length())
    # Severe fog can make gain-normalized receiver noise enormous. Remove a
    # positive block scale before the fourth power; it does not change the
    # frequency peak and prevents overflow in frames whose data is unrecoverable.
    peak_scale = observed.abs().amax(dim=-1, keepdim=True).clamp_min(
        torch.finfo(observed.real.dtype).tiny
    )
    cfr_samples = observed / peak_scale
    fourth = cfr_samples.square().square()
    spectrum = torch.fft.fft(fourth, n=nfft, dim=-1)
    power = spectrum.abs().square()
    peak = power.argmax(dim=-1)
    signed_peak = torch.where(peak > nfft // 2, peak - nfft, peak)
    real_dtype = observed.real.dtype
    omega = signed_peak.to(real_dtype) * (2.0 * math.pi / nfft)
    index = torch.arange(count, device=observed.device, dtype=real_dtype)
    centered = index - (count - 1) / 2.0
    max_step = 2.0 * math.pi / nfft
    for _ in range(config.cfo_newton_steps):
        derotated = fourth * torch.exp(-1j * omega[:, None] * centered[None, :])
        value = derotated.sum(dim=-1)
        derivative = (-1j * centered[None, :] * derotated).sum(dim=-1)
        curvature = (-centered.square()[None, :] * derotated).sum(dim=-1)
        slope = 2 * (derivative * value.conj()).real
        second = 2 * ((curvature * value.conj()).real + derivative.abs().square())
        stable = second < -torch.finfo(real_dtype).eps
        denominator = torch.where(stable, second, torch.ones_like(second))
        step = torch.where(stable, slope / denominator, torch.zeros_like(slope))
        omega = omega - step.clamp(min=-max_step, max=max_step)
    omega = torch.remainder(omega + math.pi, 2 * math.pi) - math.pi
    estimated_hz = omega * symbol_rate_baud / (8 * math.pi)
    phase = omega[:, None] * index[None, :] / 4.0
    compensated = observed * torch.exp(-1j * phase)
    peak_ratio = power.max(dim=-1).values / power.mean(dim=-1).clamp_min(
        torch.finfo(real_dtype).tiny
    )
    return compensated, estimated_hz, peak_ratio


def _unwrap_symmetry_phase(phase, period):
    differences = torch.diff(phase, dim=-1)
    wrapped = torch.remainder(differences + period / 2, period) - period / 2
    corrections = wrapped - differences
    cumulative = torch.cat(
        [torch.zeros_like(phase[:, :1]), corrections.cumsum(dim=-1)], dim=-1
    )
    return phase + cumulative, (corrections.abs() > period / 2).sum(dim=-1)


def _blind_phase_search(observed, modulation, config):
    i_levels, q_levels, period = _qam_axis_levels(
        modulation, observed.device, observed.real.dtype
    )
    trial_phases = (
        torch.arange(config.bps_test_phases, device=observed.device, dtype=observed.real.dtype)
        / config.bps_test_phases - 0.5
    ) * period
    rotated = observed[:, None, :] * torch.exp(-1j * trial_phases[None, :, None])
    i_distance = (rotated.real[..., None] - i_levels).square().amin(dim=-1)
    q_distance = (rotated.imag[..., None] - q_levels).square().amin(dim=-1)
    costs = F.avg_pool1d(
        i_distance + q_distance,
        kernel_size=config.bps_window_symbols,
        stride=1,
        padding=config.bps_window_symbols // 2,
        count_include_pad=False,
    )
    selected = trial_phases[costs.argmin(dim=1)]
    phase, unwrap_events = _unwrap_symmetry_phase(selected, period)
    # Global rotational ambiguity is not solved using the source symbols. The
    # initial branch near zero is an explicit simulator convention, not a pilot.
    phase = phase - torch.round(phase[:, :1] / period) * period
    corrected = observed * torch.exp(-1j * phase)
    return corrected, phase, unwrap_events


@torch.no_grad()
def recover_qam_carrier(
    received,
    modulation,
    symbol_rate_baud,
    config=None,
    known_block_magnitude=None,
):
    """Estimate CFR/CPR from received samples, without source data/true phase.

    ``known_block_magnitude`` may contain positive block-amplitude CSI only.
    It must not contain the true phase trajectory or the transmitted symbols.
    """
    config = SignalDSPConfig() if config is None else config
    if received.ndim != 2 or not torch.is_complex(received) or received.shape[-1] < 8:
        raise ValueError("Expected complex [batch, symbols>=8] receiver samples")
    if symbol_rate_baud <= 0 or not torch.isfinite(received).all():
        raise ValueError("Invalid receiver samples or symbol rate")
    if config.amplitude_mode == "known_block_magnitude":
        if known_block_magnitude is None or torch.is_complex(known_block_magnitude):
            raise ValueError("Positive REAL block-amplitude CSI is required")
        if not torch.isfinite(known_block_magnitude).all() or (known_block_magnitude <= 0).any():
            raise ValueError("Invalid block-amplitude CSI")
        amplitude = torch.broadcast_to(known_block_magnitude, (received.shape[0], 1))
    else:
        amplitude = received.abs().square().mean(dim=-1, keepdim=True).sqrt()
        amplitude = amplitude.clamp_min(torch.finfo(received.real.dtype).tiny)
    observed = received / amplitude
    zeros = torch.zeros(received.shape[0], device=received.device, dtype=received.real.dtype)
    if config.cfo_enabled:
        observed, cfo_hz, peak_ratio = _estimate_cfo_fourth_power(
            observed, symbol_rate_baud, config
        )
    else:
        cfo_hz, peak_ratio = zeros, zeros
    if config.cpr_enabled:
        observed, phase, unwrap_events = _blind_phase_search(observed, modulation, config)
    else:
        phase = torch.zeros_like(received.real)
        unwrap_events = zeros.to(torch.int64)
    diagnostics = {
        "receiver_version": SIGNAL_DSP_VERSION,
        "amplitude_mode": config.amplitude_mode,
        "estimated_cfo_hz": cfo_hz.detach().cpu().tolist(),
        "cfr_peak_to_mean_power": peak_ratio.detach().cpu().tolist(),
        "cfr_unambiguous_interval_hz": [-symbol_rate_baud / 8, symbol_rate_baud / 8],
        "estimated_phase_start_rad": phase[:, 0].detach().cpu().tolist(),
        "estimated_phase_end_rad": phase[:, -1].detach().cpu().tolist(),
        "phase_symmetry_unwrap_events": unwrap_events.detach().cpu().tolist(),
        "failure_policy": "estimates/errors retained, no frame rejection",
    }
    return observed, diagnostics


@dataclass
class SignalDSPFSOReceiver:
    """Replace phase-oracle equalization for one physical receiver branch."""

    physical_receiver: object
    modulation: str = "QPSK"
    dsp_config: object = None

    def __post_init__(self):
        if self.dsp_config is None:
            self.dsp_config = SignalDSPConfig()
        if self.physical_receiver.config.num_receivers != 1:
            raise ValueError("Signal DSP wrapper supports single Rx only; no oracle MRC")

    def reproducibility_config(self):
        return {
            "physical_receiver": self.physical_receiver.reproducibility_config(),
            "receiver_version": SIGNAL_DSP_VERSION,
            "receiver_config": asdict(self.dsp_config),
            "receiver_assumptions": SIGNAL_DSP_REFERENCE,
            "modulation": self.modulation.upper(),
        }

    @torch.no_grad()
    def __call__(self, symbols, **channel_kwargs):
        observation = self.physical_receiver(symbols, **channel_kwargs)
        if observation.is_equalized:
            raise ValueError("Expected un-equalized physical receiver observation")
        magnitude = None
        if self.dsp_config.amplitude_mode == "known_block_magnitude":
            # Deliberately read only real block magnitude, NOT its phase. Do
            # not pass complex channel_gain to recover_qam_carrier.
            magnitude = observation.channel_gain.abs().mean(dim=-1, keepdim=True)
        recovered, receiver_diagnostics = recover_qam_carrier(
            observation.symbols,
            self.modulation,
            self.physical_receiver.config.symbol_rate_baud,
            self.dsp_config,
            known_block_magnitude=magnitude,
        )
        diagnostics = dict(observation.diagnostics or {})
        diagnostics["signal_dsp"] = receiver_diagnostics
        return FSOChannelOutput(symbols=recovered, is_equalized=True, diagnostics=diagnostics)


@dataclass
class SignalDSPPaperIndexChannel:
    """Drop-in single-Rx alternative to PaperFSOIndexChannel in this notebook."""

    receiver: object
    dsp_config: object = None
    modulations: tuple = ("QPSK", "8QAM", "16QAM")
    bits_per_index: int = 14

    def __post_init__(self):
        if self.dsp_config is None:
            self.dsp_config = SignalDSPConfig()
        if self.receiver.config.num_receivers != 1:
            raise ValueError("Estimated carrier receiver is currently single-Rx only")

    @property
    def receiver_config(self):
        return {
            "physical_receiver": self.receiver.reproducibility_config(),
            "receiver_version": SIGNAL_DSP_VERSION,
            "receiver_config": asdict(self.dsp_config),
            "receiver_assumptions": SIGNAL_DSP_REFERENCE,
        }

    def reproducibility_config(self):
        return {
            "qam_mapping_version": QAM_MAPPING_VERSION,
            "modulations": tuple(name.upper() for name in self.modulations),
            "bits_per_index": self.bits_per_index,
            "receiver": self.receiver_config,
        }

    def __call__(self, indices, modulation="QPSK", return_statistics=False, **channel_kwargs):
        modulation = modulation.upper()
        if modulation not in tuple(name.upper() for name in self.modulations):
            raise ValueError(f"Unsupported modulation: {modulation}")
        receiver = SignalDSPFSOReceiver(self.receiver, modulation, self.dsp_config)
        link = QAMIndexLink(
            receiver,
            modulation=modulation,
            bits_per_index=self.bits_per_index,
            receiver_config=receiver.reproducibility_config(),
        )
        return link(indices, return_statistics=return_statistics, **channel_kwargs)


def signal_dsp_offline_self_test(device="cpu"):
    """Synthetic local checks only; does not download weights or run training."""
    device = torch.device(device)
    generator = torch.Generator(device=device).manual_seed(71819)
    count = 4096
    sample_rate = 28e9
    time = torch.arange(count, device=device) / sample_rate
    records = {}
    for modulation in ("QPSK", "8QAM", "16QAM"):
        points, _, _ = qam_constellation(modulation, device)
        ids = torch.randint(len(points), (2, count), generator=generator, device=device)
        source = points[ids]
        true_cfo = torch.tensor([700e6, -320e6], device=device)
        phase = 0.03 + 2 * math.pi * true_cfo[:, None] * time[None, :]
        amplitude = torch.tensor([[0.002], [0.0005]], device=device)
        noise = torch.complex(
            torch.randn(source.shape, generator=generator, device=device),
            torch.randn(source.shape, generator=generator, device=device),
        ) * (0.005 / math.sqrt(2))
        received = amplitude * (source * torch.exp(1j * phase) + noise)
        recovered, stats = recover_qam_carrier(
            received, modulation, sample_rate, known_block_magnitude=amplitude
        )
        recovered_ids = (recovered[..., None] - points).abs().square().argmin(dim=-1)
        errors = int((recovered_ids != ids).sum())
        estimated_cfo = torch.tensor(stats["estimated_cfo_hz"], device=device)
        assert errors == 0, (modulation, errors, stats)
        # Finite-frame fourth-moment self-noise is nonzero for QAM. Require
        # error below one 4th-harmonic FFT bin; BPS must still give exact data.
        assert (estimated_cfo - true_cfo).abs().max() < sample_rate / (4 * count), stats
        records[modulation] = {"symbol_errors": errors, "estimated_cfo_hz": stats["estimated_cfo_hz"]}

    # Nonzero phase trajectory must be inferred and unwrapped, not divided out
    # with a hidden simulator phase. Deliberately turn CFR off for this check.
    points, _, _ = qam_constellation("QPSK", device)
    ids = torch.randint(len(points), (1, count), generator=generator, device=device)
    source = points[ids]
    phase = torch.linspace(0, 3, count, device=device)[None, :]
    recovered, stats = recover_qam_carrier(
        source * torch.exp(1j * phase), "QPSK", sample_rate,
        SignalDSPConfig(cfo_enabled=False), torch.ones((1, 1), device=device),
    )
    recovered_ids = (recovered[..., None] - points).abs().square().argmin(dim=-1)
    assert torch.equal(ids, recovered_ids), stats
    assert stats["phase_symmetry_unwrap_events"][0] >= 1

    # A global pi/2 ambiguity is intentionally NOT resolved using source data.
    received = source * torch.exp(torch.tensor(1j * math.pi / 2, device=device))
    recovered, _ = recover_qam_carrier(
        received, "QPSK", sample_rate,
        known_block_magnitude=torch.ones((1, 1), device=device),
    )
    recovered_ids = (recovered[..., None] - points).abs().square().argmin(dim=-1)
    assert not torch.equal(ids, recovered_ids), "Undisclosed source-aided phase ambiguity resolution"
    print("Signal-estimated CFR/CPR offline self-test passed:", records)
    return records


In [ ]:
"""Pinned official Bellard libbpg installer for Colab/Linux or local Win64 tests.

No API credentials, import-time work or third-party binary republishing.
Checksums identify the retrieved official distribution, not publisher signatures.
"""
import hashlib
import json
import os
from pathlib import Path
import platform
import re
import shutil
import subprocess
import tarfile
import urllib.request
import zipfile


BPG_RELEASE = "0.9.8"
BPG_SOURCE_URL = "https://bellard.org/bpg/libbpg-0.9.8.tar.gz"
# Independently matched to FreeBSD graphics/libbpg/distinfo (2029021 bytes).
BPG_SOURCE_SHA256 = "c0788e23bdf1a7d36cb4424ccb2fae4c7789ac94949563c4ad0e2569d3bf0095"
BPG_SOURCE_CHECKSUM_REFERENCE = "https://raw.githubusercontent.com/freebsd/freebsd-ports/main/graphics/libbpg/distinfo"
BPG_WIN64_URL = "https://bellard.org/bpg/bpg-0.9.8-win64.zip"
BPG_WIN64_SHA256 = "44034757c7dbf44498bfdb4c28b8ed3fbc498b32e7d18de4b98a6995371d457c"
BPG_CMAKE_VERSION = "3.28.3"
BPG_BUILD_RECIPE = "libbpg098-cmake3283-v2"


def _validated_cmake_version(output):
    found = re.search(r"cmake version (\d+\.\d+\.\d+)",output)
    if not found or found[1] != BPG_CMAKE_VERSION:
        raise RuntimeError("This verified libbpg build requires cmake==3.28.3; "
                           "rerun the notebook installation cell before codec setup. "
                           "CMake3.31+ changes the legacy YASM linker modules.")
    return found[1]


def _get_verified_archive(url,digest,path):
    if re.fullmatch(r"[0-9a-f]{64}",digest) is None:
        raise ValueError("Expected a pinned lowercase SHA256 digest")
    if path.exists() and hashlib.sha256(path.read_bytes()).hexdigest() == digest:
        return
    partial = path.with_suffix(path.suffix+".partial")
    with urllib.request.urlopen(url,timeout=60) as response, partial.open("wb") as handle:
        shutil.copyfileobj(response,handle)
    observed = hashlib.sha256(partial.read_bytes()).hexdigest()
    if observed != digest:
        raise RuntimeError("BPG archive SHA256 mismatch; do not execute it. "
                           f"URL={url}; bytes={partial.stat().st_size}; "
                           f"expected={digest}; observed={observed}")
    partial.replace(path)


def _safe_member(root,name):
    target = (root/name).resolve()
    if not target.is_relative_to(root.resolve()):
        raise ValueError("Unsafe codec archive path")
    return target


def install_official_bpg(cache_dir):
    """Return (encoder,decoder,provenance); Linux build requires apt dependencies.

    On Colab install build-essential/cmake/yasm/libpng-dev/libjpeg-dev/zlib1g-dev
    first. Uses official bundled x265; GUI and optional JCTVC are disabled.
    Never installs system packages or changes PATH on Windows.
    """
    root = Path(cache_dir).resolve()
    root.mkdir(parents=True,exist_ok=True)
    if platform.system() == "Windows":
        url,digest = BPG_WIN64_URL,BPG_WIN64_SHA256
        archive = root/"bpg-0.9.8-win64.zip"
        target = root/"win64"
        _get_verified_archive(url,digest,archive)
        encoder,decoder = target/"bpgenc.exe",target/"bpgdec.exe"
        if not encoder.is_file() or not decoder.is_file():
            with zipfile.ZipFile(archive) as source:
                for item in source.infolist():
                    path = _safe_member(target,item.filename)
                    if item.is_dir():
                        path.mkdir(parents=True,exist_ok=True)
                    else:
                        path.parent.mkdir(parents=True,exist_ok=True)
                        with source.open(item) as src,path.open("wb") as dest:
                            shutil.copyfileobj(src,dest)
        build_command = "official win64 portable distribution; no build"
    elif platform.system() == "Linux":
        url,digest = BPG_SOURCE_URL,BPG_SOURCE_SHA256
        archive = root/"libbpg-0.9.8.tar.gz"
        _get_verified_archive(url,digest,archive)
        # Never reuse the failed 3.31 build tree from earlier notebook versions.
        extract_root = root/BPG_BUILD_RECIPE
        target = extract_root/"libbpg-0.9.8"
        encoder,decoder = target/"bpgenc",target/"bpgdec"
        manifest_path = extract_root/"completed_build.json"
        expected = {"recipe":BPG_BUILD_RECIPE,"archive_sha256":digest,"cmake_version":BPG_CMAKE_VERSION}
        valid_cached_build = False
        if manifest_path.is_file() and encoder.is_file() and decoder.is_file():
            try:
                manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
                valid_cached_build = (all(manifest.get(k)==v for k,v in expected.items())
                    and manifest.get("encoder_sha256")==hashlib.sha256(encoder.read_bytes()).hexdigest()
                    and manifest.get("decoder_sha256")==hashlib.sha256(decoder.read_bytes()).hexdigest())
            except (OSError,ValueError):
                pass
        if not valid_cached_build:
            with tarfile.open(archive,"r:gz") as source:
                for item in source.getmembers():
                    path = _safe_member(extract_root,item.name)
                    if not path.is_relative_to(target):
                        raise ValueError("Unexpected top-level BPG source directory")
                    if item.isdir():
                        path.mkdir(parents=True,exist_ok=True)
                    elif item.isfile():
                        path.parent.mkdir(parents=True,exist_ok=True)
                        with source.extractfile(item) as src,path.open("wb") as dest:
                            shutil.copyfileobj(src,dest)
                        path.chmod(item.mode & 0o777)
                    else:
                        raise ValueError("Unexpected link/device in source archive")
            # Keep upstream source unchanged and use the tested CMake version.
            environment = dict(os.environ)
            try:
                import cmake
                environment["PATH"] = str(cmake.CMAKE_BIN_DIR)+os.pathsep+environment.get("PATH","")
            except ImportError:
                pass
            version = subprocess.run(["cmake","--version"],env=environment,check=True,
                                     capture_output=True,text=True,timeout=15).stdout
            cmake_version = _validated_cmake_version(version)
            command = ["make","-j2","bpgenc","bpgdec","USE_BPGVIEW=","USE_JCTVC=",
                       "CMAKE_OPTS=-DENABLE_LIBNUMA=OFF"]
            log = root/"libbpg_build.log"
            print("Building official BPG0.9.8 once; compiler log:",log,flush=True)
            try:
                with log.open("wb") as handle:
                    # Official Makefile configures only if x265.out does not
                    # exist. Configure explicitly so a failed earlier configure
                    # cannot trap the next attempt in an incomplete directory.
                    common = ["-DENABLE_LIBNUMA=OFF","-DENABLE_SHARED=OFF","-DENABLE_CLI=OFF"]
                    variants = (
                        ("12bit",["-DHIGH_BIT_DEPTH=ON","-DEXPORT_C_API=OFF","-DMAIN12=ON"]),
                        ("10bit",["-DHIGH_BIT_DEPTH=ON","-DEXPORT_C_API=OFF","-DMAIN10=ON"]),
                        ("8bit",["-DLINKED_10BIT=ON","-DLINKED_12BIT=ON"]))
                    for name,flags in variants:
                        build = target/"x265.out"/name
                        build.mkdir(parents=True,exist_ok=True)
                        subprocess.run(["cmake","-S",str(target/"x265/source"),"-B",str(build),*common,*flags],
                                       env=environment,check=True,timeout=120,stdout=handle,stderr=subprocess.STDOUT)
                    subprocess.run(command,cwd=target,env=environment,check=True,timeout=900,
                                   stdout=handle,stderr=subprocess.STDOUT)
            except (subprocess.CalledProcessError,subprocess.TimeoutExpired) as error:
                full_log = log.read_text(encoding="utf-8",errors="replace")
                errors = [line for line in full_log.splitlines() if any(token in line.lower()
                          for token in ("fatal error:"," error:","cmake error","undefined reference","***"))]
                detail = "Error lines:\n"+"\n".join(errors[-20:])+"\nCompiler tail:\n"+full_log[-4000:]
                raise RuntimeError("Official BPG source build failed. "+detail) from error
            if not encoder.is_file() or not decoder.is_file():
                raise RuntimeError("Build exited successfully without both BPG executables")
            manifest = {**expected,"encoder_sha256":hashlib.sha256(encoder.read_bytes()).hexdigest(),
                        "decoder_sha256":hashlib.sha256(decoder.read_bytes()).hexdigest()}
            manifest_path.write_text(json.dumps(manifest,indent=2),encoding="utf-8")
        build_command = "CMake3.28.3 explicit8/10/12bit configure; make -j2 bpgenc bpgdec USE_BPGVIEW= USE_JCTVC= CMAKE_OPTS=-DENABLE_LIBNUMA=OFF"
    else:
        raise RuntimeError("This notebook supports Colab Linux; local smoke testing also supports Win64")
    return str(encoder),str(decoder),{
        "release":BPG_RELEASE,"official_distribution_url":url,"archive_sha256":digest,
        "publisher_signature_verified":False,"platform":platform.platform(),
        "build_command":build_command,
        "build_recipe":BPG_BUILD_RECIPE if platform.system()=="Linux" else "official-win64-098",
        "encoder_binary_sha256":hashlib.sha256(encoder.read_bytes()).hexdigest(),
        "decoder_binary_sha256":hashlib.sha256(decoder.read_bytes()).hexdigest(),
        "redistribution":"Do not commit codecs/DLLs to GitHub. Upstream encoder GPLv2, decoder mixed BSD/LGPL2.1; see bundled licenses."}


In [ ]:
"""Real libbpg CLI transport, with no subprocess/network work at import.

Provenance: https://bellard.org/bpg/ ; official libbpg-0.9.8.tar.gz SHA256
c0788e23bdf1a7d36cb4424ccb2fae4c7789ac94949563c4ad0e2569d3bf0095.
The 256x256 input/output are RGB uint8. Compression explicitly uses x265,
8-bit YCbCr 4:2:0, full range, no alpha/metadata, and lossy q=0..51.

Example (the PNG input must already be RGB8 256x256)::

    bpgenc -e x265 -b 8 -c ycbcr -f 420 -m 4 -q 28 -o image.bpg image.png
    bpgdec -b 8 -o recovered.png image.bpg

Linux source build requires a 64-bit target, make, gcc/g++, cmake, yasm,
libpng-dev, libjpeg-dev and zlib1g-dev. Build only the two utilities with
``make -j2 USE_BPGVIEW= CMAKE_OPTS=-DENABLE_LIBNUMA=OFF bpgenc bpgdec``;
this avoids the optional SDL viewer. The official source bundles its x265
fork: replacing that fork with a current system libx265 is not equivalent.
The verified build pins CMake3.28.3:3.31+ also breaks its old ASM_YASM linker
module, while CMake4 removes its CMP0025/CMP0054 OLD behavior. The installer
configures every variant explicitly so failed configure caches can recover.
Yasm cannot simply be disabled on x86: that x265 source rejects doing so.
The official README supports Windows cross-compilation from Linux, not a
native MSVC build; official portable Windows binaries are also provided.
Building/installing/downloads are intentionally the caller's responsibility.
"""

from dataclasses import dataclass
import hashlib
import math
import operator
import os
from pathlib import Path
import shutil
import subprocess
import tempfile
import warnings

import numpy as np
from PIL import Image, UnidentifiedImageError


BPG_CODEC_VERSION = "libbpg_cli_rgb8_v1"
BPG_MAGIC = b"BPG\xfb"
LIBBPG_SOURCE_URL = "https://bellard.org/bpg/libbpg-0.9.8.tar.gz"
LIBBPG_SOURCE_SHA256 = "c0788e23bdf1a7d36cb4424ccb2fae4c7789ac94949563c4ad0e2569d3bf0095"


class BPGError(RuntimeError):
    """Fatal codec setup/encoding problem, not an ordinary damaged frame."""


class BPGExecutableNotFound(BPGError):
    pass


class BPGEncodeError(BPGError):
    pass


class BPGRateBudgetError(BPGError):
    """No q in the full lossy range fits; never silently increases the budget."""

    def __init__(self, max_bytes, tested_q_sizes):
        self.max_bytes = max_bytes
        self.tested_q_sizes = tuple(sorted(tested_q_sizes))
        self.minimum_size_q, self.minimum_size = min(self.tested_q_sizes, key=lambda item: (item[1], item[0]))
        super().__init__(f"No lossy BPG q=0..51 fits {max_bytes} bytes; minimum measured "
                         f"size is {self.minimum_size} bytes at q={self.minimum_size_q}")


@dataclass(frozen=True)
class EncodedBPG:
    data: bytes
    q: int
    max_bytes: int
    image_hw: tuple
    tested_q_sizes: tuple
    search_mode: str
    optimality: str
    encoder_settings: dict
    cache_hits: int = 0

    @property
    def num_bytes(self):
        return len(self.data)

    @property
    def bpp(self):
        return 8 * len(self.data) / (self.image_hw[0] * self.image_hw[1])

    def metadata(self):
        return {"codec": BPG_CODEC_VERSION, "q": self.q, "actual_bytes": self.num_bytes,
                "actual_bpp": self.bpp, "budget_bytes": self.max_bytes,
                "bpg_header_included": True, "image_hw": list(self.image_hw),
                "search_mode": self.search_mode, "optimality": self.optimality,
                "tested_q_sizes": {str(q): size for q, size in self.tested_q_sizes},
                "tested_q_count": len(self.tested_q_sizes), "cache_hits": self.cache_hits,
                "encoder_settings": dict(self.encoder_settings),
                "source_url": LIBBPG_SOURCE_URL, "source_sha256": LIBBPG_SOURCE_SHA256}


@dataclass(frozen=True)
class DecodeResult:
    status: str
    image: object = None
    error: str = ""
    returncode: object = None

    @property
    def ok(self):
        return self.status == "ok"


def _integer(value, name, minimum, maximum=None):
    if isinstance(value, (bool, np.bool_)):
        raise ValueError(f"{name} must be an integer")
    try:
        value = operator.index(value)
    except TypeError as exc:
        raise ValueError(f"{name} must be an integer") from exc
    if value < minimum or (maximum is not None and value > maximum):
        raise ValueError(f"{name} is outside its permitted range")
    return value


def bytes_to_bits(data):
    """Every byte, including the BPG header, in MSB-first transmission order."""
    if not isinstance(data, (bytes, bytearray, memoryview)):
        raise TypeError("data must be bytes-like")
    return np.unpackbits(np.frombuffer(bytes(data), dtype=np.uint8), bitorder="big")


def bits_to_bytes(bits):
    """Reverse bytes_to_bits; rejects truncation, nonbinary values and padding."""
    bits = np.asarray(bits)
    if bits.ndim != 1 or bits.size % 8:
        raise ValueError("bits must be one-dimensional and byte-aligned")
    if not np.isin(bits, [0, 1]).all():
        raise ValueError("bits must contain only 0 or 1")
    return np.packbits(bits.astype(np.uint8), bitorder="big").tobytes()


def _read_ue7(data, offset):
    value = 0
    for i in range(5):
        if offset >= len(data):
            raise ValueError("Truncated BPG header")
        byte = data[offset]
        offset += 1
        if i == 0 and byte == 0x80:
            raise ValueError("Noncanonical BPG header integer")
        value = (value << 7) | (byte & 127)
        if value > 0xffffffff:
            raise ValueError("Oversized BPG header integer")
        if not byte & 128:
            return value, offset
    raise ValueError("Oversized BPG header integer")


def _validate_header(data, expected_hw):
    """Bound dimensions before handing an untrusted bitstream to native code."""
    if len(data) < 9 or not data.startswith(BPG_MAGIC):
        raise ValueError("Missing or truncated BPG header")
    if (data[4] & 15) != 0:
        raise ValueError("Expected an 8-bit BPG stream")
    if data[4] >> 5 > 5 or data[5] >> 4 > 4:
        raise ValueError("Invalid BPG pixel format/color space")
    if data[5] & 1:
        raise ValueError("Animated BPG is not supported by the still-image baseline")
    width, offset = _read_ue7(data, 6)
    height, offset = _read_ue7(data, offset)
    if (height, width) != expected_hw:
        raise ValueError(f"BPG dimensions {(height, width)} do not match {expected_hw}")
    payload_size, offset = _read_ue7(data, offset)
    if data[5] & 8:
        extension_size, offset = _read_ue7(data, offset)
        offset += extension_size
    if offset >= len(data) or (payload_size and payload_size > len(data) - offset):
        raise ValueError("Truncated BPG payload/extension")


def _command_prefix(command):
    if isinstance(command, (str, os.PathLike)):
        result = [os.fspath(command)]
    else:
        result = [os.fspath(item) for item in command]
    if not result or not result[0]:
        raise ValueError("Executable command cannot be empty")
    executable = shutil.which(result[0])
    if executable is None:
        raise BPGExecutableNotFound(f"BPG executable not found: {result[0]}. "
                                    f"Install official libbpg: {LIBBPG_SOURCE_URL}")
    result[0] = str(Path(executable).resolve())
    return result


def _run(command, directory, timeout):
    # Files prevent an invalid/native decoder's diagnostics from filling RAM.
    with tempfile.TemporaryFile() as stdout, tempfile.TemporaryFile() as stderr:
        try:
            process = subprocess.run(command, cwd=directory, stdin=subprocess.DEVNULL,
                                     stdout=stdout, stderr=stderr, timeout=timeout, check=False)
        except FileNotFoundError as exc:
            raise BPGExecutableNotFound(f"BPG executable disappeared: {command[0]}") from exc
        except OSError as exc:
            raise BPGError(f"Cannot launch BPG executable {command[0]}: {exc}") from exc
        stderr.seek(0, os.SEEK_END)
        length = stderr.tell()
        stderr.seek(max(0, length - 4096))
        detail = stderr.read(4096).decode("utf-8", errors="replace").strip()
        return process.returncode, detail


class BPGCodec:
    """Bounded native codec wrapper; a one-image cache avoids duplicate q work.

    Missing/unlaunchable executables are fatal. Damaged received streams return
    DecodeResult failures; no image or score is synthesized by the codec. The
    evaluator must declare its failure convention and retain all frames. The
    paper's explicit LPIPS=1 failure marker and a separately measured gray
    reconstruction metric are permitted, provided they are clearly separated.
    Success requires exit status zero AND a valid expected-size RGB8 image;
    a nonzero exit is failure even if the process left a partial PNG behind.

    binary_checked picks the best *tested* feasible q without claiming global
    optimality for nonmonotonic sizes. exhaustive proves the smallest feasible
    q. If q51 does not fit, binary_checked falls back to exhaustive rather than
    falsely reporting that no q can fit.
    """

    def __init__(self, encoder="bpgenc", decoder="bpgdec", timeout=30.0,
                 expected_hw=(256, 256), speed=4):
        self.encoder = _command_prefix(encoder)
        self.decoder = _command_prefix(decoder)
        self.timeout = float(timeout)
        if not math.isfinite(self.timeout) or self.timeout <= 0:
            raise ValueError("timeout must be finite and positive")
        if len(expected_hw) != 2:
            raise ValueError("expected_hw must be (height, width)")
        self.expected_hw = tuple(_integer(v, "image dimension", 1, 16384) for v in expected_hw)
        self.speed = _integer(speed, "speed", 1, 9)
        self._cached_image = None
        self._encoded = {}
        self._cache_hits = 0

    @property
    def encoder_settings(self):
        return {"encoder": "x265", "bit_depth": 8, "color_space": "ycbcr",
                "chroma_format": "420", "speed": self.speed, "lossless": False,
                "input_mode": "RGB", "output_dtype": "uint8", "metadata": False}

    def _prepare_image(self, image):
        if not isinstance(image, Image.Image) or image.mode != "RGB":
            raise ValueError("image must be a PIL RGB image")
        if image.size != self.expected_hw[::-1]:
            raise ValueError(f"Input dimensions must be {self.expected_hw[::-1]}")
        key = hashlib.sha256(image.tobytes()).digest(), self.speed
        if key != self._cached_image:
            self._cached_image, self._encoded = key, {}
        return image

    def encode(self, image, q):
        """Return genuine complete .bpg bytes at one lossy quantizer value."""
        image = self._prepare_image(image)
        q = _integer(q, "q", 0, 51)
        if q in self._encoded:
            self._cache_hits += 1
            return self._encoded[q]
        with tempfile.TemporaryDirectory(prefix="fso_bpg_encode_") as directory:
            source, output = Path(directory) / "source.png", Path(directory) / "encoded.bpg"
            # A fresh RGB image deliberately strips ICC/EXIF and alpha metadata.
            Image.frombytes("RGB", image.size, image.tobytes()).save(source, format="PNG")
            command = self.encoder + ["-e", "x265", "-b", "8", "-c", "ycbcr", "-f", "420",
                                      "-m", str(self.speed), "-q", str(q), "-o", str(output), str(source)]
            try:
                returncode, detail = _run(command, directory, self.timeout)
            except subprocess.TimeoutExpired as exc:
                raise BPGEncodeError(f"bpgenc timed out at q={q}") from exc
            if returncode != 0:
                raise BPGEncodeError(f"bpgenc exited {returncode} at q={q}: {detail}")
            if not output.is_file():
                raise BPGEncodeError(f"bpgenc produced no output at q={q}")
            data = output.read_bytes()
            try:
                _validate_header(data, self.expected_hw)
            except ValueError as exc:
                raise BPGEncodeError(f"Invalid bpgenc output at q={q}: {exc}") from exc
        self._encoded[q] = data
        return data

    def encode_to_budget(self, image, max_bytes, search_mode="binary_checked"):
        """Budget includes every BPG byte, but not caller-owned transport framing."""
        self._prepare_image(image)
        max_bytes = _integer(max_bytes, "max_bytes", 1)
        if search_mode not in ("binary_checked", "exhaustive"):
            raise ValueError("search_mode must be binary_checked or exhaustive")
        tested, cache_hits_before = {}, self._cache_hits

        def measure(q):
            if q not in tested:
                tested[q] = self.encode(image, q)
            return len(tested[q]) <= max_bytes

        mode = search_mode
        if search_mode == "exhaustive":
            for q in range(52):
                if measure(q):
                    break
        elif not measure(0):
            if not measure(51):
                mode = "exhaustive_fallback"
                for q in range(1, 51):
                    if measure(q):
                        break
            else:
                lo, hi = 0, 51
                while hi - lo > 1:
                    mid = (lo + hi) // 2
                    if measure(mid):
                        hi = mid
                    else:
                        lo = mid
                best = min(q for q, data in tested.items() if len(data) <= max_bytes)
                for q in range(max(0, best - 2), min(51, best + 2) + 1):
                    measure(q)
        # Previously measured q values for this exact image are evidence too:
        # never ignore a better known feasible stream when reusing the cache.
        tested.update(self._encoded)
        feasible = [q for q, data in tested.items() if len(data) <= max_bytes]
        sizes = tuple(sorted((q, len(data)) for q, data in tested.items()))
        if not feasible:
            # The q51 failure route has measured every q before reaching here.
            raise BPGRateBudgetError(max_bytes, sizes)
        q = min(feasible)
        proven = mode != "binary_checked" or q == 0
        optimality = ("lowest feasible q over 0..51 (all lower q measured)" if proven else
                      "lowest tested feasible q; not exhaustive/global under nonmonotonic size")
        return EncodedBPG(tested[q], q, max_bytes, self.expected_hw, sizes, mode, optimality,
                          self.encoder_settings, self._cache_hits - cache_hits_before)

    def decode(self, data):
        """Decode one hard received stream; no fallback is synthesized here."""
        if not isinstance(data, (bytes, bytearray, memoryview)):
            raise TypeError("data must be bytes-like")
        data = bytes(data)
        try:
            _validate_header(data, self.expected_hw)
        except ValueError as exc:
            return DecodeResult("invalid_stream", error=str(exc))
        with tempfile.TemporaryDirectory(prefix="fso_bpg_decode_") as directory:
            source, output = Path(directory) / "received.bpg", Path(directory) / "decoded.png"
            source.write_bytes(data)
            try:
                returncode, detail = _run(self.decoder + ["-b", "8", "-o", str(output), str(source)],
                                          directory, self.timeout)
            except subprocess.TimeoutExpired:
                return DecodeResult("timeout", error=f"bpgdec exceeded {self.timeout:g}s")
            if returncode != 0:
                return DecodeResult("decoder_exit", error=detail or "bpgdec failed", returncode=returncode)
            if not output.is_file():
                return DecodeResult("invalid_output", error="bpgdec produced no output")
            # A valid RGB8 PNG is comfortably smaller; reject runaway output.
            maximum_png_bytes = self.expected_hw[0] * self.expected_hw[1] * 8 + 65536
            if output.stat().st_size > maximum_png_bytes:
                return DecodeResult("invalid_output", error="bpgdec output exceeds sanity bound")
            try:
                with warnings.catch_warnings():
                    warnings.simplefilter("error", Image.DecompressionBombWarning)
                    with Image.open(output) as decoded:
                        if decoded.format != "PNG" or decoded.size != self.expected_hw[::-1]:
                            return DecodeResult("invalid_output", error="Wrong output format/dimensions")
                        if decoded.mode not in ("RGB", "RGBA", "L", "LA", "P"):
                            return DecodeResult("invalid_output", error="Expected an 8-bit PNG output")
                        decoded.load()
                        pixels = np.asarray(decoded.convert("RGB"), dtype=np.uint8).copy()
            except (OSError, ValueError, UnidentifiedImageError, Image.DecompressionBombError,
                    Image.DecompressionBombWarning) as exc:
                return DecodeResult("invalid_output", error=f"Unreadable decoder image: {exc}")
            if pixels.shape != self.expected_hw + (3,) or pixels.dtype != np.uint8:
                return DecodeResult("invalid_output", error="Invalid RGB uint8 reconstruction")
            return DecodeResult("ok", pixels, returncode=0)


In [ ]:
"""Actual rate-3/4 (2048,1536) systematic repeat-accumulate LDPC.

The FSO-SC paper specifies rate 3/4 and block length 2048 (Sec. IV,
printed p.9173), but does not publish its parity-check matrix or decoder.
This module therefore supplies a reproducible *replacement matrix*, not
the undisclosed paper code.  Its systematic repeat-accumulate construction
uses H=[A|D], where A has column weight three and D is lower bidiagonal.
This is the sparse systematic RA/IRA family described in H. Jin's thesis,
Chapter 7: https://thesis.caltech.edu/3191/1/HJinthesis.pdf .  The particular
balanced, seeded, four-cycle-free A below is an implementation assumption;
no published performance claim or optimized degree distribution is implied.

Decoder inputs are genuine soft log(P(bit=0)/P(bit=1)) likelihood ratios.
Parity convergence is not a CRC and does not establish correct recovery.
Only PyTorch plus the Python standard library are required; all iterative
encoding/decoding operations stay on the module's CPU or CUDA device.
"""


import hashlib
import math
import random
import struct

import torch
from torch import nn


def _construct_rows(seed: int) -> list[list[int]]:
    """Build A with balanced check degrees, excluding all length-4 cycles.

An information column cannot reuse a pair of checks.  Adjacent check
pairs are already occupied by a column of the bidiagonal D, so they are
excluded too.  Local Python RNG keeps global training RNGs untouched.
"""
    m, k = 512, 1536
    rng = random.Random(seed)
    rows: list[list[int]] = [[] for _ in range(m)]
    degrees = [0] * m
    used_pairs = {(i, i + 1) for i in range(m - 1)}
    for variable in range(k):
        selected: list[int] = []
        for _ in range(3):
            candidates = [
                check for check in range(m)
                if check not in selected
                and all(tuple(sorted((check, old))) not in used_pairs for old in selected)
            ]
            if not candidates:
                raise RuntimeError("LDPC construction exhausted four-cycle-free candidates")
            minimum = min(degrees[check] for check in candidates)
            chosen = rng.choice([check for check in candidates if degrees[check] == minimum])
            selected.append(chosen)
            degrees[chosen] += 1
            rows[chosen].append(variable)
        for a in range(3):
            for b in range(a + 1, 3):
                used_pairs.add(tuple(sorted((selected[a], selected[b]))))
    for check in range(m):
        rows[check].append(k + check)
        if check:
            rows[check].append(k + check - 1)
        rows[check].sort()
    return rows


class SystematicRALDPC(nn.Module):
    """Fixed dimensions; batched systematic encoder and soft min-sum decoder.

    ``decode_llr`` returns ``(information_bits, stats)``. All statistics are
    tensors on the input device; iterations are zero for initially valid
    codewords. A successful syndrome does not rule out undetected errors.
    ``seed``, normalization, iteration limit and LLR clipping must be fixed
    before any image-quality sweep, not selected to fit reported curves.
    """

    n = 2048
    k = 1536
    m = 512
    rate = 0.75

    def __init__(
        self,
        *,
        max_iterations: int = 50,
        normalization: float = 0.8,
        seed: int = 20261028,
        llr_clip: float = 30.0,
    ) -> None:
        super().__init__()
        if isinstance(max_iterations, bool) or not isinstance(max_iterations, int) or max_iterations < 1:
            raise ValueError("max_iterations must be a positive integer")
        if not math.isfinite(float(normalization)) or not 0.0 < normalization <= 1.0:
            raise ValueError("normalization must be finite and in (0,1]")
        if not math.isfinite(float(llr_clip)) or llr_clip <= 0:
            raise ValueError("llr_clip must be finite and positive")
        if isinstance(seed, bool) or not isinstance(seed, int):
            raise ValueError("seed must be an integer")
        self.max_iterations = max_iterations
        self.normalization = float(normalization)
        self.seed = seed
        self.llr_clip = float(llr_clip)
        rows = _construct_rows(seed)
        self.edge_count = sum(map(len, rows))
        # Canonical byte layout: row-major (check, variable) uint16 little endian.
        canonical = b"".join(struct.pack("<HH", i, j) for i, row in enumerate(rows) for j in row)
        self.matrix_sha256 = hashlib.sha256(canonical).hexdigest()
        self._row_degree_range = (min(map(len, rows)), max(map(len, rows)))
        width = max(map(len, rows))
        neighbors = torch.zeros((self.m, width), dtype=torch.long)
        valid = torch.zeros((self.m, width), dtype=torch.bool)
        information_mask = torch.zeros((self.m, width), dtype=torch.bool)
        for check, row in enumerate(rows):
            neighbors[check, :len(row)] = torch.tensor(row, dtype=torch.long)
            valid[check, :len(row)] = True
            information_mask[check, :len(row)] = torch.tensor([v < self.k for v in row])
        self.register_buffer("check_variables", neighbors)
        self.register_buffer("check_mask", valid)
        self.register_buffer("information_mask", information_mask)

    def metadata(self) -> dict:
        return {
            "name": "systematic_ra_2048_1536_seeded_v1",
            "n": self.n, "k": self.k, "m": self.m, "rate": self.rate,
            "seed": self.seed, "matrix_sha256": self.matrix_sha256,
            "matrix_hash_encoding": "row-major (check,variable), uint16 little endian pairs",
            "edge_count": self.edge_count,
            "information_column_weight": 3,
            "check_degree_min": self._row_degree_range[0],
            "check_degree_max": self._row_degree_range[1],
            "four_cycles": 0,
            "decoder": "flooding normalized min-sum with syndrome early stopping",
            "normalization": self.normalization,
            "max_iterations": self.max_iterations,
            "llr_clip": self.llr_clip,
            "llr_convention": "log(P0/P1); positive means bit zero",
            "paper_dimensions_matched": True,
            "paper_matrix_reproduced": False,
            "matrix_assumption": "H=[seeded balanced column-weight-3 A | lower-bidiagonal D]",
            "construction_reference": "H. Jin thesis, Chapter 7, https://thesis.caltech.edu/3191/1/HJinthesis.pdf",
            "convergence_is_crc": False,
        }

    def _check_input(self, value: torch.Tensor, width: int, *, bits: bool) -> None:
        if not isinstance(value, torch.Tensor) or value.ndim != 2 or value.shape[1] != width:
            raise ValueError(f"expected a tensor with shape [batch,{width}]")
        if value.device != self.check_variables.device:
            raise ValueError("input and LDPC module must share a device; use ldpc.to(device)")
        if value.is_complex():
            raise ValueError("expected real bits or real-valued LLRs")
        if bits:
            if not bool(((value == 0) | (value == 1)).all()):
                raise ValueError("bits must contain only zeros and ones")
        elif not value.is_floating_point() or not bool(torch.isfinite(value).all()):
            raise ValueError("LLRs must be finite floating-point values")

    @torch.no_grad()
    def encode_bits(self, bits: torch.Tensor) -> torch.Tensor:
        self._check_input(bits, self.k, bits=True)
        # Padding and parity neighbor slots are masked, not included in Au.
        information_indices = self.check_variables.clamp_max(self.k - 1)
        local = bits[:, information_indices].to(torch.int64)
        au = (local * self.information_mask).sum(dim=-1).remainder(2)
        parity = au.cumsum(dim=1).remainder(2).to(torch.uint8)
        return torch.cat((bits.to(torch.uint8), parity), dim=1)

    @torch.no_grad()
    def syndrome(self, codeword: torch.Tensor) -> torch.Tensor:
        self._check_input(codeword, self.n, bits=True)
        return self._syndrome_unchecked(codeword)

    def _syndrome_unchecked(self, codeword: torch.Tensor) -> torch.Tensor:
        return (codeword[:, self.check_variables].to(torch.int64) * self.check_mask).sum(-1).remainder(2).to(torch.uint8)

    @torch.no_grad()
    def decode_llr(self, llr: torch.Tensor) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:
        self._check_input(llr, self.n, bits=False)
        channel = llr.to(torch.float32).clamp(-self.llr_clip, self.llr_clip)
        batch = channel.shape[0]
        posterior = channel.clone()
        hard = (posterior < 0).to(torch.uint8)
        initial_weight = self._syndrome_unchecked(hard).sum(dim=1)
        syndrome_weight = initial_weight.clone()
        converged = initial_weight == 0
        iterations = torch.zeros(batch, dtype=torch.int64, device=channel.device)
        r = torch.zeros((batch,) + tuple(self.check_variables.shape), device=channel.device)
        q = channel[:, self.check_variables]
        flat_indices = self.check_variables.reshape(1, -1).expand(batch, -1)
        slots = torch.arange(self.check_variables.shape[1], device=channel.device)
        for iteration in range(1, self.max_iterations + 1):
            if bool(converged.all()):
                break
            active = ~converged
            signs = torch.where(q < 0, -1.0, 1.0)
            signs = torch.where(self.check_mask, signs, 1.0)
            abs_q = torch.where(self.check_mask, q.abs(), float("inf"))
            minima, locations = abs_q.topk(2, dim=-1, largest=False, sorted=True)
            excluded_minimum = torch.where(slots == locations[..., :1], minima[..., 1:2], minima[..., :1])
            new_r = self.normalization * signs.prod(-1, keepdim=True) * signs * excluded_minimum
            new_r = torch.where(self.check_mask, new_r, 0.0)
            # Freeze converged blocks. They must not drift while others iterate.
            r = torch.where(active[:, None, None], new_r, r)
            candidate = channel.clone().scatter_add_(1, flat_indices, r.reshape(batch, -1))
            posterior = torch.where(active[:, None], candidate, posterior)
            hard = (posterior < 0).to(torch.uint8)
            syndrome_weight = self._syndrome_unchecked(hard).sum(dim=1)
            iterations[active] = iteration
            converged = converged | (syndrome_weight == 0)
            q = (posterior[:, self.check_variables] - r).clamp(-self.llr_clip, self.llr_clip)
        return hard[:, :self.k], {
            "converged": converged,
            "iterations": iterations,
            "syndrome_weight": syndrome_weight,
            "initial_syndrome_weight": initial_weight,
        }


In [ ]:
"""Explicit noise interpretations; embeddable after the legacy channel namespace.

No training/model changes and no default profile. These are sensitivity profiles,
not settings selected to match the published LPIPS curve or a full VPI replica.
"""

import math


_PROFILE_FLAGS = {
    "paper_eq3_4_background_only": (True, False, False),
    "coherent_receiver_extension_v6": (True, True, True),
    "background_plus_ase_ablation": (True, True, False),
}
PROFILE_NAMES = tuple(_PROFILE_FLAGS)
_NOISE_FLAGS = (
    "include_background_noise", "include_ase_noise", "include_shot_noise"
)


def _profile_flags(name):
    if name not in _PROFILE_FLAGS:
        raise ValueError(f"Unknown explicit channel profile {name!r}; choose {PROFILE_NAMES}")
    return dict(zip(_NOISE_FLAGS, _PROFILE_FLAGS[name]))


def make_channel_profile(name, config_class, **overrides):
    """Construct the caller's config class; noise flags cannot contradict its label."""
    flags = _profile_flags(name)
    protected = set(overrides).intersection(_NOISE_FLAGS)
    if protected:
        raise ValueError(f"Profile noise flags cannot be overridden: {sorted(protected)}")
    return config_class(**flags, **overrides)


def profile_metadata(name):
    """Return fresh JSON-safe provenance, including what this profile cannot claim."""
    flags = _profile_flags(name)
    basis = {
        "paper_eq3_4_background_only": (
            "Chen et al. Eq. (3) defines n(t) as background-light noise; Eq. (4) "
            "gives its collected optical power. No receiver shot or ASE term is "
            "printed in those equations."
        ),
        "coherent_receiver_extension_v6": (
            "Unchanged v6 interpretation: background plus propagated transmitter "
            "EDFA ASE and input-referred single-polarization heterodyne shot noise."
        ),
        "background_plus_ase_ablation": (
            "Diagnostic ablation only: retain v6 background and transmitter ASE "
            "but omit its receiver shot term. Not an independently published model."
        ),
    }
    return {
        "profile": name,
        "noise_flags": flags,
        "basis": basis[name],
        "diagnostic_only": name == "background_plus_ase_ablation",
        "full_vpi_reproduction": False,
        "physical_superiority_claim": False,
        "curve_fitted": False,
        "limitations": [
            "Eq. (4) optical background power to complex matched-filter variance "
            "is an implementation approximation, not a disclosed receiver transfer.",
            "FOV unit, optical filter width, Malaga phase/beta, fading coherence "
            "and proprietary coherent receiver/DSP details remain assumptions.",
            "Background-only omission of quantum shot noise is an equation-level "
            "idealization, not a physically complete coherent receiver.",
            "ASE bandwidth/post-EDFA launch reference and shot bandwidth/polarization "
            "are inferred; neither establishes the undisclosed VPI configuration.",
        ],
        "provenance": {
            "paper": "https://ieeexplore.ieee.org/document/10969992/",
            "published_channel": "Chen et al., Sec. II-B, Eqs. (3)-(4); Table I",
            "v6_shot_reference": "https://doi.org/10.1103/PhysRevA.77.032116",
            "selection_rule": "Choose explicitly before examining evaluation outcomes.",
        },
    }


def _positive_power_dbm(power_w):
    return 10.0 * math.log10(power_w) + 30.0 if 0.0 < power_w < math.inf else None


def summarize_channel_budget(config, condition, budget_function):
    """Scalar no-fade, single-branch Es/N0; excludes fading and DSP errors.

    Powers are in W; the active noise terms denote E|n|^2 at the receiver.
    ASE is propagated through the same power attenuation as the signal. Raw
    formula outputs remain visible even for disabled noise terms. No Torch is
    imported; pass the existing compute_paper_link_budget function explicitly.
    """
    flags = {key: bool(getattr(config, key)) for key in _NOISE_FLAGS}
    matches = [name for name in PROFILE_NAMES if _profile_flags(name) == flags]
    if len(matches) != 1:
        raise ValueError("Config noise flags do not match an explicit channel profile")
    budget = budget_function(config, **dict(condition))
    attenuation = float(budget["total_attenuation_db"])
    launch = float(budget["launch_power_w"])
    if not math.isfinite(attenuation) or not math.isfinite(launch) or launch < 0.0:
        raise ValueError("Attenuation and nonnegative launch power must be finite")
    transmission = 10.0 ** (-attenuation / 10.0)
    signal = launch * transmission
    computed = {
        "background": float(budget["background_noise_w"]),
        "transmitter_ase": float(budget["ase_noise_w"]),
        "receiver_shot": float(budget["receiver_shot_noise_w"]),
    }
    if any(not math.isfinite(value) or value < 0.0 for value in computed.values()):
        raise ValueError("Computed noise powers must be finite and nonnegative")
    active = {
        "background": computed["background"] if flags[_NOISE_FLAGS[0]] else 0.0,
        "transmitter_ase": computed["transmitter_ase"] * transmission
        if flags[_NOISE_FLAGS[1]] else 0.0,
        "receiver_shot": computed["receiver_shot"] if flags[_NOISE_FLAGS[2]] else 0.0,
    }
    active["total"] = sum(active.values())
    snr = signal / active["total"] if active["total"] > 0.0 else None
    snr = snr if snr is None or math.isfinite(snr) else None
    snr_db = 10.0 * math.log10(snr) if snr is not None and snr > 0.0 else None
    return {
        "profile": matches[0],
        "condition": dict(condition),
        "noise_flags": flags,
        "total_attenuation_db": attenuation,
        "weather_attenuation_db": float(budget["atmospheric_db"]),
        "signal_power_no_fading_w": signal,
        "signal_power_no_fading_dbm": _positive_power_dbm(signal),
        "computed_noise_variance_w": computed,
        "computed_noise_power_dbm": {
            key: _positive_power_dbm(value) for key, value in computed.items()
        },
        "active_received_noise_variance_w": active,
        "active_received_noise_power_dbm": {
            key: _positive_power_dbm(value) for key, value in active.items()
        },
        "no_fading_esn0_linear": snr,
        "no_fading_esn0_db": snr_db,
        "snr_status": "finite" if snr_db is not None else "zero_signal_or_noise",
        "snr_scope": "single branch; unit-energy symbols; no fading or DSP errors",
    }


def qpsk_awgn_ber_from_esn0_db(esn0_db):
    """Gray QPSK coherent AWGN BER: Q(sqrt(Es/N0)); not a fading/DSP prediction."""
    esn0_db = float(esn0_db)
    if not math.isfinite(esn0_db):
        raise ValueError("Es/N0 must be finite dB")
    if esn0_db > 300.0:
        return 0.0
    return 0.5 * math.erfc(math.sqrt(10.0 ** (esn0_db / 10.0) / 2.0))


In [ ]:
"""BPG/real-soft-LDPC comparison over the unchanged symbol-rate FSO/DSP.

No codec substitute, BER threshold erasure, truth-aided decoder or training.
Known file length, real block-amplitude CSI and noise PSD are disclosed receiver
assumptions. The simulator's phase and transmitted bits are not passed to DSP
or LDPC. Transmitted bits are used only for AFTER-decoding error statistics.
"""
import hashlib
import json
import math

import numpy as np
import torch


BPG_EVALUATION_VERSION = "fso_sc_bpg_comparison_v1"


def bpg_comparison_conditions():
    """14 predeclared development conditions, not the entire paper sweep."""
    conditions = []
    def add(group,power,modulation,zenith,pointing=20.,weather="clear"):
        kwargs = {"launch_power_dbm":float(power),"modulation":modulation,
                  "zenith_angle_deg":float(zenith),"pointing_error_urad":float(pointing),
                  "background_light_radiance_w_m2_sr_um":1e-4,
                  "weather":weather,"scattered_power_ratio":.95}
        key = json.dumps(kwargs,sort_keys=True,separators=(",",":"))
        conditions.append({"id":hashlib.sha256(key.encode()).hexdigest()[:16],
                           "group":group,"channel_kwargs":kwargs})
    for power in (30,33,36):
        for modulation in ("QPSK","8QAM","16QAM"):
            add("power_at_zenith30",power,modulation,30)
    for modulation in ("QPSK","8QAM","16QAM"):
        add("coverage_at_36dBm",36,modulation,60)
    for weather in ("snow","fog"):
        add("adverse_weather",36,"QPSK",0,pointing=15,weather=weather)
    return conditions


def soft_qam_llr(symbols,modulation,noise_variance,namespace,clip=30.):
    """Exact log-MAP bit LLR, log(P0/P1), E|complex noise|²=N0.

    The equalized noise variance uses disclosed real-amplitude/noise-PSD CSI,
    not the simulator's actual noise sample or exact phase trajectory. Residual
    phase-estimation errors are deliberately NOT oracle-corrected here.
    """
    if not torch.is_complex(symbols) or symbols.ndim != 2:
        raise ValueError("Expected complex [batch,symbols]")
    points,labels,_ = namespace["qam_constellation"](modulation,symbols.device)
    n0 = torch.as_tensor(noise_variance,device=symbols.device,dtype=torch.float64)
    if not bool(torch.isfinite(n0).all()) or bool((n0 < 0).any()):
        raise ValueError("Noise variance must be finite and nonnegative")
    if n0.ndim == 0:
        n0 = n0.reshape(1,1)
    n0 = torch.broadcast_to(n0,symbols.shape).clamp_min(1e-15)
    # Float64 avoids overflow/cancellation in the severe-fog diagnostic. No
    # precision change is made to the semantic model or the carrier recovery.
    distances = (symbols.to(torch.complex128)[...,None]-points.to(torch.complex128)).abs().square()
    scores = -distances/n0[...,None]
    llr = torch.stack([torch.logsumexp(scores[...,labels[:,bit]==0],dim=-1)
                       -torch.logsumexp(scores[...,labels[:,bit]==1],dim=-1)
                       for bit in range(labels.shape[1])],dim=-1)
    if not bool(torch.isfinite(llr).all()):
        raise FloatingPointError("Nonfinite soft QAM LLR; do not drop the frame")
    return llr.clamp(-clip,clip).to(torch.float32)


@torch.inference_mode()
def transmit_bpg_bytes(payload,physical,condition,namespace,seed,device="cpu",ldpc=None,dsp_config=None):
    """Actually transmit every BPG byte; optional true LDPC, then estimated DSP.

    File length is known framing side information (not a protected header or
    CRC). QAM tail padding and complete LDPC block padding are counted.
    """
    if not isinstance(payload,bytes) or not payload:
        raise ValueError("A nonempty actual BPG byte stream is required")
    device = torch.device(device)
    source = torch.from_numpy(np.unpackbits(np.frombuffer(payload,dtype=np.uint8)).copy()).to(device)
    source_count = source.numel()
    if ldpc is None:
        transmitted_bits = source.reshape(1,-1)
        blocks,information_padding = 0,0
    else:
        blocks = math.ceil(source_count/ldpc.k)
        information_padding = blocks*ldpc.k-source_count
        padded = torch.cat((source,torch.zeros(information_padding,dtype=torch.uint8,device=device)))
        transmitted_bits = ldpc.encode_bits(padded.reshape(blocks,ldpc.k)).reshape(1,-1)
    modulation = condition["channel_kwargs"]["modulation"]
    _,_,bits_per_symbol = namespace["qam_constellation"](modulation,device)
    coded_count = transmitted_bits.numel()
    qam_padding = (-coded_count)%bits_per_symbol
    bits_for_qam = torch.cat((transmitted_bits,torch.zeros((1,qam_padding),device=device,dtype=torch.uint8)),dim=-1)
    symbols = namespace["qam_modulate"](bits_for_qam.reshape(1,-1,bits_per_symbol),modulation)
    kwargs = {key:value for key,value in condition["channel_kwargs"].items() if key != "modulation"}
    observation = physical(symbols,generator=torch.Generator(device=device).manual_seed(int(seed)),**kwargs)
    if observation.is_equalized or observation.channel_gain is None:
        raise ValueError("Need an unequalized single-Rx physical observation")
    config = namespace["SignalDSPConfig"]() if dsp_config is None else dsp_config
    if config.amplitude_mode != "known_block_magnitude":
        raise ValueError("This baseline declares real block-amplitude CSI; no hidden alternate estimator")
    magnitude = observation.channel_gain.abs().mean(dim=-1,keepdim=True)
    recovered,dsp_diagnostics = namespace["recover_qam_carrier"](
        observation.symbols,modulation,physical.config.symbol_rate_baud,config,
        known_block_magnitude=magnitude)
    hard = namespace["qam_demodulate"](recovered,modulation).reshape(1,-1)[:,:coded_count]
    raw_errors = int((hard != transmitted_bits).sum())
    budget = observation.diagnostics["budget"]
    receiver_noise = (budget["background_noise_w"] if physical.config.include_background_noise else 0.)
    receiver_noise += budget["receiver_shot_noise_w"] if physical.config.include_shot_noise else 0.
    equivalent_n0 = receiver_noise/magnitude.to(torch.float64).square().clamp_min(1e-300)
    if physical.config.include_ase_noise:
        equivalent_n0 += budget["ase_noise_w"]/budget["launch_power_w"]
    ldpc_statistics = None
    if ldpc is None:
        decoded_bits = hard.reshape(-1)[:source_count]
    else:
        soft = soft_qam_llr(recovered,modulation,equivalent_n0,namespace,clip=ldpc.llr_clip)
        llr = soft.reshape(-1)[:coded_count].reshape(blocks,ldpc.n)
        information,ldpc_statistics = ldpc.decode_llr(llr)
        decoded_bits = information.reshape(-1)[:source_count]
    received_bytes = np.packbits(decoded_bits.cpu().numpy()).tobytes()
    result = {"received_bytes":received_bytes,"seed":int(seed),
              "bit_errors":int((decoded_bits != source).sum()),"valid_bits":source_count,
              "pre_fec_bit_errors":raw_errors,"pre_fec_valid_bits":coded_count,
              "source_bytes":len(payload),"source_bits":source_count,
              "coded_bits":coded_count,"qam_padding_bits":qam_padding,
              "transmitted_bits":coded_count+qam_padding,"channel_symbols":symbols.numel(),
              "ldpc_blocks":blocks,"information_padding_bits":information_padding,
              "noise_variance_equalized":float(equivalent_n0.mean()),
              "weak_CFR_peak_frames_diagnostic_only":sum(v<10 for v in dsp_diagnostics["cfr_peak_to_mean_power"]),
              "ldpc_nonconverged_blocks":0,"ldpc_iterations_mean":0.,
              "frame_integrity_policy":"Declared BPG header validation/native decoding decides usability; no CRC or truth-based erasure"}
    if ldpc_statistics is not None:
        result.update(ldpc_nonconverged_blocks=int((~ldpc_statistics["converged"]).sum()),
                      ldpc_iterations_mean=float(ldpc_statistics["iterations"].float().mean()))
    return result


@torch.inference_mode()
def evaluate_bpg_condition(codec,streams,images_cpu,metric,physical,condition,names,namespace,
                           device="cpu",ldpc=None,base_seed=20261028,progress=None):
    """Use Fig.4's explicit failure marker plus a separately measured gray LPIPS.

    A successful (possibly damaged/mosaic) BPG image is scored as decoded. Bit
    errors or failed LDPC parity do not force an erasure. Decoder/library errors
    that indicate missing software must propagate rather than become failures.
    """
    if len(images_cpu) != len(names) or set(names) != set(streams) or len(set(names)) != len(names):
        raise ValueError("Image/name/actual BPG stream coverage differs")
    device = torch.device(device)
    records = []
    for index,name in enumerate(names):
        source = streams[name]
        seed = namespace["channel_seed"](condition,name,0,base_seed)
        if physical is None:
            result = {"received_bytes":source,"seed":seed,"bit_errors":0,"valid_bits":len(source)*8,
                      "pre_fec_bit_errors":0,"pre_fec_valid_bits":0,"source_bytes":len(source),
                      "source_bits":len(source)*8,"coded_bits":0,"qam_padding_bits":0,
                      "transmitted_bits":0,"channel_symbols":0,"ldpc_blocks":0,
                      "information_padding_bits":0,"ldpc_nonconverged_blocks":0,"ldpc_iterations_mean":0.}
        else:
            result = transmit_bpg_bytes(source,physical,condition,namespace,seed,device,ldpc)
        try:
            decoded = codec.decode(result.pop("received_bytes"))
        except Exception as error:
            if hasattr(error,"add_note"):
                error.add_note(f"BPG evaluation image={name}, condition={condition.get('id')}")
            raise
        failed = not decoded.ok
        height,width = images_cpu.shape[-2:]
        if failed:
            raster = np.full((height,width,3),128,dtype=np.uint8)
        else:
            raster = np.asarray(decoded.image,dtype=np.uint8)
            if raster.shape != (height,width,3):
                raise ValueError("Codec contract violated: successful decode with wrong shape")
        prediction = torch.from_numpy(raster.copy()).permute(2,0,1).unsqueeze(0).to(device,dtype=torch.float32)/127.5-1.
        target = images_cpu[index:index+1].to(device)
        actual = metric(prediction,target).reshape(-1)
        if actual.numel() != 1 or not bool(torch.isfinite(actual).all()):
            raise FloatingPointError(f"Nonfinite LPIPS for image={name}; do not discard or replace the frame")
        measured = float(actual[0])
        record = {"image_name":str(name),"repeat":0,"lpips":1. if failed else measured,
                  "rendered_lpips":measured,"decode_failed":failed,"decode_status":decoded.status,
                  "decode_error":decoded.error if failed else None,
                  "mse":float((prediction-target).square().mean()),
                  "lpips_policy":"1 is paper-style header-validation/native-decode failure marker; rendered_lpips is separately measured",
                  **result}
        records.append(record)
        if progress is not None:
            progress(index+1,len(names))
    summary = namespace["summarize_records"](records,seed=base_seed)
    summary.update(decode_failures=sum(r["decode_failed"] for r in records),
                   decode_failure_rate=float(np.mean([r["decode_failed"] for r in records])),
                   rendered_lpips_mean=float(np.mean([r["rendered_lpips"] for r in records])),
                   source_bpp_mean=float(np.mean([r["source_bits"]/(height*width) for r in records])),
                   transmitted_bpp_mean=float(np.mean([r["transmitted_bits"]/(height*width) for r in records])),
                   channel_symbols_mean=float(np.mean([r["channel_symbols"] for r in records])),
                   ldpc_nonconverged_blocks=sum(r["ldpc_nonconverged_blocks"] for r in records),
                   pre_fec_ber=sum(r["pre_fec_bit_errors"] for r in records)/max(sum(r["pre_fec_valid_bits"] for r in records),1),
                   failure_policy="Retain all images; declared BPG header-validation/native-decode failure -> paper marker1, companion measured gray LPIPS",
                   metric_scope="Source file bits only for post-FEC BER; coded bits including LDPC padding for pre-FEC BER")
    # BPG has no semantic indices. These denominators are absent rather than
    # evidence of an error-free index/symbol receiver.
    summary["index_error_rate"] = None
    summary["symbol_error_rate"] = None
    return {"version":BPG_EVALUATION_VERSION,"condition":condition,"records":records,"summary":summary}

BPG_COMPARISON_CODE_SHA256 = "2dd25e3b9686592500871ccc478c100286c9317b950f3156ffd5b8f357b83f08"
RATE_CONTROL_CODE_SHA256 = "6404c3d4b6d813659343b4696b17b07e9e3f5f7ee902aa2ed5c469e1344a3cb9"


In [ ]:
"""Received-sample-only rectangular-8QAM carrier acquisition repair.

Append this source after ``fso_sc_signal_dsp.py`` in a notebook.  Standalone
imports are also supported.  QPSK and 16QAM dispatch to the unmodified v1
receiver, and the existing SignalDSPConfig dataclass is not changed.

The fourth-moment periodogram is only a coarse frequency estimator: a
nonuniform finite payload can bias its peak even without receiver noise.
Rectangular 8QAM also has *data-dependent* phase aliases on constant-amplitude
runs; these are not the constellation's pi rotational symmetry.  Independent
BPS windows can therefore jump between aliases on zero-padded coded data.

The repair refines frequency AND constant phase by the AWGN constellation
likelihood, then uses windowed decision-directed phase maximum likelihood
inside the acquired basin.  For fixed decisions that phase estimate is
arg(sum(y * conj(decision))).  No source bits, channel phase, codec state,
parity checks, or image-quality metric enters this receiver.  The initial
phase-branch-near-zero assumption and known real block magnitude are retained
from v1; fundamental blind phase ambiguity is not claimed to be solved.

Related work: Yan et al., JLT 37, 5856-5866 (2019) document nonuniform-QAM
failure of conventional fourth-power frequency estimation.  Meiyappan et al.,
JLT 31, 2055-2069 (2013) study decision-aided ML phase/frequency recovery.
These motivate the estimator family, not a claim that this simple local
likelihood search reproduces either paper's particular algorithm.
"""

import math

import torch
import torch.nn.functional as F


if "recover_qam_carrier" not in globals():
    pass  # already embedded above
_rect8_legacy_recover_qam_carrier = globals().get(
    "_rect8_legacy_recover_qam_carrier", recover_qam_carrier
)


RECT8_DSP_VERSION = "rect8_constellation_ml_cfr_dd_ml_cpr_v2"
RECT8_DSP_REFERENCE = {
    "affected_modulation": "8QAM",
    "scope": "rectangular 8QAM only; QPSK and 16QAM use unchanged v1",
    "coarse_cfr": "v1 fourth-power FFT and Newton periodogram refinement",
    "fine_cfr": "received-sample constellation-distance likelihood; frequency and constant phase jointly fitted",
    "cpr": "two windowed decision-directed ML phase updates, arg(sum(y*conj(decision)))",
    "acquisition_search": "one unoversampled fourth-harmonic CFO bin either side of the coarse estimate",
    "acquisition_samples": "at most 2048 uniformly spaced received samples; no data-dependent sampling",
    "fine_frequency_trials": 17,
    "fine_newton_steps": 5,
    "phase_trials_and_window": "unchanged SignalDSPConfig phase count and window length",
    "phase_ambiguity": "initial phase branch nearest zero; unidentifiable likelihood ties use that convention",
    "no_pilots": True,
    "uses_transmitted_bits_or_symbols": False,
    "uses_true_channel_phase": False,
    "uses_decoder_or_codec_feedback": False,
    "amplitude_default": "known positive real block magnitude only, as in v1",
    "failure_policy": "estimates and errors retained; no frame rejection",
    "legacy_bps_reference": "https://doi.org/10.1109/JLT.2008.2010511",
    "nonuniform_qam_cfr_related_work": "https://opg.optica.org/jlt/abstract.cfm?uri=jlt-37-23-5856",
    "decision_aided_ml_related_work": "https://opg.optica.org/jlt/abstract.cfm?uri=jlt-31-13-2055",
    "related_work_scope": "estimator-family motivation only; neither paper's exact algorithm is reproduced",
}


def rect8_repair_metadata():
    """JSON-safe provenance for the notebook's additional receiver stage."""
    return {"receiver_version": RECT8_DSP_VERSION, **RECT8_DSP_REFERENCE}


def _rect8_nearest_point(value):
    """Exact nearest point for I={-3,-1,1,3}/sqrt(6), Q={-1,1}/sqrt(6)."""
    root = math.sqrt(6.0)
    i = (2 * ((value.real * root + 3) / 2).round().clamp(0, 3) - 3) / root
    q = (2 * (value.imag >= 0).to(value.real.dtype) - 1) / root
    return torch.complex(i, q)


def _rect8_joint_likelihood_acquisition(observed, symbol_rate_baud, config, fit_frequency):
    """Local frequency acquisition with continuous constant-phase refinement.

    A fourth-harmonic Fourier bin corresponds to Fs/(4*N) carrier Hz, or
    pi/2 radians of end-to-end carrier phase.  This fixes the search width
    from the acquisition resolution, not from any image/BER experiment.
    Equally likely phase solutions are resolved by the declared initial
    near-zero phase convention, with a floating-point roundoff tolerance.
    """
    batch, count = observed.shape
    dtype, device = observed.real.dtype, observed.device
    sample_indices = torch.linspace(0, count - 1, min(count, 2048), device=device).round().long()
    sample_time = sample_indices.to(dtype) / (count - 1) - 0.5
    initial_phases = (torch.arange(config.bps_test_phases, device=device, dtype=dtype)
                      / config.bps_test_phases - 0.5) * math.pi
    slopes = (torch.linspace(-math.pi / 2, math.pi / 2, 17, device=device, dtype=dtype)
              if fit_frequency else torch.zeros(1, device=device, dtype=dtype))
    fitted_hz, fitted_intercept, fitted_cost = [], [], []
    for row in observed:
        sample = row[sample_indices]
        scale = sample.abs().amax().clamp_min(1.0)
        phase_candidates, slope_candidates = [], []
        # Bound temporary memory independently of frame/batch length.
        for phase_chunk in initial_phases.split(16):
            angle = phase_chunk[:, None, None] + slopes[None, :, None] * sample_time
            rotated = sample * torch.exp(-1j * angle)
            costs = ((rotated - _rect8_nearest_point(rotated)) / scale).abs().square().mean(-1)
            phase_candidates.append(phase_chunk)
            slope_candidates.append(slopes[costs.argmin(-1)])
        phase = torch.cat(phase_candidates)
        slope = torch.cat(slope_candidates)
        # Newton optimization for the local piecewise-smooth likelihood.
        # Keeping all phase seeds also retains data-dependent ambiguous basins.
        for _ in range(5):
            rotated = sample * torch.exp(-1j * (phase[:, None] + slope[:, None] * sample_time))
            decisions = _rect8_nearest_point(rotated)
            cross = (rotated / scale) * decisions.conj()
            h00 = cross.real.mean(-1)
            g0 = cross.imag.mean(-1)
            if fit_frequency:
                h01 = (cross.real * sample_time).mean(-1)
                h11 = (cross.real * sample_time.square()).mean(-1)
                g1 = (cross.imag * sample_time).mean(-1)
                determinant = h00 * h11 - h01.square()
                stable = (h00 > 0) & (determinant > torch.finfo(dtype).eps * (h00 * h11).abs())
                denominator = torch.where(stable, determinant, torch.ones_like(determinant))
                phase_step = torch.where(stable, (h11 * g0 - h01 * g1) / denominator, 0.0)
                slope_step = torch.where(stable, (h00 * g1 - h01 * g0) / denominator, 0.0)
            else:
                stable = h00 > torch.finfo(dtype).tiny
                phase_step = torch.where(stable, g0 / h00.clamp_min(torch.finfo(dtype).tiny), 0.0)
                slope_step = torch.zeros_like(slope)
            phase = phase + phase_step.clamp(-math.pi / 8, math.pi / 8)
            slope = (slope + slope_step.clamp(-math.pi / 4, math.pi / 4)).clamp(-math.pi / 2, math.pi / 2)
        rotated = sample * torch.exp(-1j * (phase[:, None] + slope[:, None] * sample_time))
        costs = ((rotated - _rect8_nearest_point(rotated)) / scale).abs().square().mean(-1)
        intercept = torch.remainder(phase - slope / 2 + math.pi / 2, math.pi) - math.pi / 2
        tolerance = 32 * torch.finfo(dtype).eps
        eligible = costs <= costs.amin() + tolerance
        selection = torch.where(eligible, intercept.abs(), float("inf")).argmin()
        fitted_hz.append(slope[selection] * symbol_rate_baud / (2 * math.pi * (count - 1)))
        fitted_intercept.append(intercept[selection])
        fitted_cost.append(costs[selection])
    frequency = torch.stack(fitted_hz)
    intercept = torch.stack(fitted_intercept)
    index = torch.arange(count, device=device, dtype=dtype)
    phase = intercept[:, None] + frequency[:, None] * (2 * math.pi / symbol_rate_baud) * index
    return observed * torch.exp(-1j * phase), frequency, intercept, torch.stack(fitted_cost)


def _rect8_decision_directed_phase(observed, config):
    """Windowed constant-phase ML, initialized inside the acquired basin."""
    corrected = observed
    scale = observed.abs().amax(-1, keepdim=True).clamp_min(1.0)
    phase = torch.zeros_like(observed.real)
    unwrap_events = torch.zeros(observed.shape[0], device=observed.device, dtype=torch.int64)
    for _ in range(2):
        decisions = _rect8_nearest_point(corrected)
        cross = (observed / scale) * decisions.conj()
        kwargs = dict(kernel_size=config.bps_window_symbols, stride=1,
                      padding=config.bps_window_symbols // 2, count_include_pad=False)
        average = torch.complex(F.avg_pool1d(cross.real[:, None], **kwargs)[:, 0],
                                F.avg_pool1d(cross.imag[:, None], **kwargs)[:, 0])
        phase, unwrap_events = _unwrap_symmetry_phase(average.angle(), 2 * math.pi)
        corrected = observed * torch.exp(-1j * phase)
    return corrected, phase, unwrap_events


@torch.no_grad()
def recover_qam_carrier(received, modulation, symbol_rate_baud, config=None, known_block_magnitude=None):
    """V1-compatible API; only rectangular-8QAM takes the repaired path."""
    if modulation.upper() != "8QAM":
        return _rect8_legacy_recover_qam_carrier(
            received, modulation, symbol_rate_baud, config, known_block_magnitude
        )
    config = SignalDSPConfig() if config is None else config
    if received.ndim != 2 or not torch.is_complex(received) or received.shape[-1] < 8:
        raise ValueError("Expected complex [batch, symbols>=8] receiver samples")
    if symbol_rate_baud <= 0 or not torch.isfinite(received).all():
        raise ValueError("Invalid receiver samples or symbol rate")
    if config.amplitude_mode == "known_block_magnitude":
        if known_block_magnitude is None or torch.is_complex(known_block_magnitude):
            raise ValueError("Positive REAL block-amplitude CSI is required")
        if not torch.isfinite(known_block_magnitude).all() or (known_block_magnitude <= 0).any():
            raise ValueError("Invalid block-amplitude CSI")
        amplitude = torch.broadcast_to(known_block_magnitude, (received.shape[0], 1))
    else:
        amplitude = received.abs().square().mean(-1, keepdim=True).sqrt()
        amplitude = amplitude.clamp_min(torch.finfo(received.real.dtype).tiny)
    observed = received / amplitude
    zeros = torch.zeros(received.shape[0], device=received.device, dtype=received.real.dtype)
    if config.cfo_enabled:
        observed, coarse_hz, peak_ratio = _estimate_cfo_fourth_power(observed, symbol_rate_baud, config)
    else:
        coarse_hz, peak_ratio = zeros, zeros
    if config.cfo_enabled or config.cpr_enabled:
        acquired, fine_hz, initial_phase, cost = _rect8_joint_likelihood_acquisition(
            observed, symbol_rate_baud, config, config.cfo_enabled
        )
        if config.cpr_enabled:
            observed = acquired
        else:
            index = torch.arange(received.shape[-1], device=received.device, dtype=received.real.dtype)
            observed = observed * torch.exp(-1j * fine_hz[:, None] * (2 * math.pi / symbol_rate_baud) * index)
    else:
        fine_hz, initial_phase, cost = zeros, zeros, zeros
    if config.cpr_enabled:
        observed, phase, unwrap_events = _rect8_decision_directed_phase(observed, config)
        phase = phase + initial_phase[:, None]
    else:
        phase = torch.zeros_like(received.real)
        unwrap_events = zeros.to(torch.int64)
    diagnostics = {
        "receiver_version": RECT8_DSP_VERSION,
        "amplitude_mode": config.amplitude_mode,
        "estimated_cfo_hz": (coarse_hz + fine_hz).detach().cpu().tolist(),
        "coarse_cfo_hz": coarse_hz.detach().cpu().tolist(),
        "fine_cfo_correction_hz": fine_hz.detach().cpu().tolist(),
        "acquisition_normalized_distance": cost.detach().cpu().tolist(),
        "cfr_peak_to_mean_power": peak_ratio.detach().cpu().tolist(),
        "cfr_unambiguous_interval_hz": [-symbol_rate_baud / 8, symbol_rate_baud / 8],
        "estimated_phase_start_rad": phase[:, 0].detach().cpu().tolist(),
        "estimated_phase_end_rad": phase[:, -1].detach().cpu().tolist(),
        "phase_symmetry_unwrap_events": unwrap_events.detach().cpu().tolist(),
        "failure_policy": "estimates/errors retained, no frame rejection",
    }
    return observed, diagnostics


def rect8_repair_self_test(device="cpu"):
    """Small synthetic preflight; no model, image, codec, or external data."""
    device = torch.device(device)
    generator = torch.Generator(device=device).manual_seed(20261028)
    count, sample_rate = 7168, 28e9
    ids = torch.randint(8, (1, count), generator=generator, device=device)
    ids[:, 5000:] = 0
    i = torch.tensor([-3, -3, -1, -1, 3, 3, 1, 1], device=device) / math.sqrt(6)
    q = torch.tensor([-1, 1, -1, 1, -1, 1, -1, 1], device=device) / math.sqrt(6)
    points = torch.complex(i, q)
    source = points[ids]
    index = torch.arange(count, device=device, dtype=source.real.dtype)
    records = []
    for cfo in (0.0, 700e6):
        received = source * torch.exp(1j * (0.03 + 2 * math.pi * cfo / sample_rate * index))
        recovered, diagnostics = recover_qam_carrier(
            received, "8QAM", sample_rate, known_block_magnitude=torch.ones((1, 1), device=device)
        )
        errors = int(((recovered[..., None] - points).abs().argmin(-1) != ids).sum())
        if errors:
            raise AssertionError(("rect8 preflight failed", cfo, errors, diagnostics))
        records.append({"cfo_hz": cfo, "symbol_errors": errors,
                        "estimated_cfo_hz": diagnostics["estimated_cfo_hz"],
                        "acquisition_normalized_distance": diagnostics["acquisition_normalized_distance"]})
    return {"receiver_version": RECT8_DSP_VERSION, "synthetic_only": True,
            "symbols_per_condition": count, "zero_tail_symbols": 2168, "conditions": records}


In [ ]:
"""Rate-only, dataset-mean matching of genuine native BPG streams.

The official libbpg 0.9.8 CLI parses ``-q`` with ``atoi`` and its x265 glue
uses integer constant-QP mode; fractional QP is not an exposed rate control.
Source: https://bellard.org/bpg/libbpg-0.9.8.tar.gz (bpgenc.c, x265_glue.c).

For each image we retain an under-target native stream and the first tested
lower-QP stream above the target. A deterministic subset-sum then promotes
some images using *only encoded byte counts*. No reconstruction, metric,
SNR, decoder outcome, padding, metadata stuffing, or source-image change is
used. Freeze the selected streams before evaluating any channel condition.

This matches the DATASET MEAN, not each image's rate. Individual streams may
exceed the per-image reference. Report their actual minimum/maximum/rates.
Subset-sum is exact over the retained candidates, not over all possible
codec settings. The default native QP search remains explicitly non-global.
The legacy strict-budget codec is deliberately unchanged.
"""

from dataclasses import dataclass
import hashlib
import math
import operator
from types import MappingProxyType



RATE_CONTROL_VERSION = "bpg_native_integer_qp_dataset_mean_v1"


def _positive_integer(value, name):
    if isinstance(value, bool):
        raise ValueError(f"{name} must be a positive integer")
    try:
        value = operator.index(value)
    except TypeError as exc:
        raise ValueError(f"{name} must be a positive integer") from exc
    if value < 1:
        raise ValueError(f"{name} must be a positive integer")
    return value


@dataclass(frozen=True)
class NativeBPGCandidate:
    data: bytes
    q: int
    image_hw: tuple

    def __post_init__(self):
        if not isinstance(self.data, bytes) or not self.data.startswith(BPG_MAGIC):
            raise ValueError("candidate must contain complete native BPG bytes")
        if isinstance(self.q, bool) or not isinstance(self.q, int) or not 0 <= self.q <= 51:
            raise ValueError("candidate q must be an integer from 0 to 51")
        if len(self.image_hw) != 2 or any(_positive_integer(v, "image dimension") != v for v in self.image_hw):
            raise ValueError("candidate image_hw must contain two positive integers")

    @property
    def num_bytes(self):
        return len(self.data)

    @property
    def bpp(self):
        return 8 * self.num_bytes / math.prod(self.image_hw)

    @property
    def sha256(self):
        return hashlib.sha256(self.data).hexdigest()

    def metadata(self):
        return {"q": self.q, "actual_bytes": self.num_bytes, "actual_bpp": self.bpp,
                "image_hw": list(self.image_hw), "sha256": self.sha256,
                "bpg_header_included": True, "padding_bytes": 0}


@dataclass(frozen=True)
class BPGRateBracket:
    image_name: str
    target_bytes: int
    low: NativeBPGCandidate
    high: object
    tested_q_sizes: tuple
    search_mode: str
    optimality: str
    encoder_settings: dict
    cache_hits: int = 0

    def __post_init__(self):
        if not isinstance(self.image_name, str) or not self.image_name:
            raise ValueError("image_name must be a nonempty string")
        _positive_integer(self.target_bytes, "target_bytes")
        if self.low.num_bytes > self.target_bytes:
            raise ValueError("low candidate exceeds the reference target")
        if self.high is not None:
            if self.high.num_bytes <= self.target_bytes or self.high.q >= self.low.q:
                raise ValueError("high candidate must exceed target at a lower integer QP")
            if self.high.image_hw != self.low.image_hw:
                raise ValueError("candidate image dimensions differ")

    def metadata(self):
        return {"image_name": self.image_name, "target_bytes": self.target_bytes,
                "low": self.low.metadata(),
                "high": self.high.metadata() if self.high is not None else None,
                "tested_q_sizes": {str(q): size for q, size in self.tested_q_sizes},
                "tested_q_count": len(self.tested_q_sizes), "search_mode": self.search_mode,
                "optimality": self.optimality, "encoder_settings": dict(self.encoder_settings),
                "cache_hits": self.cache_hits}


def encode_rate_bracket(codec, image, image_name, target_bytes=1792,
                        search_mode="binary_checked"):
    """Measure an under/over integer-QP pair without decoding or scoring.

    ``low`` means fewer bytes, not lower QP. The higher-rate candidate is
    the first actual over-target stream encountered below low.q. Already
    tested same-image results are reused by the unmodified native codec.
    ``exhaustive`` proves the lowest feasible QP; the default does not.
    """
    target_bytes = _positive_integer(target_bytes, "target_bytes")
    encoded = codec.encode_to_budget(image, target_bytes, search_mode=search_mode)
    low = NativeBPGCandidate(encoded.data, encoded.q, tuple(encoded.image_hw))
    sizes = dict(encoded.tested_q_sizes)
    high = None
    cache_before = getattr(codec, "_cache_hits", 0)
    for q in range(low.q - 1, -1, -1):
        data = codec.encode(image, q)
        sizes[q] = len(data)
        candidate = NativeBPGCandidate(data, q, tuple(encoded.image_hw))
        if candidate.num_bytes > target_bytes:
            high = candidate
            break
        low = candidate
    return BPGRateBracket(image_name, target_bytes, low, high, tuple(sorted(sizes.items())),
                          encoded.search_mode, encoded.optimality,
                          dict(encoded.encoder_settings), encoded.cache_hits +
                          getattr(codec, "_cache_hits", 0) - cache_before)


@dataclass(frozen=True)
class SelectedBPG:
    candidate: NativeBPGCandidate
    bracket: BPGRateBracket
    selected_level: str

    @property
    def data(self):
        return self.candidate.data

    @property
    def q(self):
        return self.candidate.q

    @property
    def num_bytes(self):
        return self.candidate.num_bytes

    @property
    def bpp(self):
        return self.candidate.bpp

    def metadata(self):
        result = self.candidate.metadata()
        result.update(codec=BPG_CODEC_VERSION, rate_control=RATE_CONTROL_VERSION,
                      rate_matching_scope="dataset_mean_not_per_image",
                      per_image_reference_bytes=self.bracket.target_bytes,
                      per_image_hard_cap_bytes=None, selected_level=self.selected_level,
                      tested_q_sizes={str(q): size for q, size in self.bracket.tested_q_sizes},
                      tested_q_count=len(self.bracket.tested_q_sizes),
                      search_mode=self.bracket.search_mode, optimality=self.bracket.optimality,
                      cache_hits=self.bracket.cache_hits,
                      encoder_settings=dict(self.bracket.encoder_settings),
                      bracket=self.bracket.metadata())
        return result


@dataclass(frozen=True)
class MatchedBPGSources:
    entries: object
    target_bytes_per_image: int
    seed: int
    subset_sum_states: int

    def metadata(self):
        entries = list(self.entries.values())
        sizes = [entry.num_bytes for entry in entries]
        target_total = len(entries) * self.target_bytes_per_image
        total = sum(sizes)
        total_pixels = sum(math.prod(entry.candidate.image_hw) for entry in entries)
        manifest = "\n".join(f"{name}\t{entry.q}\t{entry.candidate.sha256}"
                             for name, entry in sorted(self.entries.items()))
        return {"rate_control": RATE_CONTROL_VERSION, "selection_inputs": ["image_name", "encoded_byte_count"],
                "quality_metric_used_for_selection": False, "snr_used_for_selection": False,
                "rate_matching_scope": "dataset_mean_not_per_image", "num_images": len(entries),
                "target_bytes_per_image": self.target_bytes_per_image, "target_total_bytes": target_total,
                "actual_total_bytes": total, "aggregate_shortfall_bytes": target_total - total,
                "mean_actual_bytes": total / len(entries), "actual_mean_bpp": total * 8 / total_pixels,
                "target_mean_bpp": target_total * 8 / total_pixels,
                "min_actual_bytes": min(sizes), "max_actual_bytes": max(sizes),
                "min_actual_bpp": min(entry.bpp for entry in entries),
                "max_actual_bpp": max(entry.bpp for entry in entries),
                "per_image_hard_cap_bytes": None,
                "images_above_per_image_reference": sum(size > self.target_bytes_per_image for size in sizes),
                "selected_high_count": sum(entry.selected_level == "high" for entry in entries),
                "selected_low_count": sum(entry.selected_level == "low" for entry in entries),
                "padding_bytes": 0, "seed": self.seed, "subset_sum_states": self.subset_sum_states,
                "aggregate_optimality": "largest achievable total <= target over retained native candidate pairs",
                "qp_search_global_optimality_claimed": False,
                "source_manifest_sha256": hashlib.sha256(manifest.encode()).hexdigest(),
                "reuse_policy": "freeze this exact mapping across both BPG branches and every channel condition"}


def select_rate_matched_sources(brackets, target_bytes_per_image=1792, seed=20261028):
    """Exact bounded subset-sum on measured size increments, never quality.

    Ties use seeded image-name hashes, independently of caller iteration
    order. The aggregate never exceeds N*target, although individual images
    can exceed target. An unattainable target is reported as a real shortfall.
    Runtime/memory scale with aggregate shortfall, not with image pixels.
    """
    target = _positive_integer(target_bytes_per_image, "target_bytes_per_image")
    if isinstance(seed, bool):
        raise ValueError("seed must be an integer")
    try:
        seed = operator.index(seed)
    except TypeError as exc:
        raise ValueError("seed must be an integer") from exc
    brackets = list(brackets.values() if hasattr(brackets, "values") else brackets)
    if not brackets:
        raise ValueError("at least one rate bracket is required")
    names = [bracket.image_name for bracket in brackets]
    if len(set(names)) != len(names):
        raise ValueError("image names must be unique")
    if any(bracket.target_bytes != target for bracket in brackets):
        raise ValueError("all brackets must use the requested target")
    ordered = sorted(brackets, key=lambda item: (hashlib.sha256(
        f"{seed}\0{item.image_name}".encode()).digest(), item.image_name))
    deficit = len(ordered) * target - sum(bracket.low.num_bytes for bracket in ordered)
    mask = (1 << (deficit + 1)) - 1
    reachable, history = 1, [1]
    increments = []
    for bracket in ordered:
        increment = bracket.high.num_bytes - bracket.low.num_bytes if bracket.high is not None else 0
        increments.append(increment)
        # A huge out-of-range increment must not allocate a huge bigint.
        if 0 < increment <= deficit:
            reachable = (reachable | reachable << increment) & mask
        history.append(reachable)
    remaining = reachable.bit_length() - 1
    selected = {}
    for index in range(len(ordered) - 1, -1, -1):
        bracket = ordered[index]
        use_high = not ((history[index] >> remaining) & 1)
        candidate = bracket.high if use_high else bracket.low
        if use_high:
            remaining -= increments[index]
        selected[bracket.image_name] = SelectedBPG(candidate, bracket, "high" if use_high else "low")
    if remaining != 0:
        raise AssertionError("subset-sum reconstruction failed")
    selected = MappingProxyType(dict(sorted(selected.items())))
    return MatchedBPGSources(selected, target, seed, reachable.bit_count())


def prepare_rate_matched_sources(codec, images, target_bytes_per_image=1792,
                                 seed=20261028, search_mode="binary_checked", progress=None):
    """Encode then freeze one mean-matched source mapping.

    ``images`` is a mapping or iterable of (name, PIL RGB image). ``progress``
    receives (completed_count, total_count, bracket) after each native image;
    it is a status callback and must not mutate the source-selection policy.
    For parallel/pre-cached preparation call encode_rate_bracket separately
    and pass the complete fixed set to select_rate_matched_sources.
    """
    items = list(images.items() if hasattr(images, "items") else images)
    if len({name for name, _ in items}) != len(items):
        raise ValueError("image names must be unique")
    brackets = []
    for name, image in items:
        bracket = encode_rate_bracket(codec, image, name, target_bytes_per_image, search_mode)
        brackets.append(bracket)
        if progress is not None:
            progress(len(brackets), len(items), bracket)
    return select_rate_matched_sources(brackets, target_bytes_per_image, seed)


In [ ]:
"""Result-blind, exact-file-disjoint final holdout selection and verification.

This module reads identities only: no images are decoded, evaluated or written.
Exact SHA-256 separation does not establish geographic or near-image independence.
Freeze the returned manifest once; evaluation must verify it, never select anew.
"""

import hashlib
import json
from pathlib import Path, PurePosixPath
import zipfile


SELECTION_VERSION = "fso_sc_holdout_sha256_rank_v1"
MANIFEST_VERSION = "fso_sc_final_holdout_manifest_v1"
DEFAULT_SEED = 20261029
IMAGE_SUFFIXES = frozenset({".png", ".jpg", ".jpeg"})


def _require(ok, message):
    if not ok:
        raise ValueError("Holdout verification failed: " + message)


def canonical_sha256(value):
    """Hash canonical UTF-8 JSON; this is not the hash of pretty-printed JSON."""
    raw = json.dumps(value, sort_keys=True, ensure_ascii=False, allow_nan=False,
                     separators=(",", ":")).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as source:
        for chunk in iter(lambda: source.read(8 * 1024**2), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _sha(value):
    return isinstance(value, str) and len(value) == 64 and all(
        character in "0123456789abcdef" for character in value)


def _names(values, label):
    _require(not isinstance(values, (str, bytes)), f"{label} must be a sequence")
    try:
        result = list(values)
    except TypeError as error:
        raise ValueError(f"Holdout verification failed: invalid {label}") from error
    _require(all(isinstance(name, str) and name for name in result), f"invalid {label}")
    _require(len(result) == len(set(result)), f"duplicate {label}")
    return result


def _inventory(hash_by_name, label):
    _require(isinstance(hash_by_name, dict) and hash_by_name, f"empty/invalid {label}")
    names = _names(hash_by_name, label + " names")
    _require(all(_sha(hash_by_name[name]) for name in names), f"invalid SHA-256 in {label}")
    _require(len(set(hash_by_name.values())) == len(names), f"duplicate exact files in {label}")
    return sorted(names)


def zip_image_inventory(path, expected_images=None):
    """Return all PNG/JPEG member file hashes, rejecting duplicate names/content.

    Streams uncompressed member bytes without extraction or raster decoding.
    Nonimage metadata (for example a collection manifest) is not selected.
    """
    result = {}
    with zipfile.ZipFile(path) as archive:
        for info in archive.infolist():
            if info.is_dir() or PurePosixPath(info.filename).suffix.lower() not in IMAGE_SUFFIXES:
                continue
            name = info.filename
            parts = name.split("/")
            _require("\\" not in name and not name.startswith("/")
                     and all(part not in {"", ".", ".."} for part in parts)
                     and ":" not in parts[0], f"unsafe image member name: {name!r}")
            _require(name not in result, f"duplicate image member name: {name!r}")
            digest = hashlib.sha256()
            with archive.open(info) as source:
                for chunk in iter(lambda: source.read(8 * 1024**2), b""):
                    digest.update(chunk)
            result[name] = digest.hexdigest()
    names = _inventory(result, "ZIP image inventory")
    if expected_images is not None:
        _require(type(expected_images) is int and expected_images > 0,
                 "expected_images must be a positive integer")
        _require(len(names) == expected_images, "ZIP image count differs from frozen expectation")
    return {name: result[name] for name in names}


def select_holdout_names(names, development_names, development_file_hashes,
                         hash_by_name, count=200, seed=DEFAULT_SEED):
    """Return sorted, result-blind names excluding prior names AND exact files.

    The old development list and its aligned hashes must match this same pool.
    All pool files must be distinct: a renamed duplicate fails closed rather
    than silently altering the population or enabling a partition-leakage claim.
    Content-hash ranking fixes selection across Python versions, input order,
    and file renaming. No filename/coordinate is needed in a public manifest.
    """
    names = _names(names, "pool names")
    pool_names = _inventory(hash_by_name, "test pool")
    _require(set(names) == set(pool_names), "pool names/hash coverage differs")
    development_names = _names(development_names, "development names")
    _require(development_names, "development list must not be empty")
    development_file_hashes = list(development_file_hashes)
    _require(len(development_names) == len(development_file_hashes)
             and all(_sha(value) for value in development_file_hashes),
             "development names/hash coverage differs")
    _require(len(set(development_file_hashes)) == len(development_file_hashes),
             "duplicate exact files in development list")
    for name, digest in zip(development_names, development_file_hashes):
        _require(name in hash_by_name and hash_by_name[name] == digest,
                 f"development identity differs: {name!r}")
    _require(type(count) is int and count > 0, "count must be a positive integer")
    _require(type(seed) is int, "seed must be an integer")
    old_names, old_hashes = set(development_names), set(development_file_hashes)
    eligible = [name for name in pool_names if name not in old_names
                and hash_by_name[name] not in old_hashes]
    _require(len(eligible) >= count, "insufficient exact-file-disjoint candidates")
    ranked = sorted(eligible, key=lambda name: (
        hashlib.sha256(f"{seed}\0{hash_by_name[name]}".encode("ascii")).hexdigest(),
        hash_by_name[name]))
    return sorted(ranked[:count])


def build_holdout_manifest(hash_by_name, development_names, development_file_hashes, *,
                           test_zip_sha256, previous_summary_sha256, count=200,
                           seed=DEFAULT_SEED, expected_total=1000,
                           training_hash_by_name=None, training_zip_sha256=None):
    """Construct metadata to freeze once before viewing any new holdout result.

    An optional training inventory provides a pinned, local exact-file audit;
    without it the manifest explicitly says ``not_verified``. The training ZIP
    is not needed again at evaluation time. This audit says nothing about nearby
    patches or the provenance of a checkpoint's actual training inputs.
    """
    all_names = _inventory(hash_by_name, "test pool")
    _require(type(expected_total) is int and expected_total > 0
             and len(all_names) == expected_total, "test pool size differs from expectation")
    development_names = list(development_names)
    development_file_hashes = list(development_file_hashes)
    selected = select_holdout_names(all_names, development_names, development_file_hashes,
                                   hash_by_name, count=count, seed=seed)
    _require(_sha(test_zip_sha256) and _sha(previous_summary_sha256),
             "missing/invalid input file SHA-256")
    development_hashes = sorted(development_file_hashes)
    training_audit = {"status": "not_verified", "scope": "exact file SHA-256 only"}
    if training_hash_by_name is None:
        _require(training_zip_sha256 is None, "training ZIP hash without training inventory")
    else:
        training_names = _inventory(training_hash_by_name, "training inventory")
        _require(_sha(training_zip_sha256), "missing/invalid training ZIP SHA-256")
        shared_names = set(all_names).intersection(training_names)
        shared_hashes = set(hash_by_name.values()).intersection(training_hash_by_name.values())
        _require(not shared_names and not shared_hashes, "test/training exact identity overlap")
        training_audit.update({
            "status": "verified_no_overlap", "training_images": len(training_names),
            "training_zip_sha256": training_zip_sha256,
            "training_inventory_sha256": canonical_sha256(training_hash_by_name),
            "all_test_name_overlap_count": 0, "all_test_file_sha256_overlap_count": 0,
            "selected_file_sha256_overlap_count": 0,
            "checkpoint_training_provenance": "not_established_by_this_file_audit",
        })
    manifest = {
        "version": MANIFEST_VERSION, "selection_version": SELECTION_VERSION,
        "identifiers": "sha256_file_content",
        "seed": seed, "count": count, "expected_total_images": expected_total,
        "eligible_images": len(all_names) - len(development_names),
        "test_zip_sha256": test_zip_sha256,
        "previous_summary_sha256": previous_summary_sha256,
        "previous_list_sha256": canonical_sha256(development_hashes),
        "all_file_sha256": sorted(hash_by_name.values()),
        "development_file_sha256": development_hashes,
        "selected_file_sha256": sorted(hash_by_name[name] for name in selected),
        "training_exact_file_audit": training_audit,
        "independence_scope": "exact files only; geographic/near-image independence not established",
        "selection_policy": "one frozen result-blind selection; no result-based reselection",
    }
    manifest["manifest_sha256"] = canonical_sha256(manifest)
    return manifest


def verify_holdout_manifest(manifest, hash_by_name, *, test_zip_sha256=None,
                            expected_manifest_sha256=None):
    """Verify frozen membership and complete pool before evaluation; never write.

    Training audit metadata describes the frozen local audit, not a fresh scan.
    Pass an independently pinned manifest hash to detect wholesale substitution.
    """
    _require(isinstance(manifest, dict), "manifest must be an object")
    actual_digest = canonical_sha256({key: value for key, value in manifest.items()
                                      if key != "manifest_sha256"})
    _require(manifest.get("manifest_sha256") == actual_digest, "manifest digest differs")
    if expected_manifest_sha256 is not None:
        _require(_sha(expected_manifest_sha256) and actual_digest == expected_manifest_sha256,
                 "manifest differs from independent pin")
    _require(manifest.get("version") == MANIFEST_VERSION
             and manifest.get("selection_version") == SELECTION_VERSION
             and manifest.get("identifiers") == "sha256_file_content",
             "unsupported frozen manifest/selection version")
    all_names = _inventory(hash_by_name, "test pool")
    _require(type(manifest.get("expected_total_images")) is int
             and len(all_names) == manifest["expected_total_images"], "test pool size differs")
    _require(manifest.get("all_file_sha256") == sorted(hash_by_name.values()),
             "complete frozen pool identity differs")
    _require(_sha(manifest.get("test_zip_sha256"))
             and _sha(manifest.get("previous_summary_sha256")), "invalid provenance file SHA-256")
    if test_zip_sha256 is not None:
        _require(test_zip_sha256 == manifest["test_zip_sha256"], "test ZIP SHA-256 differs")
    development_hashes = manifest.get("development_file_sha256")
    _require(isinstance(development_hashes, list) and development_hashes
             and all(_sha(value) for value in development_hashes)
             and development_hashes == sorted(set(development_hashes)),
             "invalid frozen development hashes")
    _require(manifest.get("previous_list_sha256") == canonical_sha256(development_hashes),
             "previous development-list digest differs")
    name_by_hash = {digest: name for name, digest in hash_by_name.items()}
    _require(all(value in name_by_hash for value in development_hashes),
             "frozen development identities missing from pool")
    development_names = [name_by_hash[value] for value in development_hashes]
    selected = select_holdout_names(all_names, development_names,
        development_hashes, hash_by_name,
        count=manifest.get("count"), seed=manifest.get("seed"))
    _require(manifest.get("eligible_images") == len(all_names) - len(development_hashes),
             "eligible count differs")
    _require(manifest.get("selected_file_sha256") == sorted(hash_by_name[name] for name in selected),
             "frozen holdout membership differs from result-blind rule")
    audit = manifest.get("training_exact_file_audit")
    _require(isinstance(audit, dict) and audit.get("status") in {"not_verified", "verified_no_overlap"}
             and audit.get("scope") == "exact file SHA-256 only", "invalid training audit")
    if audit["status"] == "verified_no_overlap":
        _require(type(audit.get("training_images")) is int and audit["training_images"] > 0
                 and _sha(audit.get("training_zip_sha256"))
                 and _sha(audit.get("training_inventory_sha256")), "invalid training audit provenance")
        _require(all(type(audit.get(key)) is int and audit[key] == 0 for key in (
            "all_test_name_overlap_count", "all_test_file_sha256_overlap_count",
            "selected_file_sha256_overlap_count")), "nonzero/invalid training overlap audit")
    return selected

FROZEN_HOLDOUT_MANIFEST = {'version': 'fso_sc_final_holdout_manifest_v1', 'selection_version': 'fso_sc_holdout_sha256_rank_v1', 'identifiers': 'sha256_file_content', 'seed': 20261029, 'count': 200, 'expected_total_images': 1000, 'eligible_images': 800, 'test_zip_sha256': '558916f53c3a1cabf04d3bda5d061a120a0ab3c5bc597bb56956e2f0e6adf21a', 'previous_summary_sha256': 'e24e6afca12576b8a87438b9be1e4572e56151b9729ead361024b58d83752933', 'previous_list_sha256': '49d62f7eb925dc0b5418c47a08541e81b51d1b078717840c3fc0c1cda45552a4', 'all_file_sha256': ['...'], 'development_file_sha256': ['0066845163f38a25828bfa1e29516b0a472661a09893d66fdf611694ad8f8284', '0127d7350d9710721527acfacfed0492dc9bacb0cee58108f994bee8cf03fec3', '0371dd8e6d255c728c45103c8adc87fbf93770559b1b2534f4fc848d1a58b9ea', '03b71f226ea44462a1ff05875a802b60fe89c41a952f879979956d32b8cc57c6', '0428ba35ff505fa6cedc57e046a455ade950dcd487cdfee7a94743c494a68d2d', '04dff21d7d781ff8ea2ed80f6da1a165e7797f9a398b4fd930bb935d84fae8d8', '05fa9cbc11ca5d438f4a7fd8f2f263f85f0f014f034dcc099a711dee58022289', '0679c8b0537a146003135e865121b10b489c18b661562795074a4c46987f2edd', '075cf56c692f03d94602f7174152164e29f6fcc3e213fe20b86394d29c044af2', '0e2ddf287e7584f51552daf7d41f4d85e9cd406da782015f5031a95cd95871a0', '0fa3fae44c1eb015fe9adc1c7dd2da907acce30ef080449e5b1d53b71a9d843b', '1269e5f1b5b64c8442881cf6aae6380d387eced7312ec5e89d677466967ee3d2', '13cf06d91aecfbbf8ce84553c0cee40fb0a2f8a428aa78434b5371780686a56d', '141fe37ce0253cb31db8fd9b604c4c9673f5b7de9aea16d7c00c4758aa6f35e6', '1628c2c04b97b49be78116becb52a0c881654de00604da82928aa7b4dae90013', '176260b73073aae1dcb99f05af28555b4682c100e063d4d4c45aebc6bb209a48', '18d63182f56c67e61fc78dc3179ad89ab7e443d6c95f431721202fe0129de547', '18e95729761855bd29115435ea4aa3720418e954fab04dc1996a423317d5ba75', '1932b91f7bbe4cde3fe574328ab25ee7c7c92f8a1c8889ade9a5bcce8bb4a6fa', '19c4310b6e09bc4b22082dfb82e88d33c8fd53daea76c08bbf32c099c93e22e2', '1b8930c4e710f3484d1b6bbde884a4c24f83cd8b77a694a59716e849ee506841', '1bf9feae32fe5ce0a380a3d37b4754ce1b18b63d05d6bb931854519ea1a661b6', '1d36b5bb3a75f95e0c2d624ba5acf329117b2a331f7f83624b0a819966a5e605', '1ed1c79090c93e3f3ba4503b11b19d8f4a027c9ddcb4394fc3b6859584ea80f8', '20f7ff1137e25b4238efe36281032f8e219e48ff2bec837c02d6941c83b2c46c', '2295f88b3afc6eb315197029301c0611399638894e36d518de076c7c95112cf2', '241f921d4cd85481e077b18605fb2d2b1ae4d52f954c86a8a743e603af0b75ec', '24cd687a287375d73d5cf90864e7c0314a180fe81af899cce5fe86ddb7f26d64', '2591258305dd07328766dcffb7232b23ae78a88a61d9a878ed913105cd21c3e8', '26b9c2917759aa9d90241b26b7fc1de4e64ad52b1a6307a5ba33095d054e19c2', '279cc38806a4e37f44a799845db932d4839b34e7b14b933146bcff349661d78f', '27eb5e3c07fcc322a18a8a5a63fd03877e16b86c0516e6fe4cd656da5e280c05', '29f63c30a424c41864cf024bff7badecc8a25f141f50236c7eaca44c851d148f', '2a40eac16287efd2e163305ea3a4b517ab642c23b229cc77ffdeb6f600e4e544', '2e63938b2fcd265175ae05748f4e266ac8dc01bd2a38294bd42d515f3a21ac85', '2e8151a6bb8aeec62a83a29bdb93303bb7265cb28ef5deb9fc2bb6f933e4801e', '2fad80f5c24928cfd53dce72ff07573f693291059baf3d78a37b139b195b8796', '3084c85b7fa890e77560bc72e757fadae863d9d32bfa30b3cb28cd74188ebcd9', '30c131076a387144a12eea1f98f9a4fca2f5af812c0e4a57d7325adff5fb0c49', '3220a60a4764025feb88510610f61f0ba228041e66a56f3dc4a332ce1b9a884b', '32aaab923cda5b9aa647da5818096ed7c6f2c9f857bf5c04f6238428fcd9bf4d', '32b668e9e325e12ef684de614ce85d30fd2b2dd3496fa6a196f3f70270a65728', '332c392379f7a2307826bc57368d861dae2c596475f3193d861ca586cf3ad873', '339b927f038ba1c46dff3a7497ba07609578b8a85ee910819aebe439770f57dd', '3558079ab6c1f10699922b3dae791721a79dea43128312de14bcc85f46d9165b', '36559d2545c16bb7eca9ee449b893c48596d2114b7b000a1819e16d4035e826e', '367126122831aa7424a9498c641885faf8461456fd6e86b1d8b915d920401d81', '36e5a076cf2ea05749a06cb2bcc26e4fdf39cc250f8139d40b8182d03c7148f4', '37e7e20d527ee6051d49773937c09beb8d178c20de96786ec44494927e9e5190', '38285e3424177b69ec49d8db865a0c83452012dd4a00b46e9dda6f406836110c', '383f1af4bb2eaa49811f0027d7ec0f89db9b340aef36616367c692b5c81d5f24', '384e0cda59102c2e30428d0327ed6d9de19a1ed8a5634423fc8d1456d950f920', '38ef6a56173fd38bd4129130a9632bc3767d5369a2d486900f8568e43eb8d9f4', '38f327e85514eb45fef5ae3eb06117ab2e16df93ec4a8b85e475d3e04265c69b', '39e1dba8068d2af620de27421472afbe76be97fdad46683b25745fc6deb883a8', '3a4cde911a3bec8da83cd0b644ee297788d5b85a0af9f2eff2a5cfcd779c333e', '3b8b929104cf13efab270b7e56ba0f38183857955095ec84191495eac11c740f', '3cbfaa6d4563c235b0907c2375dd44f3b96d591099def6946e870d66a2a86547', '3dcd17e02aae26c95a49837a391966b0b650d44d575c751e3a47bb1c73d3c23e', '40d9ebc0983e831d0f8780cac97d2dd788a681bc11f56f85d429311dceee5902', '413f07cfa1eb10f6413ea8d977b392d05de767bf4bda512038954b9a004a13a0', '41ff58423114103524a85024eb29549a416af37b154de6966992f43847ca0564', '446896b6489600ee23701379b84232fca38edc64e8c4cc90efeb579006be6662', '4477fcdd39a8d7496347489d19943f5e3034a19fdb1fbe7efc4ac80f967a71f8', '45b78077660897b40a0f02bd1166111058fc7a0c885f090a64562b5c95af7791', '46533741c06ed67856bbaf606dcd67357979e06c9434991d8cbc88aa5ede851d', '47d78f81622fab2b16724c81c6cf0c578e2fb2796f232bd4eb24e594373559ca', '499c8b5c58b4cce859361df6854f179566072c4426c8d9ed020749ffe7f9b50f', '4c722fa5afb35046c62ddb85504502d535cbed2fecdd9fbc1ee6b464a11a8d0a', '4d8dcb6103b5cde63d643b86d5232a092c0adb5ac64d5805cecacfa4301867a9', '4da39f2289e89580e18ece044b3df3ea9694b8cccf0583aa6f8d2f7384b71f10', '4e88410983856bf3a09e91b936db2ced9d720d09259f120d6f10011917b94fbd', '51272a4151c4e994491cdc6c2ec65d64bf3dd2d70e4c223b578531635eb11042', '513a05cf9fffd22ef77fff7686a9bf0054c06ed46ea506fde62763b867415e20', '51d5b11aca2a022fceeb7f40f853cca843acb0927dbcd1451525a6d048c6a52d', '5229daa45017552fd32669dcf4a96e45dff97671f5adcff6aa2e46f7d0140112', '531c76caf1ac8c69f1d4d07a8e17a86c20f4e45977dd676d9c267f3a1b448e74', '5416f31fef1d2f4b52d001216e07fbe78ba73b88251ae698689f9a6b218f34b9', '54679d0adb8a82eac4afde1ab7fdcde2308e1156254d8fa040d6e5236cfb6e3c', '553338da2701b87521a32ba805028a46d66ee0c6c6ad4e85c50df9cf50e167b6', '5715abe0581dba527e32f73cc3a6d92c67ba69448b31636135501012da89841b', '57c7f9893706b98cb1b263db5da76f6b7336380684efcfe9a167b77384cf5c6b', '58008f06339e96301472f4146cec2558e25e79df4d0e0aa8511d08a9fe5d409e', '5958f91d8bb273bbf211d7ab1511d885afee981e219b4943eae1ecfab3370b17', '5ae768884ba8c293738e481645b30e939930a6494ed00f2df6c1b2a7140148de', '5bb51e4ddda8bf153b6623460c5029d355d7ccd835a9ce1b63e46f29bc27dae0', '5c120f923a95c00b5a9c1f6328bbc9cda0850ab7cd9fe68c35b60bbe51ece0f3', '5cd3b15957e86d85be27e14c05c35b8b1d3a0aa9646995404117f630b18a758e', '5d542cf81e85c032a23824fb06a5954b8471f77bb7ea92fdef962002e4413a37', '5debd9bd1bcf027bd3df503b3fb73a68924fbd5b3d7c55852f379c06cb61f238', '5f09033528e4da13c2ebefd8dafc8d749293d6a12e1021a6c2ff78aa0e54d157', '6055a941f23c0adfabdf76e54581c320ad8f2315d69bee5521d5416ecaef7a6c', '6212fdd2df4135dff2a4f1e8952041ef98143dd4dbb0974f0fb172a21ca4adc5', '625023052b7d4ec21f9449e9e2983590976dca8ad4040b42961775bdd94b7e13', '634ee8ab28a8b1ddfa650054230cdcc97c970f8c51c45c719c34b0605964200e', '63ebb29d693bb899b4f8271559b08766c525fd555519f0d20d70df319dee630a', '6546982643a9a818a288b099c5f87e0c7f16a18597b104da060239b405a0313a', '6875b7f3b4696412011b80ccac0ae7b1afafab19e70a195bce1186d49e021cda', '6a68787b742f537eb20b088a5c9834bb03c28d0eb03ff671f9b8ff3f07395388', '6b379aecea6514f6a76fd867ed5812830569fd228f9779b6726999cc3d3f3ecd', '6d1f0862f5e340d61854b140619859d5941cf9835788f3e71711c839ead556b4', '749a2c0ae14ffac85a74ae6ad8f518175790c4ca1d997895ffa9b863ccb62dbb', '74e2ae59d0c07c953be1be0036dc681324bb7b350dd43db9ae09049e358003d9', '75b884dd33d394d344c21ee8e7b66421b43ea8cfa0982cbb76f78685428cd9f2', '762bee91ab3300e45c3e67c791694f5e9e40dd48676458697ecdc45d09a14569', '76ebba41843d5e5007eca1207ba93950938ac0d2c82163bcd6bbee0bdb55e861', '77cde4cee01ec4f412069aa2ae6ca3d50154e6a95587596652f4d002d83325d7', '792506db3de2185698771bcba1ab498bc7ec04e1d6d40406167cadccdd378770', '79d2ee709fe90a80392c3ea347b245853a2df6885b5a08e12695e95a5b1fa5ae', '7d1c26659b5abc30b4461304e0d55a45938a0202fb4dda162642ff6cdbc7583b', '7d33d0329a2b9d2c2784176df8e36680b9ccd16896c9677e9cc96307c4e7c5b0', '7dcf4a149ee83236fc6467851c1a992b4107d833c99d7977324bc24ed989c927', '7e74b61f3e6334fb3c0b359f71c9cd53e1982b50cd5427a94f36940db9de8fb2', '801ed1c5358365b330eebb20cb4f7480fe83a96cb240544509db85b37410deaf', '82aa3ccdf29aeaa6f8b3a1255532f8f58c550f0f343615d0ba8d0cd6b762267a', '82ba018e345e09ec588090eaa6d5030da344d6393ecbc0b091d7d6c01027403d', '832d52d7219035a23a80c986cd41c2961c1be6aa7f83300e928d9a7435d5f893', '83f8436a8ad19dc5f504b38407177ce9a761b8550b2dd78f6394676221c2bca8', '867884ca840bb4cffc94cb14784a59146ec17d21445fcbd912a76dcd56a288d9', '881e5969b0cc5ae398f845580772401efe205756f3a17d63549f23ff5c6a0a6d', '8d7c37d7b3c2ae9536a79dc73c978546bdae97f5050531989354ec29f2a7dd59', '8ec11e81552d3696e99982476a898b20ef63825e77e3cf2a7c5cc0d8411a381f', '92e0727f52430f0f457dcd108ca3ae6df86e38f153d3281f2f9d7c3c0d635805', '948c52fac43e824a898f1b322ef815d1c85ae5d2e84001d078585ad0ce53347c', '94d75d14c434f35a4cd69c76973a093a02287c2ad01ce8a01f0d6052a04e19a3', '95dcf2b60941b5910ab09a008eddc61fad56dfb03a59b3a7ab1b6f845cd0fe03', '9792b8709c6ac19d37bcb83aaca038f7c34b171f0cc3c0ad907179d70eab5fd2', '98f7731a4426d8c50ad3f9203f1e1d3bbf643aed9564e29139f29f9f0dcab26e', '9a4aee6a95b54619f7c8541a6497e69133bde39d782f18aa6b119ecdd6da0ffb', '9d7258d88bad246c31aae1fe27311016399fed9409d188ef8ec6b5971d130e8f', '9df229ed3b72c6f9ca679843b977eedc9fd4a0c7d9cba087242950483e98266f', '9ef8d947eb2796927fb9fa8926b44df99a1888cb92b40060f78ce6f532bac088', '9f74a9537358bd01beedb7ae65a5abeb9ba74c06d1cd036aa70256e3b7793029', '9f879f6d8aa9e5eb62cb3af349cfe8fc15e8eb9fc6262cb8a377688c597dd242', 'a27e8db63340cb177a85d996eba4e9fe23764fb05fb9687133ee99910a0ea34a', 'a31bcc6af1b7dc739114ef1d8037be760b99a6fec1c93efd3fb594bc06f99358', 'a35875fa300436200c4fe9b56687c01feb06f1ce504fc6d3977013643a925f37', 'a399dc2f02c8d4432dc14604458e3ab591b6205a8bad07c965efc84ffb7384a6', 'a3bd77ef4e109265d159ac450fd96b2b2bd3630fafdb57fe57fa57a0519637bf', 'a409ed9ff99ba4e15aeaa32db6bfb186a72204a376899033f62a1ba270355940', 'a55621a66ebff5ef62a96a300b1167c7331c2a9fad28396cdba67b9cf9d73455', 'a6b632bcdc8e1b77a219c6464867b4fb3ae55d453cd0fa6244e29d9a54bb80a3', 'a7b5aab179ed65fc4b0ebe2ebb28a31a8535f93fa3e1f899dd3c6f81053b8deb', 'a7e3f1b3772181ebc9dd1f14028659b654087640535343159281822e5644994f', 'a8f21e01fe3828636c0a50e126554627ca10371815e24d99d2b167ac6f896cdf', 'a990fc5b32d6958f31feeecf0357c7e79ec463f418f1a341ece481314f7d1f55', 'aa5563a85b650cd318402efed4f90bdaaf1d040a6603f1af4fecd17bb1a3d2c3', 'aa9fede7e598760298a43ddc1ea9aee4b4b4e6efd6c716f84196cb293abe8c09', 'ae2015c7a2ca470b32af4321f30eb839dfc3ef015fcbc19e59092fa11c1f13a8', 'af2bbfaa1cdd9aec1d5eef54f195b70122c3039bcad509ecc78233ffd5e52f6a', 'b310a4a603a4b7b49c4f73fd15cf1a8500625a65fc9509547ef0cfd5b13a3678', 'b73baa26e681c985f3b9798957e03bde2b4e573896b3904287a88c80a2b93b3e', 'ba31ae258a567cd94531ef9009dcaa08622bed1722bc458457659f9dbcbeb408', 'bc7eb8de8925187e8cec597e27caa7eccc6464027220e47cfaeea9c88c477c5c', 'bf025830ab71f7afeb66ca9a28e27e456a5928119ea2c18d00bcc6656170cd0f', 'bf4819ae63911316798b91d7663f6a48c686930e1517305b307da44a68f8da26', 'c0f77f115e4502dd098e37f9bbeb7cc2020665e14c5ee461e2ce4b515eeec76d', 'c1031d72142bb2a637870b199a826740a1a219249aa1f913ec28080cf3fc7f32', 'c1687df1cafab395dc4667d979a03d93ece90a53af90e000ed55918c1bf6ea82', 'c4da157f7c39aafae8c0661eb062391d442442202a3f8b740dc0e92cd6940299', 'c7114b8b0457f5d66da3f5cfdaefc188b300cc66e270768a48d0df9fca9ddae4', 'c7e618454af33dd23340e8ec92d344dd3427cba435323ecd7e853c14022297c8', 'ca04b5d93626b9c802db94d584ea872d2d4d8292ec5f082c90818ac6fbab5c8b', 'd093be4212c184efbb87daead69bb0b55d0da6e84325eac9f33e65658a1d7f8b', 'd236b796c8d65f93d3df173d26ea568d7bc794d45f97a75933729dcb0163b004', 'd277aea55028d6ecf6362b67a7c62d75ddeebcb6fb19c700d0ddf01f58be84a3', 'd2ccef9b3bd71cbe5ef6ed83c740730ceba96d16cde0fd060b34c59c74b347c9', 'd34727afb4ffd8d7c3f812156cfcb52b7dfb0498b7e5ffa3247a59fdaab40e44', 'd4dc0e275bfbadf5bf612b69fb6c9fc661fb338c7cc6ddd5a47d49b05ea85916', 'd50f39bb907d192d963ee4f7308e890ea35a55412647b6a515f491c5cf251142', 'd7cca1a9ed20b8edf8b50e8882cf0a1ae705349957bff78211dbd08f4bf5e192', 'd969db429415ebe59b6d8d77857262ef3a03a1746e0df78b22d08160ff6c1e89', 'db76760e12ebdbb1902cec332f7bdcf24dafe33fe8eba0bf7ab0a2b05006d434', 'dc2d61864f7af14f4a8eebf7898483f1acc9e5b2f7e4ca2da3caaad7679e61a8', 'dc73c114a23d9a276a30159b29644b60fc23d7e8b993dc60033a4b19897cf786', 'de929ec280ef96561b8735c5ae9494f057cb9de0720c5f0972708c7cc1fc70cc', 'e14457b5749ec675fcc92e2ee2dee0898f73fea0aa30562934ce397428435a1f', 'e2f65ef69904f43aa56eba11f5b03451df30d446cc9e0e1132c74fafe4d9032f', 'e3164b923baab1fbca4ab356b80fb9e1a9daa944bfd78d9268b11ae8434020ce', 'e32f5f34729d144d028482e251ee067bfe6535a534dca4354e6bd5cad54b7c83', 'e345ec6f23f6de8abc8194790b24b86bf456626076e1deebb013f4bb9ea863e0', 'e574bc98610b37e4c4b4422efdfb3666db6f72bb1b67388ee655b163cb962ca1', 'e69d0876a300db1e031d0cb0a01f3bb0e63c01276176fcd5a2cdd279c07af807', 'e6b92e3fc05a80b270909b215ecb8fe6236c4da2df3d93ce225675115c19ea84', 'e8e1d414971b5b6d696f83ddd2603a7021c07a3557c226684628ddb312569fef', 'eb024b1f260cad8cc06aa53080d3c878d461a76e712237d83e234b46a072d00d', 'eb1ca8d5f10b62163c9ed65701d2bd1e67bff9125c8b7f0bcaf5c31f1f02ac3e', 'ee7bf71097a5622c351f08abad14c11a9adec0b358c858c83302ee7d4b2fef03', 'eec49884333a7884063392202575e3f3342aa6a4ae4c0d6a9b93db5db0a64b9a', 'f06805289345f5e8b24c25507edf4d45316d9775aaddd18c34a0febf3277ca1a', 'f0b363599559a9bbf9ff2b973f655f010c1478c84d74b6fe70d388224710df09', 'f1527fec7f84aa65818f46672a38a39617ae498e62ec741c96f2a16d6a9d9d2f', 'f58ec063bab0102aa2f7d27e3b13d34b0e31e5a60af64d31776e23e1fb29e376', 'f8126fb31bd1ecea2b84152f6674f5b437e561102e63d6f4ceb03100434d7772', 'f927560f9717e9bfc3db93530abc7e8654444162c84677bdf547d4018c0d3670', 'fae6ebd585ac5b0f558231478ad79fe19dd4c58faa94beb2cb1ebfa002dce89d', 'fb2e3ba905df1ae445ec49e8177940a17407e04bd2059a64715740039fc92d4c', 'fb4aedce8f218b1e0bca705da987cbec8072744ba1b1ffa055b0f34bb0565b68', 'fb52cd526ff3768381eda8772955dd139630b1c92448d3ecedd800ecf14eec90', 'fc93a2eb65773a02a1711593d82795c65528f9212f025e48f79e6f9eddfdfa54'], 'selected_file_sha256': ['0084f8c888def7ca8e5150edd21a7248b1feb0ba3ef9e1d0b3c988f46501f818', '02cfc775aafdb1fdd0382189d3cb813f2c890a1fb6e176d907fda86ab905a684', '031dba86bd4ffa1a1d51960b39da70c5ce50b3edf183b70dda9d3218f905187b', '05c79664ff9424c0b3c30f9ec2ab084687d490fd6d0f4948b2e51176f923acab', '05e6ba245ddd2e59a3e021823ef935ebf49f6a273dcc3f2679902f74fa31940b', '09143054231e9df0f34551bc20ea40775fed5b60e64f6f9c7292ec04e772ca59', '0999f9393820e8917812ddd25413ba91f6f152fbd1220acb379a7a5474810d24', '0a91dcc15c839e62816387cb8262b74a81f9143cc5baec723099d777c9469768', '0b7dffb63da06365e02956ee0c0520dc5ff8812e339270410d5b9364f1a9eb6b', '0df6c0ecf3cb3f92cfdd8940843309292f5f8fd9d60dec27cf56b196d43827ec', '134b0136fbc951cc55e75a08a9335a524835aecbed8acf91332a655fc1af8c97', '136d362b7f3df7ad18a07b0778c9478e0efee3157baabcae0afea5da412fc9d6', '15747090059d549da3c39e1f08c4928adca0378b0a00559922556cd3d0e64b45', '167b3d90ebbffc9837a5cee597b72b178c524652090642a58c0286621737dbb9', '191224e97d30f7ea88221396753fe3b179640d179f1248addbbd5e1ecafcdd75', '1a4b3599f15372469347e6a3ba262d663076626d90890c31417d50e36a856149', '1b902ece7dd95344dc5b01077ecce2550ab8b0535d5a0b885ae4d2db86964f3d', '1d65cbf7f778fc51185fd3d4a883e44d60ae3df2d0333b55e31d90664e9889fa', '1f2b852bde28f71a1e9334e223dc81d717addd893e7870a5295fc78d33df212b', '20aaedee6d4486a09ff247ad24d80a562f0be2a338e35bdf8f82740d1255b0cf', '20ff7ce26d56fdbb1f57239cc7f13d9c7187a2fe76d8d41fc4040cc67b793228', '21a1ca36770cac71ae26295310db18513925786dbf6fdc9c8e4ed8f99dd50154', '22790101b93cb7b9e5f4136c6ac0ad559271f5d3130e1e488e863ce1ad5ec16e', '228c1a35ffdf65cb9a368379ed028cbe98e0968d4a827522c6d7ccf09f574eb0', '22b050d12e7582bd0b6b6a39bb5cdfc3b5b23ce3b376b88d5871877ec6b4b5d0', '2420281d08d9fd14717298d71717302f35a9299601607e41b548b37011e86c9f', '242f9140b8650ed6065382d89b4df19bbc281fb8388f7e76b096daa17c6efe01', '24f79f0a5e3b82171a9702b32f698668549ec806f5df4eae2f2eec402caed843', '25f4bdb171cb66d609c3b41e5af6d9fe6c4b43a20478fa8817e4204ab33634e6', '260c6068d56fddfecdaa367fbf870801776df37aaa2d954a6a0ddab092769a0c', '263a7ec3ee646b32ca436cae9a4baeddeeeb3cff8b98c67aac39a8b099a6f7b9', '27fd2d7e1e17bef7a17b67a751f6e48dd6d58c870df93fe72935a67224a9cf8f', '2d90ad29da4cc1144a890a98c668f275336330d6a7b6ec3aebe8ea0b71b0c26c', '2ef5e210eadc3638aecb896f3c3e906c284585ff0a1d1476b94ad3333feaf314', '2fb3107b17ebb27324d93eaea065b85dc2f4538c305e8f1045df56fb583f7909', '31d7041f2573fe087dd00a9faa0f6c372fb958bea8b15a956af218977e933f4d', '31f704c359ea51a9f5f599c7f331d0656792ba4f3fd6417cd35bd970d19c608b', '321eabc5858dd5f95463c798407ba16a9a26459943269cdf0228ba025f20825f', '322bb75b4f5a05b0eba7ba80b796474b81e5f47951d18342e2b9223b137a2c46', '32955221a7888fae9aa1ba5a7bab8d6ff03e326ae972c3594cf4e49a1988334b', '34e30aa6fed35f75372c32706537f0c8053da8706567a99639742d36e23b297c', '35eff67e26cac12bc12c75cdaee75d76b3dddb27ea3b4a0eb259b300addf3633', '3684e33c3883dab27bbab61eaec02a028c332b10416c3668810ff80593273227', '36984c7cebec9af59bdd0ced39e22c29d5208c02346001388601a989adfbd098', '36df2fa4a13411acf35dbfae062a15ca58c58331e52a20019fffb17c3ebbe8a7', '3983a97cb60d76efcdbb1b037e8b03a770cba61d413d2dfc58fd1017207d8635', '3a52e5716470b7ee0a695cb522e1d38858bba011de37e93a3f31bac90874f3e3', '3b2c492954627e8bc310bc32893d825da555b631c34c5acaf633ca62e8d5a401', '3fca60957977d0994804c0158199755aff91032f2c9c91db55154255ba36eb8a', '3fdcbc48a62c2fcff67e436a6bdf7aae0a40d277694895c7e80144ba41a8ebe4', '41bb5562b8edca1028ff34db563dd1d43445aeed5b9e1d080bbe63b0c4e657e9', '445ecfb409bc55ea2edbf185a3301d54bb48978c552d66da81ab5f654ab3ba26', '455a8e3207deaab6af6caa2a85c4577aaebd1b494f6e5d2fb47ce1f7471a0347', '455fbff1af91c14371e3ae707558819cf97c55f50e08f1273ff17c3e8352bb55', '45fb36f975631ab002fef0cf6cbb878ecc1ee949eba5519c8dd53f70aec6cbdf', '47b2eb51a1f560d0be0833c6d040a97d523054c16c052d0c6d0d8500f7be5507', '481d4d1b796c64e8d9447a128154774a0e68d3168d06751c942704ef18e2b9e7', '48644c96046d9bfb85294d1df49ac65c615515b59c6ea7af721b2f0b2a88933a', '486b18e3bd5b423384fe51027e0cdb1384e827ae1045c8a5b33626ab98d4e25c', '4909caa91644f73e0c459679143722235a1aef1fc58621cce427861966c6a28d', '494177b028681bf52c409f064078f954db33534a72e96370c1a7bfa11429cc39', '49b38edf9a960824ab73270077ed5e9d52aea300ffe454336d35daf0dd90f812', '49d7aed962c2ed37ab2c75bc32a946b70f07ac1d8fe56a91c15e1d4822dadf34', '4b656b4f43b3de43565b38e1f5a013193ae8fbceb330505aa30f46e0c7d266aa', '4c0dc526b6db4afeace727eba01436f121fa40d20c366e48efbf98f46f50fda0', '4d135c5e1622a00434e847ceda99c2a093a2127712ad3ab51594e18616bdc798', '50c36b1dc6fe948923c1653b715c4806ebc86f061b2ab48f0e95c42eec544b94', '521642452f15e3c142bd07fbcd015e9cb141fd29f947b584fcff64f9987e07fc', '536ea5cfa9c5e967fc3134241720847df974adf0d33ba3a7607fa577f2fe7ad3', '54792b4b4041509792612a46a3a9e3c61cf7db058cfd3ce54df9bfba958658b1', '54bebe36f250ef417216594b1cef00bb5594aee3ea36f3df54c0f18e540a8ada', '55dcb7e2220bfa7fc6500252b8db0113bd2ef50b15d43a583f032d87825dd791', '5647ff3c18cadc44baa80528bf6aaa790f05a1b0d67d96ac4fd99112c5c42238', '583a382cc04cc2c8b0db2a06c108994f8b5d0f14162db5b9630da75f6764fbf7', '589ea0e14d751b5ecefc3ed53d7685f72fcbf1caa3ad6b616129d3797702235e', '58d4d35e699fe59f3d5e0d147fb2702d37c4c02a817c3d5c99d5fb54986c0504', '5a6a952153dd90401ec40eceda54ab9c37709676eaa21f8ad83503a7b24b1951', '5ae385ed224de1f7172cd56c30ab55188becbe0a9bb7dcbaa8b0a247dd73845c', '5d83c7a55d1684799a32354258b06da03937744a04ef843d47c73fae116151ca', '5e30e31cd36bfe003c71b8d0e7675c5dc82c4f70f637963a4540c744f8466c64', '5ebce6409062d0d09c88ecd8b8873a68e3bf252d4d43f896a8bb2ff06b767a80', '5f77be7c9b614908c147be524eb802b464eefb1fd8b08bca03f286e1842d057b', '600043039e4262ce95e81c82a970c426f132bc413e97c0c6b73afc992e08c52c', '602b7b7180d02c143d7d1364110a2e96113a6dfeb70d6c2300e114afa004f47a', '605195b4df6d5c9b8694a313337bd3b8a2ef4759c1813a638bf967c728f7f3d7', '61357eca6b6068fb3a47597031319d933d97893076c86d3c9ab346d913e1b600', '6197f56fd48512c651dc9d17532a2f08afb691c1b0902e02212680d496f73683', '630da7ce1bfde039240eb9b37b0f3546ac9d2a24eb724f33dd7cad0eacd0fab3', '648b75001ebf872b35bbee5d498a94ec5d35c0086b8e03981ecb45f3b1f0c257', '6494be55906ce443273f21d6829cb2edbf38a73933f0b5299c1d1e0c5a43f887', '66886428bcd081bb7db43e6220dcbdd0c02cd99015c8a102cdceaaf7b8bc2236', '66a658465049aa95d0105690b8d79d9eb5f556bb97aa58e4fb0aab3c264b094a', '671226cea6820bb36a38267e9eea5591ae1a7f1c351b57a59d8cb326af7701c5', '678ef7d21f94b80531d7e67eac5bc0c63f7387cfc8fd0f32f4aa3bf0a060e854', '67bd27e0e0202fc0f7c911f71233c63c2191df086d0c5fe99cec8cea9e1ae46f', '680a0673b02d0e1157d321f8597d09419f1d4ddcae44d515eb310b718621fdb5', '6878cbee5f711d1b6b67cba9f3258f126e269d62e795e6a43e7641e0683f903b', '68fea67ec0719386b07199fa625761fde0926e75a6b84fe1cc84455c2d03a5f9', '6b3b65b93269d46972a31f9a62d88575f7215c594fbb51c9701bfcd3f9781b28', '6c6b71c394797b942e87cd7037a729547e0baa1a83bf9bed818f70200f1d78b9', '6ce7d2bad658bb48845625e87136ad74422072322416f430039151fc498f2b43', '71284f9ad2bf5619ed99d70f89839a85092eb67bb45c9e8da52eebf958ee49e3', '7524c3ae8661a7c7d361b8149056d31acb09e9e31ec5fc2f5a4e62889552e3ef', '76cd5408e575953f7162c493f1c2b30773be0d4ac843bedeb60267391603d23b', '79f7031a4e43a6c097ac52e3cd3c2f2e0902c7f4992259092c855dc591712ea1', '7a2391d2a2a38dbef609e990a824b07332bff012adb280dce711c019b0636fa9', '7a935559ec33c860f6fc4eb3d7fb5f1488cefc60d5412f377562826633a7bd00', '7aefcb954cfe40eb083fb2f35e0b334ef45b43c2ffffc25c825a0b6a3df1d839', '7bcdd0f5faf5011f4a12267f0968b33edd53a8a6b496eb9896cf41e73155c450', '83e5a8ed400d291c17f48e8b432b8633aaa0de3c3b482808552a7928ac3c6a62', '855bbb0ca119676f9c9135fa70d90547f606e5ae78faa5b0d8a463097f4b9ce6', '85d25a5325a781e3e0d276e47a4cfbced0e428e63ffcda50f2654ab2be699c3d', '87c9fa95545393b913efee2667d28f4fe46e23a08315602be8a4f3052dd1a729', '8865258e4cb393d1cd57ea265936df75da7fc46aa91e6ba16bb21e0d9d16a6a4', '8a090e223e0ea8b594b339816bdb6205b4c46c401be9a5b937f72ae965652299', '8ba078f1f9220437c84953f3491b2fa59b7caf3bb6f4fff2861e67669ed00bc3', '8d47b475097e840bf895f5ecaf19c05df56d90d47b21cd858f090e1c677ff29e', '8e9d5716cdb8b34a4e88bbc0bfd89a7239b26dc28a5677cf1bc892ae513b9a85', '8fa574972578872d1b818a67c4dfa1fd585e66f8c605023fb0d81075abd81339', '900deae74c7de1a6635f99bc11278ddd09d98078b7df8dd0839103d51ab0ad97', '90f28dcd3b0705cde9bf71db1eb36d1471be78d7b5c64a61c77db2c9c006b5cf', '923016553382e157b61c598820d60ea4e1f86839523968396be794ac69231567', '96c1db9596316e1c7a4f0d7000782a35c749292993a6eda9c180eb0ba65ee73a', '9746e86839c203f87344ae97c67204004eb99c35a04074a2b3a76e1c62600798', '976a330772d02cedab2f0d183f7f793f93117ab62e2f6efd7c25eee1634a4265', '97a2b95067c1c51e4bfe50d4a19eaaa18758f3e5b92080b19eac2d429dfd4d51', '982bbfff8cdc53dd939dd43265f4c5c8787ba2dc52b5e6ec37ad757c69684554', '986f5977972ec1fc0879556c5cbe75e1b871c7f5aff819996003d6c8552d53bc', '9cbabc345948633cb689c7409dea95d1416fd82141725b327d701d2e5e2fdf4d', '9e13fc095ac630e20c2148357a9042d615fbd3c8d9a986a70dcf85d193f95825', 'a08b6a901fca192459f24e64173f78247a955a404f6b54d706ff8c71a4bb0660', 'a20efed7426e1a5e2ac2337fce00bcbdd8259d7b3444df7c7f14368f8579c69a', 'a232c63616c5d104835a81c72c440af4a185f1929da16c473bd7c4f07be9bb12', 'a3db4522f6a2f4ab697c3d108848d708113fc8ef5114ae564d5fb214dc304d5f', 'a4faa72f0286e0f7e7ebc4d7c43cf6e553eb0414eb83ac5618d0c1b22a0037e0', 'a52c26a9be9cd50a57d2516e4c157704a2a80ccd21337649d4b638870c9f09bb', 'a54662435c1d76fa7e591b18593a38fb653a4f807158a3fcf72625a4e4679f7a', 'a6914c89e4c89af82929f0126a26d08e2297f650a8ceec97e0cf000a9fb65d4a', 'a7a207334af3649a356e6a9b2a05c05300eef762aa73627e9773d329704051f4', 'a976e717049353a5df7197df29909477b5f31d37d9890a43f066280588e6acdb', 'aa0b920e1fbabc76a620c87fbd5218f9de182d66e2e9a2f07fd3fdd3e85a25c4', 'ac52365e2ff8a12563648794f6cda371e1815d371f05cb65ccd3ccbf49a4b2f9', 'ae21b433cce10924ab235527fd783b53a603cf09cd3668ce414262a72696c815', 'b4ebac3adf1ea2f7036580c7fc75985e13ca189a8d4c48aad11c0652679ebe17', 'b4fe52fb4349d23096a5f30437abfdf6361e281115f68bb5bdcb8eba56e4843d', 'b559c50b6ea318bd89bfa111e302cebfd84550633082ce995cc0e5fd3504c01e', 'b5c5e3230703e8fb02da5d7a7f5283b68c489585fdcfe672c5dfe12ebb3ae2aa', 'b630a2132166a89323db96d7791f8553ef92f42d09e771c252a88b0772851576', 'b75ef8e8cd44faf109ddc477d87262566dbfb69287947f859ae0c6344903eafa', 'b7694a0d8cc90a321d708c46804f1185bab74a1070aac880453dfe932e9e0ccb', 'b924bbb286ab652117d29e05849c1e13e9542867030de7b050ff1fa2e444c25b', 'b98e94478d3a1a2dc968b22b8514b183b7c73f69203366248338e98f11a5f8b1', 'bbb4d6e5741265bc87dd1898ba80c49989f97854f4c489fff088cb624166fd54', 'bc63ed066c67c28bb0d812fdfddb8eeb4989677760561b2c23398b8409141054', 'bc7069da33611ec30ff0a67d51895a7136093053d6ada339bc5426bb331bd3e5', 'bcff4ce05719705fbc30bf7af94ea1b2867134ba610f2b7b5b2cc6e97eb1b898', 'bf3a56b486e0a3b54c875a4f7071b8c72026de84fba0370eee6d73f8b6083d0b', 'bf72e50e77b84cccb3ee457ba4b78f0bd2350a15a6ecd7cafb6d4c2f3681e86e', 'c038d74af6c548476bf889955b5fe53e678ef03ae1d3822901579d15c2a38b2f', 'c24cb1b2a747e2b34fb0ee2d492898363a771dbaa738ce1dea97dff66823bb89', 'c36125e5ff9bd3a281dfa03c22d11538231fc373668feafdbbd63c1d451cba58', 'c6fa3622ed69437a80d1f23a0ba24d079d70cb17990dd72d39e35ca8eb714083', 'c8ceaa52532b8a203d78a98f3075d7ea83ddac86d83c238d68730dc7ad2243ab', 'c8edcc3f7184a8c812ec255b2fdce5450699632b4afc44ffa3632eaba984bde1', 'cc359600371abde4485f388b87616f589f5228e0cca128ad427a32c13189bb6b', 'cc4da25beebcee8043a745438aca9d5cb7db726eedb373206f0c44b2ae2de1e6', 'cd66b977fb9dad411fbf3b2830e3ad4e73f89b7c3aabc481d8c76ee85d964e2a', 'ceae795083b4bd31c52fc6c6915c6f71f8840b57dd285ede753d21757bd84211', 'cf0311467347d93d724f8a782c72a9907ca9af718d13075661e471540e30194b', 'cf513ce58259ab0ba6d6d363410cb90a78a40a33dc0086200a36fa3c4dc9324d', 'd27372a7c7e6ca823331bedee20131e5cb477e6ed886217bb00bbb690e6caa62', 'd2bafcbead55ac01b0678608926f787942d8c356fdae79b2a3930cff44890b70', 'd7c4e92cd959583908e7f3b211572c3caa69be2bfdf770ecad9dcc4f7b2ed71f', 'd8f11414f5edbc58a8375e336d8d54693a1a72ce99b4752494d29ae0971de174', 'd9e16bba25112e60241dc6610d0ecf9a5cbad1bf36cb818059e9c0cf53a280bb', 'da036e121151a78ab1fe4a4dca346f897200d65598f8370ac943ddb8805ec0c1', 'dc2dfc2cd6c6ad7a84d41976d2785c76558136266719687a319cda903c5fb043', 'dcfaf7d2bb65afd16465bbffdd36a52c415e9dca45dcb8e0cbe0ba7d640ccb33', 'dde736197952b12f8c239f70f050a129e2afc81d37fa389547f3470dd790bd2e', 'e14b2a14b3bbac0f69e14e44c4d090c53b2ef144db2c101bcc0b270c4e2c6d14', 'e415eee54ef7c2e9c9a7cc535aee24aac6693b5434dd5f06b0622341561e9457', 'e77a6e19909d4ec2bbfce3357fd7518db1042b0d5fc947db9e5275db886b545c', 'e8272dfb2f764870274fd990d606a6403450a302ee0206e224afcc5102b9a003', 'ea1680874df1971470cb0a4155aa99d92d501c8eaf27010f2ca17c2904fd2045', 'ec0c2972c0b16806d604aadc9317cbadb056f7d6c5045396629d20783c1392c5', 'ee4cec3bda0a56d6226781d4402b8f91886d09f18e107e3c893b6a6e499e4c69', 'efa84ef539854106d893867c732a9253f0dd32b89e3170363ee0c9977d7afd0b', 'f06533715e8df31c80ea9db678f2da295132190f46f7c80eaf288b5fe9bbde2e', 'f1c35a4bd9fb0eb3f9bb3e97d80846c5432561acb6d38688311d597f7d6c7c4b', 'f239b953914ed24b32b8b0986cdb15de521f70fd05bdd13964edbb30ae30bc3e', 'f51b99c78324c461e23f3c78216b5bf26e6467fe4aba1a350a5fe17494e9c479', 'f51d6d9efded51863601fa04eadd76433b8cc96e9328798d1045365937e53c21', 'f59b75a7b0b397a618686b749b51232ba37dadfa465451c66c413177a2ac29de', 'f78440d114a63869ee60b76aa3fd52069d74fcc5556dae9f7e362266a8cf029a', 'f897e5a9a258dcfbe1874c2d011be02e0cdf9773fc8ace2c509ffc5b1d56b732', 'fa0521049456a3323e3e3c5cd39274a4e6e3a9622f3bf4e9131f3245c58b61ed', 'fa7da05a1023ec96754bcb456a7bb4893f96cc7bb3f62c08c8c84a9d0c88881c', 'fb48aec8cacb40a3851e8157d5ee9a8110888306f73c931290b897ba8607a55f', 'fd91f378d07178bc4ab2ffdae4702a328f8d3e026d9d7f802109abf05df933b5', 'ffeaf0b81a670d518edcd623dde67ed4a34d40754377a0bb6b3c56a4b382d664'], 'training_exact_file_audit': {'status': 'verified_no_overlap', 'scope': 'exact file SHA-256 only', 'training_images': 1000, 'training_zip_sha256': 'ba0475b0baa350052285765150205adf8c37285d3541781c797e95ce2c03480a', 'training_inventory_sha256': 'e6f6f7a1eb188bda55a25ed718658f4c7e4429c33b1851a3720b50f1ab53b41e', 'all_test_name_overlap_count': 0, 'all_test_file_sha256_overlap_count': 0, 'selected_file_sha256_overlap_count': 0, 'checkpoint_training_provenance': 'not_established_by_this_file_audit'}, 'independence_scope': 'exact files only; geographic/near-image independence not established', 'selection_policy': 'one frozen result-blind selection; no result-based reselection', 'manifest_sha256': '5770c2a556cc6f27b5b47d55576c14f7f5934912a3e117d08260964bcf3060d9'}
EXPERIMENT_LOCK = {'version': 'fso_sc_experiment_lock_v1', 'reference_model_sha256': '89024515a0dddedf9bf83f0160dea358c4d9d481e7dab7a84be0c43d5e656366', 'development_run_id': 'e23292642de7f418', 'development_summary_sha256': 'e24e6afca12576b8a87438b9be1e4572e56151b9729ead361024b58d83752933', 'holdout_manifest_sha256': '5770c2a556cc6f27b5b47d55576c14f7f5934912a3e117d08260964bcf3060d9', 'settings': {'conditions': [{'id': '187d35153f63f3d8', 'group': 'power_at_zenith30', 'channel_kwargs': {'launch_power_dbm': 30.0, 'modulation': 'QPSK', 'zenith_angle_deg': 30.0, 'pointing_error_urad': 20.0, 'background_light_radiance_w_m2_sr_um': 0.0001, 'weather': 'clear', 'scattered_power_ratio': 0.95}}, {'id': 'dbe804001cfb8471', 'group': 'power_at_zenith30', 'channel_kwargs': {'launch_power_dbm': 30.0, 'modulation': '8QAM', 'zenith_angle_deg': 30.0, 'pointing_error_urad': 20.0, 'background_light_radiance_w_m2_sr_um': 0.0001, 'weather': 'clear', 'scattered_power_ratio': 0.95}}, {'id': 'd9dc16d457e74d56', 'group': 'power_at_zenith30', 'channel_kwargs': {'launch_power_dbm': 30.0, 'modulation': '16QAM', 'zenith_angle_deg': 30.0, 'pointing_error_urad': 20.0, 'background_light_radiance_w_m2_sr_um': 0.0001, 'weather': 'clear', 'scattered_power_ratio': 0.95}}, {'id': '7ef47e33ee6592f9', 'group': 'power_at_zenith30', 'channel_kwargs': {'launch_power_dbm': 33.0, 'modulation': 'QPSK', 'zenith_angle_deg': 30.0, 'pointing_error_urad': 20.0, 'background_light_radiance_w_m2_sr_um': 0.0001, 'weather': 'clear', 'scattered_power_ratio': 0.95}}, {'id': 'a9683ee5da014188', 'group': 'power_at_zenith30', 'channel_kwargs': {'launch_power_dbm': 33.0, 'modulation': '8QAM', 'zenith_angle_deg': 30.0, 'pointing_error_urad': 20.0, 'background_light_radiance_w_m2_sr_um': 0.0001, 'weather': 'clear', 'scattered_power_ratio': 0.95}}, {'id': '4463a00c9ad3eacd', 'group': 'power_at_zenith30', 'channel_kwargs': {'launch_power_dbm': 33.0, 'modulation': '16QAM', 'zenith_angle_deg': 30.0, 'pointing_error_urad': 20.0, 'background_light_radiance_w_m2_sr_um': 0.0001, 'weather': 'clear', 'scattered_power_ratio': 0.95}}, {'id': '520e87d1ca3fbf26', 'group': 'power_at_zenith30', 'channel_kwargs': {'launch_power_dbm': 36.0, 'modulation': 'QPSK', 'zenith_angle_deg': 30.0, 'pointing_error_urad': 20.0, 'background_light_radiance_w_m2_sr_um': 0.0001, 'weather': 'clear', 'scattered_power_ratio': 0.95}}, {'id': '5d3814cf100dd163', 'group': 'power_at_zenith30', 'channel_kwargs': {'launch_power_dbm': 36.0, 'modulation': '8QAM', 'zenith_angle_deg': 30.0, 'pointing_error_urad': 20.0, 'background_light_radiance_w_m2_sr_um': 0.0001, 'weather': 'clear', 'scattered_power_ratio': 0.95}}, {'id': '701fa3c906542180', 'group': 'power_at_zenith30', 'channel_kwargs': {'launch_power_dbm': 36.0, 'modulation': '16QAM', 'zenith_angle_deg': 30.0, 'pointing_error_urad': 20.0, 'background_light_radiance_w_m2_sr_um': 0.0001, 'weather': 'clear', 'scattered_power_ratio': 0.95}}, {'id': '35f561aa79c51f74', 'group': 'coverage_at_36dBm', 'channel_kwargs': {'launch_power_dbm': 36.0, 'modulation': 'QPSK', 'zenith_angle_deg': 60.0, 'pointing_error_urad': 20.0, 'background_light_radiance_w_m2_sr_um': 0.0001, 'weather': 'clear', 'scattered_power_ratio': 0.95}}, {'id': '5c1bbbbc4706ef63', 'group': 'coverage_at_36dBm', 'channel_kwargs': {'launch_power_dbm': 36.0, 'modulation': '8QAM', 'zenith_angle_deg': 60.0, 'pointing_error_urad': 20.0, 'background_light_radiance_w_m2_sr_um': 0.0001, 'weather': 'clear', 'scattered_power_ratio': 0.95}}, {'id': '0b62f427251592d0', 'group': 'coverage_at_36dBm', 'channel_kwargs': {'launch_power_dbm': 36.0, 'modulation': '16QAM', 'zenith_angle_deg': 60.0, 'pointing_error_urad': 20.0, 'background_light_radiance_w_m2_sr_um': 0.0001, 'weather': 'clear', 'scattered_power_ratio': 0.95}}, {'id': 'f40418d2e2b95437', 'group': 'adverse_weather', 'channel_kwargs': {'launch_power_dbm': 36.0, 'modulation': 'QPSK', 'zenith_angle_deg': 0.0, 'pointing_error_urad': 15.0, 'background_light_radiance_w_m2_sr_um': 0.0001, 'weather': 'snow', 'scattered_power_ratio': 0.95}}, {'id': 'a2696c7b141f7b90', 'group': 'adverse_weather', 'channel_kwargs': {'launch_power_dbm': 36.0, 'modulation': 'QPSK', 'zenith_angle_deg': 0.0, 'pointing_error_urad': 15.0, 'background_light_radiance_w_m2_sr_um': 0.0001, 'weather': 'fog', 'scattered_power_ratio': 0.95}}], 'profiles': {'paper_eq3_4_background_only': {'profile': 'paper_eq3_4_background_only', 'noise_flags': {'include_background_noise': True, 'include_ase_noise': False, 'include_shot_noise': False}, 'basis': 'Chen et al. Eq. (3) defines n(t) as background-light noise; Eq. (4) gives its collected optical power. No receiver shot or ASE term is printed in those equations.', 'diagnostic_only': False, 'full_vpi_reproduction': False, 'physical_superiority_claim': False, 'curve_fitted': False, 'limitations': ['Eq. (4) optical background power to complex matched-filter variance is an implementation approximation, not a disclosed receiver transfer.', 'FOV unit, optical filter width, Malaga phase/beta, fading coherence and proprietary coherent receiver/DSP details remain assumptions.', 'Background-only omission of quantum shot noise is an equation-level idealization, not a physically complete coherent receiver.', 'ASE bandwidth/post-EDFA launch reference and shot bandwidth/polarization are inferred; neither establishes the undisclosed VPI configuration.'], 'provenance': {'paper': 'https://ieeexplore.ieee.org/document/10969992/', 'published_channel': 'Chen et al., Sec. II-B, Eqs. (3)-(4); Table I', 'v6_shot_reference': 'https://doi.org/10.1103/PhysRevA.77.032116', 'selection_rule': 'Choose explicitly before examining evaluation outcomes.'}}, 'coherent_receiver_extension_v6': {'profile': 'coherent_receiver_extension_v6', 'noise_flags': {'include_background_noise': True, 'include_ase_noise': True, 'include_shot_noise': True}, 'basis': 'Unchanged v6 interpretation: background plus propagated transmitter EDFA ASE and input-referred single-polarization heterodyne shot noise.', 'diagnostic_only': False, 'full_vpi_reproduction': False, 'physical_superiority_claim': False, 'curve_fitted': False, 'limitations': ['Eq. (4) optical background power to complex matched-filter variance is an implementation approximation, not a disclosed receiver transfer.', 'FOV unit, optical filter width, Malaga phase/beta, fading coherence and proprietary coherent receiver/DSP details remain assumptions.', 'Background-only omission of quantum shot noise is an equation-level idealization, not a physically complete coherent receiver.', 'ASE bandwidth/post-EDFA launch reference and shot bandwidth/polarization are inferred; neither establishes the undisclosed VPI configuration.'], 'provenance': {'paper': 'https://ieeexplore.ieee.org/document/10969992/', 'published_channel': 'Chen et al., Sec. II-B, Eqs. (3)-(4); Table I', 'v6_shot_reference': 'https://doi.org/10.1103/PhysRevA.77.032116', 'selection_rule': 'Choose explicitly before examining evaluation outcomes.'}}}, 'profile_configs': {'paper_eq3_4_background_only': {'wavelength_m': 1.55e-06, 'satellite_altitude_m': 620000.0, 'receiver_altitude_m': 0.0, 'earth_radius_m': 6371000.0, 'earth_gravitational_parameter': 398600441800000.0, 'light_speed_m_s': 299792458.0, 'planck_constant_j_s': 6.62607015e-34, 'electron_charge_c': 1.602176634e-19, 'beam_radius_m': 0.05, 'receiver_aperture_m': 0.318, 'rms_wind_speed_m_s': 21.0, 'ground_cn2_m_neg_two_thirds': 1.7e-14, 'atmospheric_range_m': 20000.0, 'fog_range_m': 8000.0, 'field_of_view_rad': 0.1, 'optical_filter_bandwidth_um': 0.001, 'symbol_rate_baud': 28000000000.0, 'amplifier_gain_db': 30.0, 'amplifier_noise_figure_db': 4.0, 'lo_linewidth_hz': 1000.0, 'photodiode_responsivity_a_w': 0.85, 'doppler_residual_ratio': 0.1, 'malaga_omega': 0.5, 'malaga_b0': 0.25, 'malaga_phase_difference_rad': 1.5707963267948966, 'malaga_beta': None, 'max_derived_beta': 1000, 'beta_fit_policy': 'nearest_warn', 'beta_fit_relative_tolerance': 0.001, 'integration_points': 4097, 'integration_grid_power': 2.0, 'num_receivers': 1, 'combining': 'MRC', 'include_turbulence': True, 'include_background_noise': True, 'include_ase_noise': False, 'include_shot_noise': False, 'heterodyne_shot_noise_convention': 1.0, 'include_doppler': True, 'include_phase_noise': True, 'launch_power_dbm': 33.0, 'zenith_angle_deg': 30.0, 'pointing_error_urad': 20.0, 'scattered_power_ratio': 0.95, 'background_light_radiance_w_m2_sr_um': 0.0001, 'weather': 'clear'}, 'coherent_receiver_extension_v6': {'wavelength_m': 1.55e-06, 'satellite_altitude_m': 620000.0, 'receiver_altitude_m': 0.0, 'earth_radius_m': 6371000.0, 'earth_gravitational_parameter': 398600441800000.0, 'light_speed_m_s': 299792458.0, 'planck_constant_j_s': 6.62607015e-34, 'electron_charge_c': 1.602176634e-19, 'beam_radius_m': 0.05, 'receiver_aperture_m': 0.318, 'rms_wind_speed_m_s': 21.0, 'ground_cn2_m_neg_two_thirds': 1.7e-14, 'atmospheric_range_m': 20000.0, 'fog_range_m': 8000.0, 'field_of_view_rad': 0.1, 'optical_filter_bandwidth_um': 0.001, 'symbol_rate_baud': 28000000000.0, 'amplifier_gain_db': 30.0, 'amplifier_noise_figure_db': 4.0, 'lo_linewidth_hz': 1000.0, 'photodiode_responsivity_a_w': 0.85, 'doppler_residual_ratio': 0.1, 'malaga_omega': 0.5, 'malaga_b0': 0.25, 'malaga_phase_difference_rad': 1.5707963267948966, 'malaga_beta': None, 'max_derived_beta': 1000, 'beta_fit_policy': 'nearest_warn', 'beta_fit_relative_tolerance': 0.001, 'integration_points': 4097, 'integration_grid_power': 2.0, 'num_receivers': 1, 'combining': 'MRC', 'include_turbulence': True, 'include_background_noise': True, 'include_ase_noise': True, 'include_shot_noise': True, 'heterodyne_shot_noise_convention': 1.0, 'include_doppler': True, 'include_phase_noise': True, 'launch_power_dbm': 33.0, 'zenith_angle_deg': 30.0, 'pointing_error_urad': 20.0, 'scattered_power_ratio': 0.95, 'background_light_radiance_w_m2_sr_um': 0.0001, 'weather': 'clear'}}, 'dsp': {'amplitude_mode': 'known_block_magnitude', 'cfo_fft_oversampling': 4, 'cfo_newton_steps': 4, 'bps_test_phases': 64, 'bps_window_symbols': 65, 'cfo_enabled': True, 'cpr_enabled': True}, 'rect8_repair': {'receiver_version': 'rect8_constellation_ml_cfr_dd_ml_cpr_v2', 'affected_modulation': '8QAM', 'scope': 'rectangular 8QAM only; QPSK and 16QAM use unchanged v1', 'coarse_cfr': 'v1 fourth-power FFT and Newton periodogram refinement', 'fine_cfr': 'received-sample constellation-distance likelihood; frequency and constant phase jointly fitted', 'cpr': 'two windowed decision-directed ML phase updates, arg(sum(y*conj(decision)))', 'acquisition_search': 'one unoversampled fourth-harmonic CFO bin either side of the coarse estimate', 'acquisition_samples': 'at most 2048 uniformly spaced received samples; no data-dependent sampling', 'fine_frequency_trials': 17, 'fine_newton_steps': 5, 'phase_trials_and_window': 'unchanged SignalDSPConfig phase count and window length', 'phase_ambiguity': 'initial phase branch nearest zero; unidentifiable likelihood ties use that convention', 'no_pilots': True, 'uses_transmitted_bits_or_symbols': False, 'uses_true_channel_phase': False, 'uses_decoder_or_codec_feedback': False, 'amplitude_default': 'known positive real block magnitude only, as in v1', 'failure_policy': 'estimates and errors retained; no frame rejection', 'legacy_bps_reference': 'https://doi.org/10.1109/JLT.2008.2010511', 'nonuniform_qam_cfr_related_work': 'https://opg.optica.org/jlt/abstract.cfm?uri=jlt-37-23-5856', 'decision_aided_ml_related_work': 'https://opg.optica.org/jlt/abstract.cfm?uri=jlt-31-13-2055', 'related_work_scope': "estimator-family motivation only; neither paper's exact algorithm is reproduced"}, 'ldpc': {'name': 'systematic_ra_2048_1536_seeded_v1', 'n': 2048, 'k': 1536, 'm': 512, 'rate': 0.75, 'seed': 20261028, 'matrix_sha256': '12b54decfac6134e232ac460ebed1d2c5169aba3488a9ebfa638c2e806b40037', 'matrix_hash_encoding': 'row-major (check,variable), uint16 little endian pairs', 'edge_count': 5631, 'information_column_weight': 3, 'check_degree_min': 10, 'check_degree_max': 11, 'four_cycles': 0, 'decoder': 'flooding normalized min-sum with syndrome early stopping', 'normalization': 0.8, 'max_iterations': 50, 'llr_clip': 30.0, 'llr_convention': 'log(P0/P1); positive means bit zero', 'paper_dimensions_matched': True, 'paper_matrix_reproduced': False, 'matrix_assumption': 'H=[seeded balanced column-weight-3 A | lower-bidiagonal D]', 'construction_reference': 'H. Jin thesis, Chapter 7, https://thesis.caltech.edu/3191/1/HJinthesis.pdf', 'convergence_is_crc': False}, 'lpips': 'alex v0.1', 'side_information': 'perfect frame boundaries/filelength; block magnitude/noise PSD known; true phase unknown', 'no_crc_no_arq': True, 'channel_pairing': 'same seed/fading draw, not identical waveforms across different payload lengths', 'bootstrap_resamples': 2000, 'source_bitrate_target_mean_bpp': 0.21875}}
EXPERIMENT_LOCK_SHA256 = "eedd3e82847bf7793a791875dd92c6e7ddd2070cfd860a983d1ac6467c70e63f"

def verify_frozen_experiment(protocol):
    for key,value in EXPERIMENT_LOCK["settings"].items():
        if protocol.get(key)!=value:
            raise RuntimeError("Frozen experimental setting changed: "+key)
    if protocol["holdout_manifest"]["count"]!=200:
        raise RuntimeError("Frozen confirmatory image count differs")
    if protocol["data_mode"]=="reference_locked":
        if protocol["holdout_manifest"]["manifest_sha256"]!=EXPERIMENT_LOCK["holdout_manifest_sha256"]:
            raise RuntimeError("Frozen reference membership differs")


In [ ]:
encoder_path,decoder_path,codec_build = install_official_bpg(LOCAL_CACHE/"codec")
codec = BPGCodec(encoder=encoder_path,decoder=decoder_path,expected_hw=(256,256),timeout=20,speed=4)
ldpc = SystematicRALDPC(max_iterations=50,normalization=.8,seed=20261028,llr_clip=30).to(device)
# Real software smoke check before downloading/copying the final model.
smoke_image = Image.new("RGB",(256,256),(24,112,208))
smoke_data = codec.encode(smoke_image,q=30)
smoke_decoded = codec.decode(smoke_data)
if not smoke_decoded.ok:
    raise RuntimeError("Official BPG clean-image round-trip failed: "+smoke_decoded.error)
test_generator = torch.Generator(device=device).manual_seed(17)
smoke_info = torch.randint(2,(2,ldpc.k),generator=test_generator,device=device,dtype=torch.uint8)
smoke_code = ldpc.encode_bits(smoke_info)
smoke_recovered,smoke_stats = ldpc.decode_llr((1-2*smoke_code.float())*20)
if not torch.equal(smoke_info,smoke_recovered) or not bool(smoke_stats["converged"].all()):
    raise RuntimeError("Actual LDPC encoder/decoder round-trip failed")
print("Actual BPG and LDPC clean round-trips passed.")
print("LDPC:",ldpc.metadata())

# No images, true phase, LPIPS, or pretrained model are used in this check.
rect8_sanity = rect8_repair_self_test(device=str(device))
print("8QAM repaired-receiver synthetic sanity:",rect8_sanity)


In [ ]:
import lpips
import traceback

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda:handle.read(8*1024**2),b""):
            digest.update(chunk)
    return digest.hexdigest()

def save_json(path,value):
    path = Path(path)
    path.parent.mkdir(parents=True,exist_ok=True)
    temporary = path.with_suffix(path.suffix+".partial")
    temporary.write_text(json.dumps(value,ensure_ascii=False,indent=2,allow_nan=False),encoding="utf-8")
    temporary.replace(path)

test_root = prepare_uploaded_zip(DRIVE_DATA_DIR/"GEE_test.zip",LOCAL_CACHE,"gee_test",1000)
dataset = VisionDataset(test_root,MoVQCenterCropBicubic(),expected_count=1000)
all_hashes={name:file_sha256(Path(test_root)/name) for name in dataset.img_names}
current_test_zip_sha=json.loads((Path(test_root)/".prepared.json").read_text())["zip_sha256"]
if DATA_MODE=="reference_locked":
    holdout_manifest=FROZEN_HOLDOUT_MANIFEST
    names=verify_holdout_manifest(holdout_manifest,all_hashes,test_zip_sha256=current_test_zip_sha,
                                 expected_manifest_sha256=EXPERIMENT_LOCK["holdout_manifest_sha256"])
elif DATA_MODE=="own_data":
    # The development partition is reserved using file identities before any
    # evaluation; it is not selected from scores and is never evaluated here.
    ranked=sorted(all_hashes,key=lambda name:hashlib.sha256(
        f"20261028\0{all_hashes[name]}".encode("ascii")).hexdigest())
    reserved_development=ranked[:200]
    own_predeclaration=canonical_sha256({"role":"reserved_before_metrics","seed":20261028,
        "development_file_sha256":sorted(all_hashes[n] for n in reserved_development)})
    holdout_manifest=build_holdout_manifest(all_hashes,reserved_development,
        [all_hashes[n] for n in reserved_development],test_zip_sha256=current_test_zip_sha,
        previous_summary_sha256=own_predeclaration,count=200,seed=20261029)
    names=verify_holdout_manifest(holdout_manifest,all_hashes,test_zip_sha256=current_test_zip_sha)
else:
    raise ValueError("DATA_MODE must be reference_locked or own_data")
print("Final images:",len(names),"; exact overlap with reserved development:0; manifest:",holdout_manifest["manifest_sha256"])
positions = {name:i for i,name in enumerate(dataset.img_names)}
images_cpu = torch.stack([dataset[positions[name]] for name in names])
image_hashes = [file_sha256(Path(test_root)/name) for name in names]
if len(set(image_hashes)) != 200:
    raise RuntimeError("Duplicate selected files; inspect data instead of silently resampling")
test_zip_sha = json.loads((Path(test_root)/".prepared.json").read_text())["zip_sha256"]
source_settings = {"codec_build":codec_build,"encoder_settings":codec.encoder_settings,
    "selected_names":names,"selected_file_sha256":image_hashes,"test_zip_sha256":test_zip_sha,
    "target_mean_bytes":1792,"rate_control_code_sha256":RATE_CONTROL_CODE_SHA256,
    "selection_seed":20261028,"holdout_manifest_sha256":holdout_manifest["manifest_sha256"],
    "policy":"one adjacent-integer-QP stream per image; aggregate byte-only subset-sum; no quality selection"}
source_id = hashlib.sha256(json.dumps(source_settings,sort_keys=True).encode()).hexdigest()[:16]
source_cache_file = MODEL_OUTPUT_DIR/"final_holdout_v1"/"source_cache"/(source_id+".json")
source_cache = None
if source_cache_file.is_file():
    source_cache = json.loads(source_cache_file.read_text(encoding="utf-8"))
    if source_cache.get("settings") != source_settings or set(source_cache.get("entries",{})) != set(names):
        raise RuntimeError("Matched-rate source cache identity or image coverage differs")
if source_cache is None:
    print("Preparing actual BPG rate brackets; no neural quality metric is used",flush=True)
    def source_images():
        for i,name in enumerate(names):
            raster = ((images_cpu[i]+1)*127.5).round().clamp(0,255).to(torch.uint8).permute(1,2,0).numpy()
            yield name,Image.fromarray(raster)
    def source_progress(done,total,bracket):
        if done%25==0 or done==total:
            print(f"  BPG rate brackets {done}/{total}",flush=True)
    try:
        matched = prepare_rate_matched_sources(codec,source_images(),target_bytes_per_image=1792,
                                               seed=20261028,progress=source_progress)
    except Exception as error:
        save_json(source_cache_file.with_name(source_id+"_failure.json"),{
            "error_type":type(error).__name__,"error":str(error),"traceback":traceback.format_exc(),
            "policy":"stop without dropping images or faking rate by padding"})
        raise
    source_cache = {"settings":source_settings,"rate_match":matched.metadata(),"entries":{}}
    for name in names:
        item = matched.entries[name]
        source_cache["entries"][name] = {"base64":base64.b64encode(item.data).decode("ascii"),
            "metadata":item.metadata(),"sha256":hashlib.sha256(item.data).hexdigest()}
    save_json(source_cache_file,source_cache)
streams = {}
for name in names:
    entry = source_cache["entries"][name]
    stream = base64.b64decode(entry["base64"],validate=True)
    if hashlib.sha256(stream).hexdigest()!=entry["sha256"] or len(stream)!=entry["metadata"]["actual_bytes"]:
        raise RuntimeError("Matched-rate source cache byte/hash mismatch")
    streams[name] = stream
rate_target_bytes = 1792*len(names)
rate_actual_bytes = sum(map(len,streams.values()))
if (source_cache["rate_match"]["actual_total_bytes"]!=rate_actual_bytes
        or source_cache["rate_match"]["num_images"]!=len(names)):
    raise RuntimeError("Matched-rate cache aggregate metadata differs from actual source bytes")
actual_mean_bpp = rate_actual_bytes*8/(len(names)*256**2)
if abs(actual_mean_bpp-.21875)>.0001:
    raise RuntimeError("Aggregate source bitrate differs from target by more than0.0001bpp; report instead of treating it as matched")
print("Target source bpp:0.21875; actual:",actual_mean_bpp,"total byte error:",rate_actual_bytes-rate_target_bytes)

source = MODEL_OUTPUT_DIR/"stage2_noise_training_final.pth"
if not source.is_file():
    raise FileNotFoundError("Completed Stage2 final weights are required; do not retrain in this notebook")
local = LOCAL_CACHE/source.name
shutil.copyfile(source,local)
if local.stat().st_size != source.stat().st_size:
    raise RuntimeError("Incomplete final model copy")
model_sha = file_sha256(local)
if ENFORCE_REFERENCE_MODEL and model_sha!=EXPERIMENT_LOCK["reference_model_sha256"]:
    raise RuntimeError("Final model hash differs from frozen reference weights. Do not switch checkpoints based on holdout outcomes.")
# Only load your own previously produced checkpoint; never an external pickle.
payload = torch.load(local,map_location="cpu",weights_only=False)
if (payload.get("model_version") != MODEL_VERSION or payload.get("epochs_completed") != 100
        or payload.get("config",{}).get("stage") != 2
        or payload.get("config",{}).get("inferred_and_operational",{}).get("semantic_model") != SEMANTIC_MODEL_CONFIG):
    raise RuntimeError("Expected compatible completed100-epoch Stage2 final weights")
model = FSO_VQVAE().to(device).eval()
model.load_state_dict(payload["state_dict"],strict=True)
del payload
gc.collect()
with torch.inference_mode():
    encoded_indices = []
    for start in range(0,len(images_cpu),4):
        _,_,indices = model.encode(images_cpu[start:start+4].to(device))
        encoded_indices.append(indices.cpu().to(torch.int32))
    indices_cpu = torch.cat(encoded_indices)
if tuple(indices_cpu.shape)!=(200,32,32):
    raise RuntimeError("Unexpected semantic index layout")
metric = lpips.LPIPS(net="alex",version="0.1").to(device).eval()
metric.requires_grad_(False)
conditions = bpg_comparison_conditions()
profiles = ("paper_eq3_4_background_only","coherent_receiver_extension_v6")
protocol = {"version":"fso_sc_final_holdout_v1","code_sha256":BPG_COMPARISON_CODE_SHA256,
            "model_sha256":model_sha,"test_zip_sha256":test_zip_sha,
            "selected_images":names,"selected_file_sha256":image_hashes,"seed":20261028,
            "role":"frozen image-disjoint confirmation; not geographically independent",
            "data_mode":DATA_MODE,"reference_model_enforced":ENFORCE_REFERENCE_MODEL,
            "holdout_manifest":holdout_manifest,"experiment_lock_sha256":EXPERIMENT_LOCK_SHA256,
            "source_bitrate_target_mean_bpp":.21875,"source_bitrate_actual_mean_bpp":actual_mean_bpp,
            "rate_match":source_cache["rate_match"],"source_cache_id":source_id,
            "source_metadata":{name:source_cache["entries"][name]["metadata"] for name in names},
            "source_bpg_sha256":{name:source_cache["entries"][name]["sha256"] for name in names},
            "codec_build":codec_build,"ldpc":ldpc.metadata(),"conditions":conditions,
            "profiles":{p:profile_metadata(p) for p in profiles},
            "profile_configs":{p:__import__('dataclasses').asdict(make_channel_profile(p,PaperFSOSystemConfig)) for p in profiles},
            "dsp":__import__('dataclasses').asdict(SignalDSPConfig()),"rect8_repair":rect8_repair_metadata(),
            "side_information":"perfect frame boundaries/filelength; block magnitude/noise PSD known; true phase unknown",
            "no_crc_no_arq":True,"channel_pairing":"same seed/fading draw, not identical waveforms across different payload lengths",
            "failure_policy":"declaredBPGheader-validation/native-decodefailure->paper-style marker1; separately measured gray-render LPIPS",
            "fairness":"matched aggregate source bitrate; per-image rate may differ; actual FEC/padding/coded bits reported; NOT equal channel uses",
            "lpips":"alex v0.1","bootstrap_resamples":2000,
            "torch":torch.__version__,"gpu":torch.cuda.get_device_name()}
verify_frozen_experiment(protocol)
run_id = hashlib.sha256(json.dumps(protocol,sort_keys=True).encode()).hexdigest()[:16]
RESULT_DIR = MODEL_OUTPUT_DIR/"final_holdout_v1"/run_id
RESULT_DIR.mkdir(parents=True,exist_ok=True)
save_json(RESULT_DIR/"protocol.json",protocol)
print("Result directory:",RESULT_DIR)
print("Frozen physical image-method trials (no development scores reused):",len(profiles)*len(conditions)*len(names)*3)
print("BPG mean source bpp:",np.mean([len(streams[n])*8/256**2 for n in names]))

save_json(RESULT_DIR/"holdout_manifest.json",holdout_manifest)


In [ ]:
import traceback

def image_progress(done,total):
    if done%100 == 0 or done==total:
        print(f"  images {done}/{total}",flush=True)

reports = {}
tasks = [("clean","fso_sc",{"id":"clean","channel_kwargs":{}},None,None),
         ("clean","bpg",{"id":"clean","channel_kwargs":{}},None,None)]
for profile in profiles:
    physical = PaperMalagaFSOChannel(make_channel_profile(profile,PaperFSOSystemConfig))
    link = SignalDSPPaperIndexChannel(physical)
    for condition in conditions:
        tasks.extend([(profile,"fso_sc",condition,link,None),
                      (profile,"bpg",condition,physical,None),
                      (profile,"bpg_ldpc",condition,physical,ldpc)])
started = time.perf_counter()
new_seconds = []
for task_index,(profile,method,condition,channel,code) in enumerate(tasks):
    key = f"{profile}_{method}_{condition['id']}"
    path = RESULT_DIR/(key+".json")
    if path.is_file():
        result = json.loads(path.read_text(encoding="utf-8"))
        if (result.get("run_id") != run_id or result.get("profile") != profile
                or result.get("method") != method or result.get("condition") != condition
                or {r["image_name"] for r in result.get("records",[])} != set(names)
                or len(result["records"]) != len(names)):
            raise RuntimeError("Cached condition identity/coverage differs; do not mix runs")
        print("Reuse:",key)
    else:
        print(f"[{task_index+1}/{len(tasks)}] {key}",flush=True)
        tick = time.perf_counter()
        try:
            if method == "fso_sc":
                result = evaluate_condition(model,images_cpu,indices_cpu,metric,channel,condition,names,
                                            repeats=1,device=device,base_seed=20261028,progress=image_progress)
                summary = result["summary"]
                modulation = condition["channel_kwargs"].get("modulation","QPSK")
                bps = {"QPSK":2,"8QAM":3,"16QAM":4}[modulation]
                bits = indices_cpu[0].numel()*14
                actual_bits = math.ceil(bits/bps)*bps if channel else 0
                summary.update(decode_failure_rate=None,rendered_lpips_mean=summary["mean"],
                               source_bpp_mean=.21875,transmitted_bpp_mean=actual_bits/256**2,
                               channel_symbols_mean=actual_bits/bps,decode_failures=None)
            else:
                result = evaluate_bpg_condition(codec,streams,images_cpu,metric,channel,condition,names,
                                                globals(),device=device,ldpc=code,base_seed=20261028,progress=image_progress)
                if channel is None and result["summary"]["decode_failures"]:
                    raise RuntimeError("Clean BPG source decode failed; investigate codec before channel comparisons")
        except Exception as error:
            save_json(RESULT_DIR/(key+"_failure.json"),{"profile":profile,"method":method,
                "condition":condition,"error":str(error),"policy":"stop; no frames discarded or fabricated"})
            save_json(RESULT_DIR/(key+"_traceback.json"),{"error_type":type(error).__name__,
                "traceback":traceback.format_exc(),"condition":condition,"profile":profile,"method":method})
            raise
        result.update(run_id=run_id,profile=profile,method=method,elapsed_seconds=time.perf_counter()-tick)
        save_json(path,result)
        new_seconds.append(result["elapsed_seconds"])
    reports[key] = result
    values = result["summary"]
    print("  LPIPS",round(values["mean"],5),"postBER",round(values["payload_ber"],6),
          "decode fail",values["decode_failure_rate"],"coded bpp",round(values["transmitted_bpp_mean"],5),flush=True)
    if new_seconds:
        remaining = len(tasks)-task_index-1
        print(f"  Rough remaining {np.median(new_seconds)*remaining/60:.1f}min; differs with LDPC failures/cache",flush=True)
summary = {"run_id":run_id,"protocol":protocol,
           "results":{key:{k:report[k] for k in ("profile","method","condition","summary","elapsed_seconds")}
                      for key,report in reports.items()},
           "interpretation":"frozen image-disjoint confirmation; conditional partial reproduction; no automatic3dB gain claim",
           "development_scores_reused":False,"evaluation_role":"final_holdout",
           "elapsed_this_cell_seconds":time.perf_counter()-started}
save_json(RESULT_DIR/"final_holdout_summary.json",summary)
print("Completed. Send final_holdout_summary.json.")


In [ ]:
import matplotlib.pyplot as plt

fig,axes = plt.subplots(2,3,figsize=(13,7),sharey=True)
colors = {"fso_sc":"tab:blue","bpg":"tab:orange","bpg_ldpc":"tab:green"}
for row,profile in enumerate(profiles):
    for col,modulation in enumerate(("QPSK","8QAM","16QAM")):
        axis = axes[row,col]
        chosen = [c for c in conditions if c["group"]=="power_at_zenith30" and c["channel_kwargs"]["modulation"]==modulation]
        for method in ("fso_sc","bpg","bpg_ldpc"):
            values = [reports[f"{profile}_{method}_{c['id']}"]["summary"] for c in chosen]
            powers = [c["channel_kwargs"]["launch_power_dbm"] for c in chosen]
            means = np.array([v["mean"] for v in values])
            errors = np.array([[v["ci95"][0],v["ci95"][1]] for v in values]).T
            axis.errorbar(powers,means,yerr=np.maximum(np.vstack((means-errors[0],errors[1]-means)),0),
                          marker="o",label=method,color=colors[method])
        axis.set_title(modulation+"; zenith30deg")
        axis.set_xlabel("Launch power [dBm]")
        axis.set_xticks([30,33,36])
        axis.grid(alpha=.25)
    profile_label = ("Eq.(3)-(4): background only" if row==0 else "Coherent receiver extension v6")
    axes[row,0].set_ylabel(profile_label+"\nLPIPS / failure marker (lower better)")
axes[0,0].legend()
fig.suptitle("Aggregate source-rate matched, NOT equal channel uses; BPG decode failure is marker1")
fig.tight_layout()
fig.savefig(RESULT_DIR/"bpg_power_comparison.png",dpi=180)
plt.show()

import csv
with (RESULT_DIR/"bpg_comparison_table.csv").open("w",newline="",encoding="utf-8-sig") as handle:
    columns = ["profile","method","condition","lpips_paper_convention","rendered_lpips", "decode_failure_rate",
               "post_fec_ber","source_bpp","transmitted_bpp","symbols"]
    writer = csv.DictWriter(handle,fieldnames=columns)
    writer.writeheader()
    for key,result in reports.items():
        value = result["summary"]
        writer.writerow(dict(zip(columns,[result["profile"],result["method"],result["condition"]["id"],
            value["mean"],value["rendered_lpips_mean"],value["decode_failure_rate"],value["payload_ber"],
            value["source_bpp_mean"],value["transmitted_bpp_mean"],value["channel_symbols_mean"]])))
print("Saved summary, per-image condition results, plot and rate/failure table in",RESULT_DIR)
